# Runtime-optimized v3.2.2 — pixel-grouped, full-inventory, no locked test

This edition preserves the lock-free research design, all six candidate models, the repeated five-fold/three-seed buffered spatial CV, nested tuning/calibration, the size- and class-matched benchmark, and the map comparison. It reduces redundant computation as follows:

- 15 nominal random-validation holdouts (20% test fraction). The number of distinct test partitions and unique observations tested must be audited from the generated split/prediction records; do not assume 15 independent partitions or complete out-of-fold coverage.
- Two inner spatial/random folds, four final tuning draws and two calibration-fold tuning draws.
- One size-/class-matched random draw per spatial fold and model.
- Three buffer-sensitivity distances on the prespecified primary partition seed.
- Mapping ensemble restricted to the five spatial folds from the primary seed; the map uncertainty is therefore conditional on that partition, not a three-seed ensemble.
- Progress messages and intermediate CSV checkpoints are written during model fitting; raster prediction reports window progress.

**Important:** Checkpoint CSVs preserve completed metric/prediction rows for audit, but this notebook does not automatically resume a partially completed estimator fit after a runtime restart. A prior execution of this workflow produced fewer distinct test partitions than nominal seeds, so the split audit and unique prediction coverage must be reported; the nominal repeat count is not evidence of independent repetitions. Rerunning a cell will recompute that cell. The final scientific results still require a complete successful run and the preflight checks.


**v3.2.2 mixed-label policy:** The current inventory contains one reviewed two-record conflict at identical coordinates with identical values across all predictors (record IDs 46 and 3850). Both records are retained without relabeling and are forced to remain in the same 1-km sampling group. The run logs this limitation. A sensitivity run excluding both records is recommended before publication. The code stops if the conflict pattern differs from this reviewed case.


# v3.2.2 update — raster-pixel-grouped validation and stable partition-key checks

**Important:** The validation design now treats each 1-km raster cell as an indivisible sampling group. This prevents observations that share the same raster-extracted predictor support from being split between training and validation, including exact-coordinate duplicates. The same constraint is applied to random outer splits, random inner tuning folds, calibration cross-fitting, spatial partitioning, and the size-matched benchmark.

The inventory audit identifies one 1-km cell containing both presence and background labels. The default configuration stops at the full-inventory setup cell until that record pair is investigated. Do not silently delete or relabel either record. After checking the source inventory, correct the source or explicitly set `ALLOW_MIXED_LABEL_PIXEL_GROUPS = True` to retain both records in the same group and document the unresolved ambiguity.

**Rerun from the beginning.** Previous v3.1 random-validation, calibration, size-matched, bootstrap, and manuscript-summary outputs are not valid substitutes for the new grouped run. Spatial results are also recomputed because spatial fold construction now respects raster-cell groups.


# SR-GEOAI v3 — Full-inventory repeated spatial validation
## Flood-susceptibility modelling across South Asia

**Purpose.** This notebook implements the revised, validation-focused study using the complete corrected modelling inventory. It removes the locked geographic test, locked-region exclusion, locked-test post-hoc analyses, and soil-type ablation. All 3,416 complete observations from the corrected 21-predictor table are returned to the modelling population (the exact count is rechecked at runtime; it may change if the source inputs differ).

### Final analytical design
1. Rebuild the corrected 1-km predictor stack in a South Asia-centred equal-area CRS and filter observations using only the 21 predictors retained for modelling.
2. Diagnose spatial dependence and estimate a predictor-only reference range.
3. Compare repeated nested random validation with repeated nested buffered spatial validation.
4. Select spatial folds and buffer distances using geometry and prespecified training/test support requirements only; no model metric is used for design selection.
5. Quantify the contribution of training-support reduction using a class-count- and sample-size-matched random benchmark for the tree-ensemble models.
6. Evaluate buffer-distance sensitivity on fixed regional partitions as a secondary, fixed-hyperparameter experiment.
7. Evaluate probability calibration using cross-fitted calibration where hyperparameter tuning is repeated within each calibration-training fold.
8. Compare the spatial-fold ensemble surface with a size-matched random-reference surface and report map agreement, disagreement, and prediction variability.
9. Retain native-grid Gi* and country summaries as **descriptive supplementary analyses**, not independent validation.

### Interpretation limits
- This design does **not** contain an independent geographic test region. Reported performance estimates are repeated cross-validation estimates under the specified random or buffered designs, not independent external validation or design-based map accuracy.
- The selected model is chosen from repeated spatial-CV results for producing an operational full-inventory surface. Its winning cross-validation mean is subject to model-selection optimism and must not be described as an unbiased final test score.
- Predicted values are relative susceptibility scores conditional on the case/background sampling design, not population flood probabilities.
- The notebook cannot infer missing flood-event identifiers, reconstruct the original polygon-to-point decisions, or establish the exact background sampling distance/temporal matching. These remain data-provenance limitations unless recovered from the original inventory workflow.

### Execution
Run every cell from the top in a clean Kaggle environment with the original source point inventory and predictor rasters available. All previous cell outputs are cleared in this v3 copy to prevent results from earlier, locked-test analyses being mistaken for v3 results. The full model rerun is computationally intensive.

## Repository and data configuration

This notebook is distributed as source code only; the flood point inventory and predictor rasters are not included. By default, place the source data under `data/raw/flood-susceptibility-south-asia/` using the directory layout in `data/raw/README.md`. Alternatively, configure `FLOOD_GEOAI_DATA_ROOT`, `FLOOD_GEOAI_POINTS_PATH`, `FLOOD_GEOAI_RASTER_DIR`, `FLOOD_GEOAI_OUTPUT_DIR`, and `FLOOD_GEOAI_COUNTRY_VECTOR_PATH` before starting the kernel. Do not commit raw inventories, rasters, intermediate products, or generated result packages to the public repository unless their licences and redistribution permissions have been confirmed.



## 0. Configuration and input expectations

The notebook constructs the modelling table from the source point inventory and raster stack in the master dataset-construction cell. By default, the data are expected beneath `data/raw/flood-susceptibility-south-asia/`; environment variables can override the point and raster paths. Required fields are a binary `Label`, projected coordinates (`x`, `y`), and the final 21 predictor columns. The full corrected inventory is used; no geographic observations are reserved or removed for a locked test.

In [ ]:
# ============================================================
# SR-GEOAI MASTER DATASET CONSTRUCTION — CORRECTED
# ============================================================
# Reprojects the supplied predictor rasters to a custom South Asia
# Albers Equal Area grid before extracting sample values. All 24 input
# rasters are preserved for QC, but complete-case filtering uses only
# the 21 final model predictors.

import os, json, math, warnings, hashlib
from pathlib import Path
import geopandas as gpd
import rasterio as rio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rasterio.warp import transform_bounds, reproject, Resampling
from rasterio.transform import from_origin
from pyproj import CRS, Proj, Geod, Transformer

warnings.filterwarnings('default')

# ----------------------------- portable paths --------------------------------
# By default, input data are expected under ./data/raw. To use another location,
# set FLOOD_GEOAI_DATA_ROOT, FLOOD_GEOAI_POINTS_PATH, and/or
# FLOOD_GEOAI_RASTER_DIR in the environment before launching Jupyter.
DATA_ROOT = Path(os.environ.get("FLOOD_GEOAI_DATA_ROOT", "data/raw")).expanduser()
DATASET_DIR = DATA_ROOT / "flood-susceptibility-south-asia"

POINTS_PATH = Path(os.environ.get(
    "FLOOD_GEOAI_POINTS_PATH",
    str(DATASET_DIR / "Flood Training" / "Flood Training" / "Flood_Point.shp")
)).expanduser()

RASTER_DIR = Path(os.environ.get(
    "FLOOD_GEOAI_RASTER_DIR",
    str(DATASET_DIR / "Flood_1km-20260503T054519Z-3-001" / "Flood_1km")
)).expanduser()

OUTPUT_DIR = Path(os.environ.get(
    "FLOOD_GEOAI_OUTPUT_DIR",
    "outputs/sr_geoai_outputs_v3_2_2_pixel_grouped"
)).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Custom Albers Equal Area projection centred on South Asia. The standard
# parallels bracket the study region; the CRS is metric and equal-area.
# It avoids using ESRI:102028 (Asia South Albers), whose documented area of
# use ends at 30 degrees N and therefore does not cover the northern study area.
TARGET_CRS = CRS.from_proj4(
    "+proj=aea +lat_0=23 +lon_0=80 +lat_1=8 +lat_2=38 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs +type=crs"
)
TARGET_RESOLUTION_M = 1000.0
TARGET_NODATA = -9999.0
HARMONIZED_RASTER_DIR = OUTPUT_DIR / "harmonized_aea_1km"
HARMONIZED_RASTER_DIR.mkdir(parents=True, exist_ok=True)

# Optional event identifier. Configure explicitly if a stable event field exists.
EVENT_ID_COL = None
# Only unambiguously identifier-like names are auto-detected. A generic field
# named "Event" or a date field is not assumed to be a unique event identifier.
AUTO_EVENT_ID_CANDIDATES = [
    "event_id", "Event_ID", "EVENT_ID", "flood_id", "Flood_ID"
]

# ---------------------- final predictor definition ---------------------------
REMOVED_FEATURES = ["Surf_rough", "NDWI", "Ext_rain_a"]
FEATURE_COLS = [
    "Elevation", "Slope", "Aspect", "Curvature", "TWI",
    "Soil_type", "Geology", "Soil_moist", "NDVI", "LULC",
    "Dist_road", "Dist_river", "Dist_strm", "Drain_dens", "Flow_accu",
    "Runoff", "Total_rain", "Rain_anom", "Rain_days", "Ext_rain_n", "Pop_dens"
]
CATEGORICAL_FEATURES = ["Soil_type", "Geology", "LULC"]
NUMERIC_FEATURES = [c for c in FEATURE_COLS if c not in CATEGORICAL_FEATURES]

RASTERS = {
    "Elevation": "Elevation.tif",
    "Slope": "Slope.tif",
    "Aspect": "Aspect.tif",
    "Surf_rough": "Surface_roughness.tif",
    "Curvature": "Curvature.tif",
    "TWI": "TWI.tif",
    "Soil_type": "Soil.tif",
    "Geology": "Geology.tif",
    "Soil_moist": "Soil_moisture.tif",
    "NDVI": "NDVI.tif",
    "NDWI": "NDWI.tif",
    "LULC": "LULC.tif",
    "Dist_road": "Dist_from_road.tif",
    "Dist_river": "Dist_from_river.tif",
    "Dist_strm": "Dist_from_stream.tif",
    "Drain_dens": "Drainage_density.tif",
    "Flow_accu": "Flow_accumulation.tif",
    "Runoff": "Runoff.tif",
    "Total_rain": "Total_rainfall.tif",
    "Rain_anom": "Rainfall_anomaly.tif",
    "Rain_days": "Rainfall_day.tif",
    "Ext_rain_n": "95p_day.tif",
    "Ext_rain_a": "95p_mm.tif",
    "Pop_dens": "Population_density.tif",
}

source_raster_paths = {k: Path(RASTER_DIR) / v for k, v in RASTERS.items()}
missing = [str(p) for p in source_raster_paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing source raster(s):\n" + "\n".join(missing))

# --------------------------- load inventory ---------------------------------
points = gpd.read_file(POINTS_PATH)
if points.crs is None:
    raise ValueError("Flood inventory has no CRS. Assign its verified source CRS before continuing.")
if "Label" not in points.columns and "Id" in points.columns:
    points = points.rename(columns={"Id": "Label"})
if "Label" not in points.columns:
    raise KeyError(f"Could not find Label/Id field. Available columns: {points.columns.tolist()}")
if points.geometry.isna().any() or points.geometry.is_empty.any():
    raise ValueError("Flood inventory contains null or empty geometries; resolve them before extraction.")
geom_types = set(points.geometry.geom_type.unique())
if not geom_types.issubset({"Point"}):
    raise TypeError(f"Expected point inventory geometries; found {sorted(geom_types)}")

if EVENT_ID_COL is None:
    event_matches = [c for c in AUTO_EVENT_ID_CANDIDATES if c in points.columns]
    EVENT_ID_COL = event_matches[0] if len(event_matches) == 1 else None
if EVENT_ID_COL:
    print(f"Event identifier configured: {EVENT_ID_COL}")
else:
    print("WARNING: no unambiguous event identifier is available. Event-level independence cannot be verified.")

points["inventory_record_id"] = np.arange(len(points), dtype=int)
points["Label"] = pd.to_numeric(points["Label"], errors="coerce")
points = points.dropna(subset=["Label"]).copy()
points["Label"] = points["Label"].astype(int)
if not set(points["Label"].unique()).issubset({0, 1}):
    raise ValueError("Label must contain only 0 and 1; verify the inventory label encoding.")

# ------------------------- source raster signatures --------------------------
def raster_signature(path):
    with rio.open(path) as src:
        return {
            "path": str(path), "crs": str(src.crs), "width": src.width,
            "height": src.height, "transform": tuple(src.transform),
            "resolution": tuple(src.res), "bounds": tuple(src.bounds),
            "nodata": src.nodata, "dtype": src.dtypes[0]
        }

source_signatures = []
for name, path in source_raster_paths.items():
    with rio.open(path) as src:
        if src.crs is None:
            raise ValueError(f"Raster {name} has no CRS: {path}")
        source_signatures.append({"predictor": name, **raster_signature(path)})
source_signature_df = pd.DataFrame(source_signatures)
source_signature_df.to_csv(OUTPUT_DIR / "source_raster_signatures.csv", index=False)

# ---------------------- create common target grid ----------------------------
# Use the intersection of transformed raster extents so every target cell is
# within the footprint of every supplied predictor. The grid is anchored to
# integer multiples of the 1-km target resolution.
target_bounds = []
# The common grid is determined only by the 21 predictors used by the model.
# Excluded candidate rasters must not reduce the analysis footprint.
for name in FEATURE_COLS:
    path = source_raster_paths[name]
    with rio.open(path) as src:
        tb = transform_bounds(src.crs, TARGET_CRS, *src.bounds, densify_pts=41)
        if not np.all(np.isfinite(tb)):
            raise ValueError(f"Could not transform bounds for {path}; inspect source CRS and extent.")
        target_bounds.append(tb)
left = max(b[0] for b in target_bounds)
bottom = max(b[1] for b in target_bounds)
right = min(b[2] for b in target_bounds)
top = min(b[3] for b in target_bounds)
if not (right > left and top > bottom):
    raise ValueError("The 24 predictor rasters have no common footprint after reprojection.")
res = TARGET_RESOLUTION_M
left = math.floor(left / res) * res
bottom = math.floor(bottom / res) * res
right = math.ceil(right / res) * res
top = math.ceil(top / res) * res
width = int(round((right - left) / res))
height = int(round((top - bottom) / res))
target_transform = from_origin(left, top, res, res)
if width <= 0 or height <= 0:
    raise ValueError("Invalid target grid dimensions.")

# Reproject all layers onto the same physical 1-km grid. Nearest-neighbour is
# used for nominal categorical rasters; bilinear for continuous rasters.
raster_paths = {}
reprojection_rows = []
for name, source_path in source_raster_paths.items():
    out_path = HARMONIZED_RASTER_DIR / f"{name}_AEA1km.tif"
    # Aspect is circular (0 degrees is adjacent to 360 degrees), so ordinary
    # bilinear interpolation can create physically meaningless intermediate angles.
    resampling = (Resampling.nearest if name in CATEGORICAL_FEATURES or name == "Aspect"
                  else Resampling.bilinear)
    with rio.open(source_path) as src:
        source_crs_text = str(src.crs)
        dest = np.full((height, width), TARGET_NODATA, dtype=np.float32)
        reproject(
            source=rio.band(src, 1), destination=dest,
            src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
            dst_transform=target_transform, dst_crs=TARGET_CRS,
            dst_nodata=TARGET_NODATA, resampling=resampling,
            init_dest_nodata=True, num_threads=2
        )
        profile = {
            "driver": "GTiff", "height": height, "width": width,
            "count": 1, "dtype": "float32", "crs": TARGET_CRS,
            "transform": target_transform, "nodata": TARGET_NODATA,
            "compress": "deflate", "tiled": True,
            "blockxsize": 256, "blockysize": 256,
            "BIGTIFF": "IF_SAFER"
        }
        with rio.open(out_path, "w", **profile) as dst:
            dst.write(dest, 1)
            dst.set_band_description(1, name)
    raster_paths[name] = out_path
    reprojection_rows.append({
        "predictor": name, "source_path": str(source_path),
        "target_path": str(out_path), "resampling": resampling.name,
        "source_crs": source_crs_text, "target_crs": TARGET_CRS.to_string(),
        "target_width": width, "target_height": height,
        "target_resolution_m": res, "valid_target_cells": int(np.sum(dest != TARGET_NODATA))
    })

reprojection_df = pd.DataFrame(reprojection_rows)
reprojection_df.to_csv(OUTPUT_DIR / "raster_reprojection_manifest.csv", index=False)

# Validate exact alignment of all outputs.
ref_path = raster_paths[FEATURE_COLS[0]]
with rio.open(ref_path) as ref:
    raster_crs = ref.crs
    raster_shape = (ref.height, ref.width)
    raster_transform = ref.transform
    raster_bounds = ref.bounds
    raster_res = ref.res
for name, path in raster_paths.items():
    with rio.open(path) as ras:
        if (ras.crs != raster_crs or ras.shape != raster_shape or
            not ras.transform.almost_equals(raster_transform) or
            not np.allclose(ras.res, raster_res, atol=1e-9, rtol=0)):
            raise ValueError(f"Target raster alignment failed for {name}.")
        if not np.allclose(ras.res, (1000.0, 1000.0), atol=1e-6):
            raise ValueError(f"Target raster {name} is not 1,000 m by 1,000 m.")

# ----------------------- projection quality diagnostics ----------------------
# Report local linear-scale factors and sampled geodesic-vs-projected distance
# disagreement. Equal-area does not mean equidistant; distances remain approximate.
projection = Proj(TARGET_CRS)
scale_rows = []
for lat in [8, 15, 23, 30, 35, 38]:
    for lon in [60, 70, 80, 90, 100]:
        f = projection.get_factors(lon, lat)
        scale_rows.append({
            "longitude": lon, "latitude": lat,
            "meridional_scale": f.meridional_scale,
            "parallel_scale": f.parallel_scale,
            "angular_distortion_deg": f.angular_distortion
        })
projection_scale_df = pd.DataFrame(scale_rows)
projection_scale_df.to_csv(OUTPUT_DIR / "projection_scale_diagnostics.csv", index=False)

# -------------------------- point projection/extraction ----------------------
points = points.to_crs(TARGET_CRS)
xy = np.column_stack([points.geometry.x.to_numpy(), points.geometry.y.to_numpy()])
rows, cols = rio.transform.rowcol(raster_transform, xy[:, 0], xy[:, 1])
rows = np.asarray(rows, dtype=int)
cols = np.asarray(cols, dtype=int)
inside = ((rows >= 0) & (rows < raster_shape[0]) &
          (cols >= 0) & (cols < raster_shape[1]))

extracted = points.copy()
for name, path in raster_paths.items():
    with rio.open(path) as ras:
        sampled = list(ras.sample(xy, indexes=1, masked=True))
        samp = np.ma.vstack(sampled)
        vals = np.asarray(samp.data[:, 0], dtype=float)
        mask = np.ma.getmaskarray(samp)[:, 0]
        vals[mask] = np.nan
        vals[~inside] = np.nan
        if ras.nodata is not None and np.isfinite(ras.nodata):
            vals[np.isclose(vals, ras.nodata)] = np.nan
        vals[~np.isfinite(vals)] = np.nan
        extracted[name] = vals

df_raw = extracted.copy()
qc = [{"stage": "inventory_after_label_cleaning", "n_rows": int(len(df_raw)),
       "presence": int((df_raw.Label == 1).sum()), "background": int((df_raw.Label == 0).sum())}]

before_inside = len(df_raw)
df_raw = df_raw.loc[inside].copy()
qc.append({"stage": "inside_common_target_grid", "n_rows": int(len(df_raw)),
           "presence": int((df_raw.Label == 1).sum()), "background": int((df_raw.Label == 0).sum()),
           "n_removed": int(before_inside - len(df_raw))})

# IMPORTANT FIX: filter on the 21 variables actually used, not the 24 candidate
# layers. Missing values in Surf_rough, NDWI or Ext_rain_a must not discard rows.
before_complete = len(df_raw)
df = df_raw.dropna(subset=FEATURE_COLS + ["Label"]).copy()
qc.append({"stage": "complete_case_final_21_predictors", "n_rows": int(len(df)),
           "presence": int((df.Label == 1).sum()), "background": int((df.Label == 0).sum()),
           "n_removed": int(before_complete - len(df))})

if not set(df["Label"].unique()).issubset({0, 1}):
    raise ValueError("Final modelling target must contain only 0 and 1.")
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].astype(str)

X = df[FEATURE_COLS].copy()
y = df["Label"].astype(int).copy()
df["x"] = df.geometry.x
df["y"] = df.geometry.y
coords = df[["x", "y"]].to_numpy(dtype=float)

# Exact WGS84 ellipsoidal nearest-neighbour distances are used for exclusion
# buffers and separation checks. The projected Albers coordinates remain useful
# for spatial block tessellation and raster alignment, but are not treated as
# perfectly equidistant.
def geodesic_nearest_distance_m(reference_xy, query_xy, source_crs=TARGET_CRS):
    """Minimum WGS84 ellipsoidal distance from each query to a reference point.

    A haversine BallTree retrieves the eight nearest spherical candidates; the
    exact WGS84 ellipsoidal distance is calculated for each candidate and the
    minimum is retained. This is more robust than applying the ellipsoidal
    calculation to only one spherical nearest-neighbour candidate.
    """
    from sklearn.neighbors import BallTree
    reference_xy = np.asarray(reference_xy, dtype=float)
    query_xy = np.asarray(query_xy, dtype=float)
    if (reference_xy.ndim != 2 or query_xy.ndim != 2 or
            reference_xy.shape[1] != 2 or query_xy.shape[1] != 2):
        raise ValueError("reference_xy and query_xy must be N x 2 coordinate arrays.")
    if len(reference_xy) == 0:
        raise ValueError("At least one reference point is required for nearest-distance calculation.")
    if len(query_xy) == 0:
        return np.empty(0, dtype=float)

    transformer = Transformer.from_crs(source_crs, CRS.from_epsg(4326), always_xy=True)
    ref_lon, ref_lat = transformer.transform(reference_xy[:, 0], reference_xy[:, 1])
    qry_lon, qry_lat = transformer.transform(query_xy[:, 0], query_xy[:, 1])
    ref_rad = np.deg2rad(np.column_stack([ref_lat, ref_lon]))
    qry_rad = np.deg2rad(np.column_stack([qry_lat, qry_lon]))
    tree = BallTree(ref_rad, metric="haversine")
    k = min(8, len(reference_xy))
    _, candidate_index = tree.query(qry_rad, k=k)

    cand_lon = np.asarray(ref_lon)[candidate_index]
    cand_lat = np.asarray(ref_lat)[candidate_index]
    query_lon = np.repeat(np.asarray(qry_lon)[:, None], k, axis=1)
    query_lat = np.repeat(np.asarray(qry_lat)[:, None], k, axis=1)
    _, _, distances = Geod(ellps="WGS84").inv(
        cand_lon.ravel(), cand_lat.ravel(), query_lon.ravel(), query_lat.ravel()
    )
    return np.asarray(distances, dtype=float).reshape(len(query_xy), k).min(axis=1)

# Projection distance diagnostic using random pairs of inventory points.
# This records (rather than conceals) residual distance distortion in the custom CRS.
if len(coords) >= 3:
    to_geo = Transformer.from_crs(TARGET_CRS, CRS.from_epsg(4326), always_xy=True)
    lons, lats = to_geo.transform(coords[:, 0], coords[:, 1])
    rng_proj = np.random.default_rng(42)
    n_pairs = min(5000, max(1000, len(coords) * 2))
    ia = rng_proj.integers(0, len(coords), n_pairs)
    ib = rng_proj.integers(0, len(coords), n_pairs)
    distinct = ia != ib
    ia, ib = ia[distinct], ib[distinct]
    d_projected = np.linalg.norm(coords[ia] - coords[ib], axis=1)
    geod = Geod(ellps="WGS84")
    _, _, d_geodesic = geod.inv(np.asarray(lons)[ia], np.asarray(lats)[ia],
                                np.asarray(lons)[ib], np.asarray(lats)[ib])
    valid_dist = np.isfinite(d_projected) & np.isfinite(d_geodesic) & (d_geodesic > 0)
    rel = 100.0 * (d_projected[valid_dist] / d_geodesic[valid_dist] - 1.0)
    projection_distance_qc = {
        "n_pairs": int(valid_dist.sum()),
        "median_abs_relative_distance_error_pct": float(np.median(np.abs(rel))),
        "p95_abs_relative_distance_error_pct": float(np.quantile(np.abs(rel), 0.95)),
        "max_abs_relative_distance_error_pct": float(np.max(np.abs(rel)))
    }
else:
    projection_distance_qc = {"n_pairs": 0}

projection_metadata = {
    "target_crs_wkt": TARGET_CRS.to_wkt(),
    "target_crs_proj4": TARGET_CRS.to_proj4(),
    "target_resolution_m": TARGET_RESOLUTION_M,
    "height": int(height), "width": int(width),
    "transform": tuple(target_transform),
    "bounds": tuple(raster_bounds),
    "distance_distortion_sample": projection_distance_qc,
    "important_lineage_note": (
        "Input rasters were already harmonized before this notebook. Reprojection fixes the "
        "analysis grid and distance calculations but cannot recover information lost during "
        "earlier resampling. For final high-tier submission, regenerate from native sources "
        "where possible."
    )
}
(OUTPUT_DIR / "projection_and_grid_manifest.json").write_text(
    json.dumps(projection_metadata, indent=2, default=str), encoding="utf-8"
)

# Duplicate-point audit: duplicates are reported, not silently dropped.
point_key = pd.DataFrame({"x": df.geometry.x, "y": df.geometry.y, "Label": df.Label})
duplicate_summary = point_key.duplicated(["x", "y"], keep=False)
duplicate_points = df.loc[duplicate_summary].copy()
duplicate_points.drop(columns=["geometry"], errors="ignore").to_csv(
    OUTPUT_DIR / "duplicate_point_coordinate_audit.csv", index=False
)

# Inventory/event diagnostics. An absent event field is not fabricated.
if EVENT_ID_COL and EVENT_ID_COL in df.columns:
    event_diag = df.groupby(EVENT_ID_COL, dropna=False).agg(
        n_points=("Label", "size"), n_classes=("Label", "nunique"),
        n_presence=("Label", lambda s: int((s == 1).sum())),
        n_background=("Label", lambda s: int((s == 0).sum()))
    ).reset_index()
    event_diag.to_csv(OUTPUT_DIR / "event_inventory_summary.csv", index=False)

# ------------------------------ figures/QC ------------------------------------
plt.figure(figsize=(6, 5))
sns.countplot(x="Label", data=df)
plt.title("Flood inventory class distribution after 21-predictor filtering", fontsize=12, weight="bold")
plt.xlabel("Label (1 = documented flood presence; 0 = background)")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

corr_numeric = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").corr()
plt.figure(figsize=(15, 11), dpi=200)
sns.heatmap(corr_numeric, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={"size": 7},
            cbar_kws={"shrink": 0.7}, square=True)
plt.title("Numeric predictor correlation matrix", fontsize=13, weight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_matrix_numeric.png", dpi=300, bbox_inches="tight")
plt.show()

qc.append({"stage": "final_model_table", "n_rows": int(len(df)),
           "presence": int((df.Label == 1).sum()), "background": int((df.Label == 0).sum()),
           "n_removed": 0})
qc_df = pd.DataFrame(qc)
qc_df.to_csv(OUTPUT_DIR / "dataset_qc_summary.csv", index=False)

predictor_manifest = pd.DataFrame({
    "predictor": FEATURE_COLS,
    "type": ["categorical" if c in CATEGORICAL_FEATURES else "numeric" for c in FEATURE_COLS],
    "source_raster": [RASTERS[c] for c in FEATURE_COLS],
    "harmonized_raster": [str(raster_paths[c]) for c in FEATURE_COLS],
    "resampling": ["nearest" if c in CATEGORICAL_FEATURES else "bilinear" for c in FEATURE_COLS],
})
predictor_manifest.to_csv(OUTPUT_DIR / "predictor_manifest.csv", index=False)

master_export = df.drop(columns=["geometry"], errors="ignore").copy()
MASTER_OUTPUT = OUTPUT_DIR / "SR_GEOAI_Flood_Master_Dataset.csv"
master_export.to_csv(MASTER_OUTPUT, index=False)

print("\n================================================")
print("CORRECTED MASTER DATASET READY")
print("================================================")
print("Samples:", len(df))
print("Final predictors:", len(FEATURE_COLS))
print("Numeric predictors:", len(NUMERIC_FEATURES))
print("Categorical predictors:", len(CATEGORICAL_FEATURES))
print("X shape:", X.shape)
print("Class counts:\n", y.value_counts().sort_index())
print("Target CRS:", TARGET_CRS.to_string())
print("Target resolution:", raster_res)
print("Raster stack directory:", HARMONIZED_RASTER_DIR)
print("Projection distance QC:", projection_distance_qc)
print("Rows with duplicate coordinates (reported, not removed):", int(duplicate_summary.sum()))
print("Excluded predictors (not used for complete-case filtering):", REMOVED_FEATURES)
print("Master dataset:", MASTER_OUTPUT)
display(qc_df)

### Data-loss diagnostic (read-only)

Shows which predictor rasters cause missing values and how many rows the complete-case filter removes only because of the three predictors that are dropped afterwards. It changes nothing.

In [ ]:
# ============================================================
# DATA-LOSS DIAGNOSTIC: old 24-raster filter vs corrected 21-feature filter
# ============================================================
# Read-only comparison. The production modelling table in Cell 2 now filters
# only on the 21 predictors actually used by the models.
_all_cols = list(RASTERS)
_inside = extracted.loc[inside]
_missing = _inside[_all_cols].isna().sum().sort_values(ascending=False)
print("NoData values per reprojected predictor raster (inside common target grid):")
display(_missing[_missing > 0].to_frame("n_missing"))

_cc_old = _inside.dropna(subset=_all_cols + ["Label"])
_cc_new = _inside.dropna(subset=FEATURE_COLS + ["Label"])
_lost_only_excluded = _cc_new.index.difference(_cc_old.index)

def _summ(d):
    return {
        "n": int(len(d)),
        "presence": int((d["Label"] == 1).sum()),
        "background": int((d["Label"] == 0).sum()),
        "presence_share": float((d["Label"] == 1).mean()) if len(d) else np.nan,
    }

filter_comparison = pd.DataFrame({
    "inventory_inside_grid": _summ(_inside),
    "old_complete_case_all_24": _summ(_cc_old),
    "corrected_complete_case_final_21": _summ(_cc_new),
    "valid_rows_lost_only_due_to_excluded_predictors": _summ(_inside.loc[_lost_only_excluded]),
}).T

display(filter_comparison)
filter_comparison.to_csv(OUTPUT_DIR / "old_vs_corrected_filter_diagnostic.csv")
print(f"Rows recovered by filtering only on final predictors: {len(_lost_only_excluded)}")
print("The production `df`, `X`, `y` objects already use the corrected 21-predictor filter.")


In [ ]:
########################################################################################################################
import os, warnings, math, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    cohen_kappa_score, brier_score_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
warnings.filterwarnings("default")

TARGET_COL = "Label"
X_COL, Y_COL = "x", "y"
SEEDS = list(range(11, 26))  # 15 nominal grouped-random holdouts; audit distinct partitions and unique OOF coverage
TEST_SIZE = 0.20
N_SPATIAL_FOLDS = 5  # may be reduced by the geometry/support-only search in Cell 31

RUN_REPEATED_NESTED_SPATIAL_CV = True
SPATIAL_CV_SEEDS = [17, 42, 73] if RUN_REPEATED_NESTED_SPATIAL_CV else [42]
BUFFER_SELECTION_SEEDS = list(SPATIAL_CV_SEEDS)
N_MATCHED_REPEATS = 1  # one prespecified class- and size-matched draw per spatial fold/model

# Prespecified support requirements; the design search never silently relaxes them.
MIN_RETAINED_TRAIN_N = 500
MIN_TRAIN_PER_CLASS_N = 100
MIN_TEST_PER_CLASS_N = 20
SUPPORT_WARNING_TRAIN_N = 500
FDR_Q = 0.05
RANDOM_STATE_FINAL = 42
BLOCK_RANGE_MULTIPLIERS = [0.5, 0.75, 1.0, 1.5, 2.0]
OUTPUT_DIR = Path(os.environ.get("FLOOD_GEOAI_OUTPUT_DIR", "outputs/sr_geoai_outputs_v3_2_2_pixel_grouped")).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RANDOM_STATE_FINAL not in SPATIAL_CV_SEEDS:
    raise ValueError("SPATIAL_CV_SEEDS must include RANDOM_STATE_FINAL for reproducible mapping folds.")

print("Lock-free full-inventory configuration loaded.")
print("Random validation seeds:", len(SEEDS))
print("Spatial partition seeds:", SPATIAL_CV_SEEDS)
print("No geographic test region is reserved.")
print("Runtime profile: 15 random repeats; 3 spatial seeds; one matched draw; checkpointed experiments.")


def _single_threaded_clone(model):
    """Clone an estimator and limit estimator-level threading when supported."""
    m = clone(model)
    params = m.get_params(deep=True) if hasattr(m, "get_params") else {}
    nested_model = getattr(m, "named_steps", {}).get("model") if hasattr(m, "named_steps") else None
    is_logistic = isinstance(nested_model if nested_model is not None else m, LogisticRegression)
    for param_name in ("model__n_jobs", "model__nthread", "model__num_threads", "n_jobs", "nthread"):
        if param_name not in params:
            continue
        if is_logistic and param_name.endswith("n_jobs"):
            continue  # LogisticRegression does not need estimator-level parallelism.
        try:
            m.set_params(**{param_name: 1})
        except (ValueError, TypeError):
            pass
    return m

## 1. Load the master modelling table

**Preferred:** create one row per modelling observation with coordinates, target, and all predictors.

If the original notebook creates `final_df`, leave `MASTER_TABLE = None`.

In [ ]:
# The master table was constructed in Cell 2 from the source inventory and verified raster stack.
# For a true top-to-bottom rerun, use the in-memory dataframe produced there.
#
# Set RELOAD_MASTER_TABLE = True only when intentionally reproducing the modelling stage
# from the exported CSV rather than regenerating the master table.

RELOAD_MASTER_TABLE = False

if RELOAD_MASTER_TABLE:
    p = Path(os.environ.get("FLOOD_GEOAI_MASTER_TABLE", str(OUTPUT_DIR / "SR_GEOAI_Flood_Master_Dataset.csv"))).expanduser()
    if not p.exists():
        raise FileNotFoundError(f"Master table not found: {p}")
    df = pd.read_csv(p)
    print("Reloaded:", p)
else:
    print("Using the validated in-memory modelling table from Cell 2.")

print("Shape:", df.shape)
display(df.head())

In [ ]:
required = {TARGET_COL, X_COL, Y_COL}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=[TARGET_COL, X_COL, Y_COL]).copy()
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

if not set(df[TARGET_COL].unique()).issubset({0, 1}):
    raise ValueError("Label must contain only 0 and 1.")

# Use the fixed scientific predictor list created in Cell 2.
if "FEATURE_COLS" not in globals():
    raise RuntimeError("FEATURE_COLS is missing. Run Cell 2 first.")

FEATURE_COLS = list(FEATURE_COLS)
CATEGORICAL_FEATURES = [c for c in ["Soil_type", "Geology", "LULC"] if c in FEATURE_COLS]
NUMERIC_FEATURES = [c for c in FEATURE_COLS if c not in CATEGORICAL_FEATURES]

for c in CATEGORICAL_FEATURES:
    df[c] = df[c].astype(str)

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()
coords = df[[X_COL, Y_COL]].to_numpy(dtype=float)

print(f"{len(FEATURE_COLS)} scientific predictors.")
print("Categorical:", CATEGORICAL_FEATURES)
print("Numeric:", NUMERIC_FEATURES)
display(y.value_counts().rename("count"))

# 2. Spatial dependence diagnostics

The purpose is to establish whether the observations/predictors have spatial structure that makes random splitting potentially optimistic.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=2).fit(coords)
distances, _ = nn.kneighbors(coords)
nn_dist = distances[:, 1]

print(pd.Series(nn_dist).describe(percentiles=[.1,.25,.5,.75,.9,.95]))

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(nn_dist, bins=40)
ax.set_xlabel("Nearest-neighbour distance")
ax.set_ylabel("Count")
ax.set_title("Hazard-inventory nearest-neighbour distances")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig_nn_distance.png", dpi=300, bbox_inches="tight")
plt.show()

### Moran's I for predictors

Install `libpysal` and `esda` if needed. For the binary event inventory itself, point-level Moran's I should be interpreted cautiously; grid aggregation is preferable for a final paper.

In [ ]:
try:
    from libpysal.weights import KNN
    from esda.moran import Moran
    MORAN_AVAILABLE = True
except Exception:
    MORAN_AVAILABLE = False

# Categorical class codes are not suitable for ordinary numeric Moran's I because
# their integer labels have no meaningful ordering. Aspect is circular, so it is
# represented as sine/cosine components for spatial diagnostics.
spatial_diag_numeric = [c for c in NUMERIC_FEATURES if c not in CATEGORICAL_FEATURES]
diagnostic_df = df[spatial_diag_numeric].copy()

aspect_deg = pd.to_numeric(df["Aspect"], errors="coerce").to_numpy()
diagnostic_df["Aspect_sin"] = np.sin(np.deg2rad(aspect_deg))
diagnostic_df["Aspect_cos"] = np.cos(np.deg2rad(aspect_deg))

def morans_i_knn(values, xy, k=8):
    w = KNN.from_array(np.asarray(xy, float), k=k)
    w.transform = "r"
    mi = Moran(np.asarray(values, float), w, permutations=999)
    return mi.I, mi.p_sim, mi.z_sim

if MORAN_AVAILABLE:
    rows = []
    for col in diagnostic_df.columns:
        vals = np.asarray(diagnostic_df[col], float)
        good = np.isfinite(vals)
        if good.sum() < 30:
            continue
        try:
            I, p, z = morans_i_knn(vals[good], coords[good], k=min(8, int(good.sum()-1)))
            rows.append({"variable": col, "Moran_I": I, "p_value": p, "z_score": z})
        except Exception as e:
            print("Moran failed:", col, e)
    moran_df = pd.DataFrame(rows)
    moran_df.to_csv(OUTPUT_DIR / "moran_predictor_diagnostics.csv", index=False)
    display(moran_df)
else:
    moran_df = pd.DataFrame()
    print("libpysal/esda unavailable; install them for final spatial-autocorrelation diagnostics.")

# 3. Empirical spatial-dependence range

The geographic separation scale should be justified rather than arbitrarily selected. The function below estimates empirical semivariograms and an operational predictor-dependence range.

This is a diagnostic estimate, not a claim that the data follow a particular geostatistical model.

In [ ]:
def empirical_variogram(xy, values, n_bins=25, max_fraction=0.5,
                       max_pairs=200000, seed=42):
    rng = np.random.default_rng(seed)
    xy = np.asarray(xy, float)
    values = np.asarray(values, float)
    good = np.isfinite(values)
    xy, values = xy[good], values[good]

    n = len(values)
    if n < 30:
        raise ValueError("Too few observations for variogram.")

    # Standardise each variable before the variogram so a variable's unit scale
    # does not influence the distance-range selection.
    sd = np.nanstd(values)
    if not np.isfinite(sd) or sd == 0:
        raise ValueError("Variable has zero/invalid variance.")
    values = (values - np.nanmean(values)) / sd

    total_pairs = n * (n - 1) // 2
    if total_pairs <= max_pairs:
        ii, jj = np.triu_indices(n, 1)
    else:
        ii = rng.integers(0, n, max_pairs)
        jj = rng.integers(0, n, max_pairs)
        keep = ii != jj
        ii, jj = ii[keep], jj[keep]

    # Use WGS84 ellipsoidal distances for variogram lags. The regional Albers
    # projection is equal-area but not equidistant, so projected Euclidean
    # distances are not treated as exact geographic separation.
    to_geographic = Transformer.from_crs(TARGET_CRS, CRS.from_epsg(4326), always_xy=True)
    lon, lat = to_geographic.transform(xy[:, 0], xy[:, 1])
    _, _, d = Geod(ellps="WGS84").inv(
        np.asarray(lon)[ii], np.asarray(lat)[ii],
        np.asarray(lon)[jj], np.asarray(lat)[jj]
    )
    d = np.asarray(d, dtype=float)
    gamma = 0.5 * (values[ii] - values[jj]) ** 2

    max_d = np.quantile(d, max_fraction)
    keep = d <= max_d
    d, gamma = d[keep], gamma[keep]

    edges = np.linspace(0, max_d, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    semivar = np.full(n_bins, np.nan)
    bins = np.digitize(d, edges) - 1

    for b in range(n_bins):
        m = bins == b
        if m.any():
            semivar[b] = gamma[m].mean()

    return centers, semivar

def estimate_range(d, gamma):
    good = np.isfinite(gamma)
    d, gamma = d[good], gamma[good]
    if len(gamma) < 5:
        return np.nan
    tail = gamma[-max(3, len(gamma)//5):]
    sill = np.nanmedian(tail)
    idx = np.where(gamma >= 0.95 * sill)[0]
    return float(d[idx[0]]) if len(idx) else np.nan

# Use five continuous variables with straightforward numeric interpretation.
representative = [c for c in [
    "Elevation", "Slope", "TWI", "Soil_moist", "Dist_river"
] if c in NUMERIC_FEATURES]

range_rows = []
fig, ax = plt.subplots(figsize=(9, 6))
for col in representative:
    try:
        vals = pd.to_numeric(df[col], errors="coerce").to_numpy()
        d, g = empirical_variogram(coords, vals)
        r = estimate_range(d, g)
        range_rows.append({"variable": col, "range_m": r})
        ax.plot(d / 1000.0, g, marker="o", ms=3, label=f"{col} (range={r/1000:.1f} km)")
    except Exception as e:
        print("Variogram failed:", col, e)

ax.set_xlabel("Distance (km)")
ax.set_ylabel("Standardised semivariance")
ax.set_title("Empirical semivariograms for continuous spatial predictors")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_variograms.png", dpi=300, bbox_inches="tight")
plt.show()

variogram_df = pd.DataFrame(range_rows)
variogram_df.to_csv(OUTPUT_DIR / "variogram_ranges.csv", index=False)
display(variogram_df)

if variogram_df["range_m"].notna().any():
    SPATIAL_RANGE = float(variogram_df["range_m"].median())
else:
    raise RuntimeError("No valid spatial dependence range could be estimated.")

print(f"Characteristic spatial dependence range: {SPATIAL_RANGE/1000:.2f} km")

# Empirical spatial reference scale. Outer folds are selected later as
# contiguous K-means geographic regions; the range is used to express buffer
# candidates, not to create randomly allocated square blocks.
PRIMARY_BLOCK_SIZE = float(SPATIAL_RANGE)
if not np.isfinite(PRIMARY_BLOCK_SIZE) or PRIMARY_BLOCK_SIZE <= 0:
    raise ValueError(f"Invalid PRIMARY_BLOCK_SIZE: {PRIMARY_BLOCK_SIZE}")

print(f"Empirical reference range: {PRIMARY_BLOCK_SIZE/1000:.2f} km")

In [ ]:
if SPATIAL_RANGE is not None:
    candidate_block_sizes = [SPATIAL_RANGE*m for m in BLOCK_RANGE_MULTIPLIERS]
else:
    extent = max(coords[:,0].ptp(), coords[:,1].ptp())
    candidate_block_sizes = [extent/20, extent/15, extent/10, extent/7.5, extent/5]

print("Candidate reference scales (legacy diagnostic only):", candidate_block_sizes)
print(f"Empirical reference range for publication rerun: {PRIMARY_BLOCK_SIZE/1000:.2f} km")


# 4. Spatial block construction

Whole blocks, not individual observations, are assigned to validation folds.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedGroupKFold


def get_sampling_pixel_keys(xy):
    """Return row/column keys on the common 1-km predictor grid."""
    xy = np.asarray(xy, dtype=float)
    if xy.ndim != 2 or xy.shape[1] != 2 or not np.isfinite(xy).all():
        raise ValueError("xy must be a finite N x 2 coordinate array.")
    if "raster_transform" not in globals():
        raise RuntimeError("The harmonized raster transform must exist before grouping samples.")
    rows, cols = rio.transform.rowcol(raster_transform, xy[:, 0], xy[:, 1])
    return np.column_stack([np.asarray(rows, dtype=np.int64),
                            np.asarray(cols, dtype=np.int64)])


def get_sampling_pixel_group_ids(xy):
    """Integer group labels: observations sampled from one raster cell stay together."""
    keys = get_sampling_pixel_keys(xy)
    _, inverse = np.unique(keys, axis=0, return_inverse=True)
    return inverse.astype(np.int64)


def count_sampling_pixel_key_overlap(keys_a, keys_b):
    """Count common raster row/column keys across independently indexed subsets.

    IMPORTANT: get_sampling_pixel_group_ids() returns compact integer labels whose
    numbers are local to each input array. Those integer labels must never be
    compared across separately subsetted arrays. Compare the stable raster
    (row, column) keys instead.
    """
    a = np.asarray(keys_a, dtype=np.int64).reshape(-1, 2)
    b = np.asarray(keys_b, dtype=np.int64).reshape(-1, 2)
    if len(a) == 0 or len(b) == 0:
        return 0
    structured_dtype = np.dtype([("row", np.int64), ("col", np.int64)])
    a_structured = np.ascontiguousarray(a).view(structured_dtype).reshape(-1)
    b_structured = np.ascontiguousarray(b).view(structured_dtype).reshape(-1)
    return int(np.intersect1d(a_structured, b_structured).size)


def merge_group_constraints(*group_arrays):
    """Return connected-component IDs satisfying every supplied grouping constraint."""
    arrays = [np.asarray(a) for a in group_arrays if a is not None]
    if not arrays:
        raise ValueError("At least one grouping array is required.")
    n = len(arrays[0])
    if any(len(a) != n for a in arrays):
        raise ValueError("All grouping arrays must have equal length.")
    parent = np.arange(n, dtype=np.int64)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb:
            parent[rb] = ra

    for arr in arrays:
        seen = {}
        for i, value in enumerate(arr.tolist()):
            # Include type in the key so integer 1 and string "1" remain distinct.
            key = (type(value).__name__, repr(value))
            if key in seen:
                union(i, seen[key])
            else:
                seen[key] = i
    roots = np.asarray([find(i) for i in range(n)], dtype=np.int64)
    _, inverse = np.unique(roots, return_inverse=True)
    return inverse.astype(np.int64)


def assign_square_blocks(xy, block_size):
    xy = np.asarray(xy, float)
    bx = np.floor(xy[:, 0] / block_size).astype(int)
    by = np.floor(xy[:, 1] / block_size).astype(int)
    return np.array([f"{a}_{b}" for a, b in zip(bx, by)])


def make_spatial_splits(xy, y, block_size, n_splits=5, seed=42):
    pixel_groups = get_sampling_pixel_group_ids(xy)
    block_groups = assign_square_blocks(xy, block_size)
    groups = merge_group_constraints(pixel_groups, block_groups)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(cv.split(np.zeros(len(y)), y, groups=groups))
    return splits, groups


def assign_contiguous_spatial_regions(xy, n_regions=5, seed=42, n_init=30):
    """K-means regions fitted to raster-cell centroids, weighted by cell sample count.

    All observations sampled from one common 1-km raster cell receive the same
    geographic-region assignment. K-means uses coordinates only, not labels or
    model performance. Buffer distances remain geodesic.
    """
    xy = np.asarray(xy, dtype=float)
    if xy.ndim != 2 or xy.shape[1] != 2 or len(xy) < n_regions:
        raise ValueError("Insufficient/invalid coordinates for requested spatial regions.")
    if not np.isfinite(xy).all():
        raise ValueError("Coordinates contain non-finite values.")

    pixel_groups = get_sampling_pixel_group_ids(xy)
    _, inverse = np.unique(pixel_groups, return_inverse=True)
    n_groups = int(inverse.max()) + 1
    group_counts = np.bincount(inverse, minlength=n_groups)
    group_centers = np.zeros((n_groups, 2), dtype=float)
    np.add.at(group_centers, inverse, xy)
    group_centers /= group_counts[:, None]

    if n_groups < n_regions:
        raise ValueError(f"Only {n_groups} distinct raster cells for {n_regions} regions.")
    km = KMeans(n_clusters=n_regions, random_state=int(seed), n_init=n_init,
                algorithm="lloyd")
    region_by_group = km.fit_predict(group_centers, sample_weight=group_counts)
    groups = region_by_group[inverse].astype(int)
    counts = np.bincount(groups, minlength=n_regions)
    if np.any(counts == 0):
        raise RuntimeError("K-means generated an empty geographic region.")
    return groups, km.cluster_centers_, counts


def make_random_group_holdout_split(y, xy, test_size=0.20, seed=42, max_attempts=5):
    """Approximate stratified holdout while keeping every 1-km raster cell intact."""
    yv = np.asarray(y, dtype=int)
    xy = np.asarray(xy, dtype=float)
    if len(yv) != len(xy):
        raise ValueError("y and xy lengths differ in grouped random holdout.")
    if not (0 < float(test_size) < 1):
        raise ValueError("test_size must be between zero and one.")
    pixel_groups = get_sampling_pixel_group_ids(xy)
    n_groups = len(np.unique(pixel_groups))
    k = max(2, int(round(1.0 / float(test_size))))
    k = min(k, n_groups)
    if k < 2:
        raise ValueError("Too few raster-cell groups for random holdout.")

    best = None
    overall_prevalence = float(np.mean(yv))
    target_n = int(round(len(yv) * float(test_size)))
    for attempt in range(int(max_attempts)):
        cv = StratifiedGroupKFold(n_splits=k, shuffle=True,
                                  random_state=int(seed) + attempt)
        try:
            candidate_splits = list(cv.split(np.zeros(len(yv)), yv, groups=pixel_groups))
        except ValueError:
            continue
        for fold_id, (tr, te) in enumerate(candidate_splits):
            if len(np.unique(yv[tr])) != 2 or len(np.unique(yv[te])) != 2:
                continue
            if np.intersect1d(np.unique(pixel_groups[tr]),
                              np.unique(pixel_groups[te])).size:
                raise AssertionError("Raster-pixel group crossed a random train/test boundary.")
            size_error = abs(len(te) - target_n) / max(len(yv), 1)
            prevalence_error = abs(float(np.mean(yv[te])) - overall_prevalence)
            score = size_error + 0.5 * prevalence_error
            candidate = (score, tr.copy(), te.copy(), attempt, fold_id)
            if best is None or score < best[0]:
                best = candidate
        # Continue through all attempts to choose the closest 20% stratified fold.
    if best is None:
        raise RuntimeError("Could not construct a valid group-disjoint random holdout.")
    _, tr, te, attempt, fold_id = best
    metadata = {
        "split_method": "StratifiedGroupKFold_one_fold",
        "sampling_unit": "1-km raster cell",
        "attempt": int(attempt),
        "selected_fold": int(fold_id),
        "target_test_size": float(test_size),
        "realized_test_fraction": float(len(te) / len(yv)),
        "n_train_pixel_groups": int(len(np.unique(pixel_groups[tr]))),
        "n_test_pixel_groups": int(len(np.unique(pixel_groups[te]))),
    }
    return tr, te, metadata


def sample_group_disjoint_class_matched_indices(
    eligible_indices, y, coords, target_pos, target_neg, rng,
    optional_multigroup_probability=0.30
):
    """Draw exact class/size-matched rows without splitting any raster-cell group.

    It first selects complete multi-observation raster-cell groups only when
    needed (or randomly when feasible), then fills remaining class counts from
    singleton-cell groups. It raises rather than silently relaxing exact matching.
    """
    eligible_indices = np.asarray(eligible_indices, dtype=int)
    yv = np.asarray(y, dtype=int)
    coords = np.asarray(coords, dtype=float)
    if len(eligible_indices) == 0:
        raise ValueError("No eligible observations for grouped size matching.")

    local_groups = get_sampling_pixel_group_ids(coords[eligible_indices])
    group_members = {}
    for gid in np.unique(local_groups):
        group_members[int(gid)] = eligible_indices[local_groups == gid]

    single_pos, single_neg, multi = [], [], []
    for gid, members in group_members.items():
        labels = yv[members]
        n_pos = int(np.sum(labels == 1))
        n_neg = int(np.sum(labels == 0))
        record = {"gid": gid, "members": members, "pos": n_pos, "neg": n_neg}
        if len(members) == 1:
            (single_pos if n_pos == 1 else single_neg).append(int(members[0]))
        else:
            multi.append(record)

    remaining_pos = int(target_pos)
    remaining_neg = int(target_neg)
    single_pos_n, single_neg_n = len(single_pos), len(single_neg)
    selected_multi = []
    unused_multi = list(multi)

    # Force complete multi-row groups only where singleton groups alone cannot
    # supply the exact target class counts.
    while remaining_pos > single_pos_n or remaining_neg > single_neg_n:
        deficit_pos = max(0, remaining_pos - single_pos_n)
        deficit_neg = max(0, remaining_neg - single_neg_n)
        candidates = []
        for rec in unused_multi:
            if rec["pos"] > remaining_pos or rec["neg"] > remaining_neg:
                continue
            contribution = (
                min(rec["pos"], deficit_pos) / max(deficit_pos, 1)
                + min(rec["neg"], deficit_neg) / max(deficit_neg, 1)
            )
            if contribution > 0:
                candidates.append((contribution, rec))
        if not candidates:
            raise ValueError(
                "Exact class- and size-matching is infeasible without splitting a 1-km raster-cell group. "
                f"Remaining targets: presence={remaining_pos}, background={remaining_neg}; "
                f"available singleton groups: presence={single_pos_n}, background={single_neg_n}."
            )
        top_score = max(x[0] for x in candidates)
        top = [rec for score, rec in candidates if np.isclose(score, top_score)]
        chosen = top[int(rng.integers(0, len(top)))]
        selected_multi.append(chosen)
        unused_multi.remove(chosen)
        remaining_pos -= chosen["pos"]
        remaining_neg -= chosen["neg"]

    # Add optional complete groups to avoid an artificial singleton-only reference,
    # but keep the residual counts exactly fillable by singleton groups.
    if unused_multi:
        order = rng.permutation(len(unused_multi))
        for j in order:
            rec = unused_multi[int(j)]
            if rng.random() > float(optional_multigroup_probability):
                continue
            rp = remaining_pos - rec["pos"]
            rn = remaining_neg - rec["neg"]
            if (rp >= 0 and rn >= 0 and rp <= single_pos_n and rn <= single_neg_n):
                selected_multi.append(rec)
                remaining_pos, remaining_neg = rp, rn

    if remaining_pos > single_pos_n or remaining_neg > single_neg_n:
        raise AssertionError("Grouped matching feasibility check failed unexpectedly.")
    chosen_pos = rng.choice(np.asarray(single_pos, dtype=int),
                            size=remaining_pos, replace=False) if remaining_pos else np.empty(0, dtype=int)
    chosen_neg = rng.choice(np.asarray(single_neg, dtype=int),
                            size=remaining_neg, replace=False) if remaining_neg else np.empty(0, dtype=int)
    chosen_multi_idx = [idx for rec in selected_multi for idx in rec["members"]]
    selected = np.concatenate([np.asarray(chosen_multi_idx, dtype=int),
                               np.asarray(chosen_pos, dtype=int),
                               np.asarray(chosen_neg, dtype=int)])
    rng.shuffle(selected)

    if len(selected) != int(target_pos + target_neg):
        raise AssertionError("Grouped matched sample does not have the requested total size.")
    if int(np.sum(yv[selected] == 1)) != int(target_pos):
        raise AssertionError("Grouped matched sample does not preserve presence count.")
    if int(np.sum(yv[selected] == 0)) != int(target_neg):
        raise AssertionError("Grouped matched sample does not preserve background count.")
    # Validate complete-cell selection efficiently (linearithmic, not pairwise).
    eligible_keys = get_sampling_pixel_keys(coords[eligible_indices])
    selected_keys = get_sampling_pixel_keys(coords[selected])
    unique_eligible, eligible_counts = np.unique(eligible_keys, axis=0, return_counts=True)
    unique_selected, selected_counts = np.unique(selected_keys, axis=0, return_counts=True)
    eligible_count_map = {tuple(k): int(v) for k, v in zip(unique_eligible, eligible_counts)}
    for key, count in zip(unique_selected, selected_counts):
        if int(count) != eligible_count_map.get(tuple(key), -1):
            raise AssertionError("A multi-observation raster-cell group was only partially sampled.")
    return selected

## 4. Full-inventory modelling population

All complete observations are retained for the validation experiments. There is no locked geographic test and no locked-test exclusion buffer. The outer buffered spatial CV itself creates geographically separated training/test folds; observations excluded from a fold's training set because they lie inside that fold's buffer remain eligible for test assignment in other folds and remain part of the full modelling population.

In [ ]:
# ============================================================
# FULL-INVENTORY MODELLING POPULATION — NO LOCKED TEST
# ============================================================
# Each 1-km raster cell is an indivisible sampling unit because all predictors
# were extracted from the common harmonized raster grid. This is stricter than
# grouping exact-coordinate duplicates only.

required = {TARGET_COL, X_COL, Y_COL, *FEATURE_COLS}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Full corrected modelling table is missing columns: {sorted(missing)}")

df = df.copy().reset_index(drop=True)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL, X_COL, Y_COL]).copy().reset_index(drop=True)
df[TARGET_COL] = df[TARGET_COL].astype(int)
if not set(df[TARGET_COL].unique()).issubset({0, 1}):
    raise ValueError("Label must contain only 0 and 1.")

missing_final = df[FEATURE_COLS].isna().sum()
if int(missing_final.sum()) > 0:
    raise ValueError(
        "Final 21-predictor table still contains missing values. Resolve the source-data issue "
        "before validation; this cell will not silently change the inventory.\n"
        + missing_final[missing_final > 0].to_string()
    )

for c in CATEGORICAL_FEATURES:
    df[c] = df[c].astype(str)

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].astype(int).copy()
coords = df[[X_COL, Y_COL]].to_numpy(dtype=float)
if len(np.unique(y)) != 2:
    raise ValueError("Full inventory must contain both presence and background classes.")
if not np.isfinite(coords).all():
    raise ValueError("Coordinates contain non-finite values.")

# Stable raster-cell identifiers from the exact common-grid transform.
SAMPLE_PIXEL_KEYS = get_sampling_pixel_keys(coords)
SAMPLE_PIXEL_GROUP_IDS = get_sampling_pixel_group_ids(coords)
df["sample_pixel_row"] = SAMPLE_PIXEL_KEYS[:, 0]
df["sample_pixel_col"] = SAMPLE_PIXEL_KEYS[:, 1]
df["sample_pixel_group_id"] = SAMPLE_PIXEL_GROUP_IDS

# Audit both raster-cell groups and exact-coordinate duplicates.
work_audit = df.copy()
work_audit["_pixel_key"] = list(map(tuple, SAMPLE_PIXEL_KEYS.tolist()))
pixel_audit = (
    work_audit.groupby("_pixel_key", sort=False)
    .agg(
        sample_pixel_row=("sample_pixel_row", "first"),
        sample_pixel_col=("sample_pixel_col", "first"),
        n_observations=(TARGET_COL, "size"),
        n_presence=(TARGET_COL, "sum"),
        n_background=(TARGET_COL, lambda s: int((s == 0).sum())),
        n_distinct_labels=(TARGET_COL, "nunique"),
        inventory_record_ids=("inventory_record_id", lambda s: ";".join(map(str, s.tolist()))
                              if "inventory_record_id" in work_audit.columns else ""),
    )
    .reset_index(drop=True)
)
pixel_audit["is_multi_observation_pixel"] = pixel_audit["n_observations"] > 1
pixel_audit["has_mixed_labels"] = pixel_audit["n_distinct_labels"] > 1
pixel_audit.to_csv(OUTPUT_DIR / "raster_pixel_group_audit.csv", index=False)

mixed_pixel_audit = pixel_audit[pixel_audit["has_mixed_labels"]].copy()
mixed_pixel_audit.to_csv(OUTPUT_DIR / "mixed_label_raster_pixel_groups.csv", index=False)
mixed_pixel_keys = set(
    zip(mixed_pixel_audit["sample_pixel_row"].astype(int),
        mixed_pixel_audit["sample_pixel_col"].astype(int))
)
mixed_record_mask = np.asarray(
    [(int(r), int(c)) in mixed_pixel_keys for r, c in SAMPLE_PIXEL_KEYS], dtype=bool
)
mixed_pixel_records = df.loc[mixed_record_mask].drop(columns=["geometry"], errors="ignore").copy()
mixed_pixel_records.to_csv(
    OUTPUT_DIR / "mixed_label_raster_pixel_records.csv", index=False
)

coord_audit = (
    df.groupby([X_COL, Y_COL], sort=False)
    .agg(
        n_observations=(TARGET_COL, "size"),
        n_distinct_labels=(TARGET_COL, "nunique"),
        n_presence=(TARGET_COL, "sum"),
        inventory_record_ids=("inventory_record_id", lambda s: ";".join(map(str, s.tolist()))
                              if "inventory_record_id" in df.columns else ""),
    )
    .reset_index()
)
coord_audit[coord_audit["n_observations"] > 1].to_csv(
    OUTPUT_DIR / "duplicate_coordinate_group_audit.csv", index=False
)

N_SAMPLE_PIXEL_GROUPS = int(len(np.unique(SAMPLE_PIXEL_GROUP_IDS)))
N_MULTI_OBSERVATION_PIXELS = int(pixel_audit["is_multi_observation_pixel"].sum())
N_MIXED_LABEL_PIXEL_GROUPS = int(len(mixed_pixel_audit))
N_DUPLICATE_COORDINATE_GROUPS = int((coord_audit["n_observations"] > 1).sum())

# Export the modelling inventory with its immutable 1-km raster-cell group ID.
# The original master export from Cell 3 remains untouched for lineage comparison.
df.drop(columns=["geometry"], errors="ignore").to_csv(
    OUTPUT_DIR / "SR_GEOAI_Flood_Master_Dataset_Grouped.csv", index=False
)

# Reviewed mixed-label policy for the current source inventory:
# The sole mixed-label raster cell contains exactly two records (one presence,
# one background) with identical projected coordinates and identical values for
# all 21 predictors. The source table has no event/date/provenance field that
# would establish which label should be changed. Therefore, retain both rows
# without relabeling, but keep them in the SAME raster-cell group in every
# outer, inner, calibration, spatial, and size-matched split.
#
# This is not a claim that the labels are both correct. It is an explicit,
# auditable treatment of one unresolved label conflict. A sensitivity analysis
# excluding BOTH rows is recommended before manuscript submission.
ALLOW_MIXED_LABEL_PIXEL_GROUPS = True

if N_MIXED_LABEL_PIXEL_GROUPS:
    reviewed_pattern_ok = (
        N_MIXED_LABEL_PIXEL_GROUPS == 1
        and len(mixed_pixel_records) == 2
        and set(mixed_pixel_records[TARGET_COL].astype(int).tolist()) == {0, 1}
        and all(
            mixed_pixel_records[col].nunique(dropna=False) == 1
            for col in [X_COL, Y_COL, *FEATURE_COLS]
        )
    )
    if not ALLOW_MIXED_LABEL_PIXEL_GROUPS or not reviewed_pattern_ok:
        raise RuntimeError(
            "Mixed-label raster-cell audit does not match the single reviewed "
            "two-record, identical-coordinate, identical-predictor conflict. "
            "Do not proceed automatically. Inspect "
            f"{OUTPUT_DIR / 'mixed_label_raster_pixel_groups.csv'} and "
            f"{OUTPUT_DIR / 'mixed_label_raster_pixel_records.csv'}, resolve the "
            "source issue, or update the explicit policy after review."
        )
    print(
        "WARNING: retaining one reviewed mixed-label raster-cell group "
        "(two records; one presence and one background; identical coordinates "
        "and predictor values). Both records remain in the same sampling group; "
        "neither label is changed. Run the both-records-excluded sensitivity "
        "analysis before final manuscript claims."
    )

# Explicit aliases for diagnostics and mapping; full inventory only.
df_all = df.copy()
X_all = X.copy()
y_all = y.copy()
coords_all = coords.copy()
FULL_INVENTORY_IDX = np.arange(len(df), dtype=int)

print("FULL-INVENTORY MODELLING POPULATION")
print("Observations:", len(df))
print("Presence:", int((y == 1).sum()))
print("Background:", int((y == 0).sum()))
print("Predictors:", len(FEATURE_COLS))
print("Distinct 1-km raster-cell groups:", N_SAMPLE_PIXEL_GROUPS)
print("Multi-observation raster cells:", N_MULTI_OBSERVATION_PIXELS)
print("Exact duplicate-coordinate groups:", N_DUPLICATE_COORDINATE_GROUPS)
print("Mixed-label raster-cell groups:", N_MIXED_LABEL_PIXEL_GROUPS)
print("No locked test region or locked-region exclusion buffer is used.")
display(y.value_counts().rename("count"))
display(pixel_audit[pixel_audit["is_multi_observation_pixel"]].head(20))

# 6. Leakage-safe GeoAI model set

Replace/add your exact XGBoost, LightGBM, CatBoost, MLP and CNN models from the original notebook after the validation engine is verified.

The essential rule is that every data-learning preprocessing step must occur inside the training fold.

## 6a. MLP / ANN deep-learning benchmark

A small multilayer perceptron is included as the neural-network benchmark for the
static tabular predictor matrix. It is registered in `MODELS` and in the shared
hyperparameter search space, so it follows the same repeated random and buffered
nested-spatial validation, calibration and descriptive model-ranking workflow as
the other candidate models. Standardisation is inside the pipeline and is fitted
on each training partition only.

Sequence models (LSTM / Bi-LSTM) are intentionally not included: the predictors
form a static tabular vector, not an ordered temporal sequence.


In [ ]:
# ============================================================
# MLP / ANN benchmark for the static tabular predictor matrix
# ============================================================
# Small fully-connected network with the training recipe requested for the
# deep-learning benchmark: Adam, binary cross-entropy, dropout, weight decay,
# early stopping, learning-rate reduction on plateau and fixed random seeds.
#
# Implemented in NumPy as a scikit-learn estimator so that it is validated by
# exactly the same code path as the other models (clone / Pipeline /
# RandomizedSearchCV / nested spatial CV) and needs no extra dependency.
#
# Leakage control:
#   * Standardisation is NOT done here; the model is wrapped in a Pipeline with a
#     StandardScaler, so the scaler is fitted on the training fold only.
#   * Early stopping is disabled by default. Epoch count is selected by the
#     development-set tuning procedure; no random internal validation slice is used.
#   * All randomness comes from `random_state`.

import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_array, check_is_fitted


def _mlp_sigmoid(z):
    """Numerically stable logistic function (plain NumPy, so it pickles cleanly when the
    class is shipped to joblib/loky workers from a notebook)."""
    e = np.exp(-np.abs(z))
    return np.where(z >= 0, 1.0 / (1.0 + e), e / (1.0 + e))


class NumpyMLPClassifier(ClassifierMixin, BaseEstimator):
    """Binary MLP: [Dense-ReLU-Dropout] x (L-1) -> Dense-ReLU -> Dense(1)-sigmoid.

    Dropout is applied after every hidden layer except the last one, e.g.
    (64, 32, 16) -> Dense64-ReLU-Dropout, Dense32-ReLU-Dropout, Dense16-ReLU, Dense1.

    Parameters
    ----------
    hidden_layer_sizes : tuple of int
    dropout            : drop probability for the hidden layers that use dropout
    learning_rate      : initial Adam learning rate
    batch_size         : mini-batch size
    weight_decay       : L2 penalty on the weight matrices, added to the gradient
                         (same convention as torch.optim.Adam(weight_decay=...))
    max_epochs         : maximum number of epochs
    patience           : early-stopping patience (epochs without validation-loss gain)
    lr_reduce_factor / lr_reduce_patience / min_lr : ReduceLROnPlateau behaviour
    validation_fraction: share of the training data used for early stopping
    random_state       : seed for initialisation, shuffling, dropout and the split
    """

    def __init__(self, hidden_layer_sizes=(64, 32, 16), dropout=0.2,
                 learning_rate=1e-3, batch_size=32, weight_decay=0.0,
                 max_epochs=150, patience=15, lr_reduce_factor=0.5,
                 lr_reduce_patience=5, min_lr=1e-6, validation_fraction=0.15, early_stopping=False,
                 random_state=42):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.weight_decay = weight_decay
        self.max_epochs = max_epochs
        self.patience = patience
        self.lr_reduce_factor = lr_reduce_factor
        self.lr_reduce_patience = lr_reduce_patience
        self.min_lr = min_lr
        self.validation_fraction = validation_fraction
        self.early_stopping = early_stopping
        self.random_state = random_state

    # ---------------------------------------------------------------- helpers
    def _init_params(self, n_in, rng):
        sizes = [n_in] + [int(h) for h in self.hidden_layer_sizes] + [1]
        W, b = [], []
        for i in range(len(sizes) - 1):
            fan_in, fan_out = sizes[i], sizes[i + 1]
            if i < len(sizes) - 2:      # ReLU layers: He initialisation
                scale = np.sqrt(2.0 / fan_in)
            else:                        # output layer: Glorot initialisation
                scale = np.sqrt(2.0 / (fan_in + fan_out))
            W.append(rng.normal(0.0, scale, size=(fan_in, fan_out)))
            b.append(np.zeros(fan_out))
        return W, b

    def _forward(self, X, W, b, rng=None):
        """Return logits and the cache needed for back-propagation.
        Dropout (inverted) is active only when an rng is supplied."""
        n_layers = len(W)
        acts, masks, pre = [X], [], []
        a = X
        for i in range(n_layers - 1):
            z = a @ W[i] + b[i]
            pre.append(z)
            a = np.maximum(z, 0.0)
            mask = None
            if rng is not None and self.dropout > 0.0 and i < n_layers - 2:
                keep = 1.0 - self.dropout
                mask = (rng.random(a.shape) < keep) / keep
                a = a * mask
            masks.append(mask)
            acts.append(a)
        logits = (a @ W[-1] + b[-1]).ravel()
        return logits, (acts, masks, pre)

    def _backward(self, yb, logits, cache, W):
        """Gradients of mean BCE (+ L2 on weights) w.r.t. all parameters."""
        acts, masks, pre = cache
        n = len(yb)
        n_layers = len(W)
        gW, gb = [None] * n_layers, [None] * n_layers
        delta = ((_mlp_sigmoid(logits) - yb) / n).reshape(-1, 1)   # dL/dlogit
        for i in range(n_layers - 1, -1, -1):
            gW[i] = acts[i].T @ delta + self.weight_decay * W[i]
            gb[i] = delta.sum(axis=0)
            if i > 0:
                delta = delta @ W[i].T
                if masks[i - 1] is not None:
                    delta = delta * masks[i - 1]
                delta = delta * (pre[i - 1] > 0.0)
        return gW, gb

    @staticmethod
    def _bce_from_logits(yb, logits):
        # numerically stable mean binary cross-entropy
        return float(np.mean(np.logaddexp(0.0, logits) - yb * logits))

    # -------------------------------------------------------------------- API
    def fit(self, X, y):
        X = check_array(X, dtype=np.float64)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        if len(self.classes_) != 2:
            raise ValueError("NumpyMLPClassifier supports binary targets only.")
        yb = (y == self.classes_[1]).astype(np.float64)
        self.n_features_in_ = X.shape[1]

        rng = np.random.RandomState(self.random_state)
        idx = np.arange(len(yb))
        if self.early_stopping:
            n_val = int(round(self.validation_fraction * len(idx)))
            if n_val >= 2 and len(idx) - n_val >= 2:
                try:
                    tr_idx, va_idx = train_test_split(
                        idx, test_size=n_val, stratify=yb, random_state=self.random_state)
                except ValueError as exc:
                    raise ValueError(
                        "MLP early stopping was requested but a valid stratified validation split "
                        "could not be constructed. Use a larger training sample or disable early stopping."
                    ) from exc
            else:
                raise ValueError("Not enough MLP training observations for the requested early-stopping split.")
            Xtr, ytr, Xva, yva = X[tr_idx], yb[tr_idx], X[va_idx], yb[va_idx]
        else:
            # Fixed-epoch training avoids an unblocked random validation slice.
            Xtr, ytr = X, yb
            Xva, yva = None, None

        W, b = self._init_params(X.shape[1], rng)
        mW = [np.zeros_like(w) for w in W]; vW = [np.zeros_like(w) for w in W]
        mb = [np.zeros_like(x) for x in b]; vb = [np.zeros_like(x) for x in b]
        beta1, beta2, eps = 0.9, 0.999, 1e-8
        lr, step = float(self.learning_rate), 0

        best_loss, best_W, best_b = np.inf, None, None
        since_best, since_lr = 0, 0
        history = []
        bs = max(1, int(self.batch_size))

        for epoch in range(int(self.max_epochs)):
            order = rng.permutation(len(ytr))
            for s in range(0, len(order), bs):
                bi = order[s:s + bs]
                logits, cache = self._forward(Xtr[bi], W, b, rng=rng)
                gW, gb = self._backward(ytr[bi], logits, cache, W)
                step += 1
                c1, c2 = 1.0 - beta1 ** step, 1.0 - beta2 ** step
                for i in range(len(W)):
                    mW[i] = beta1 * mW[i] + (1 - beta1) * gW[i]
                    vW[i] = beta2 * vW[i] + (1 - beta2) * gW[i] ** 2
                    W[i] -= lr * (mW[i] / c1) / (np.sqrt(vW[i] / c2) + eps)
                    mb[i] = beta1 * mb[i] + (1 - beta1) * gb[i]
                    vb[i] = beta2 * vb[i] + (1 - beta2) * gb[i] ** 2
                    b[i] -= lr * (mb[i] / c1) / (np.sqrt(vb[i] / c2) + eps)

            if self.early_stopping:
                val_logits, _ = self._forward(Xva, W, b, rng=None)
                monitored_loss = self._bce_from_logits(yva, val_logits)
                history.append(monitored_loss)
                if monitored_loss < best_loss - 1e-6:
                    best_loss, since_best, since_lr = monitored_loss, 0, 0
                    best_W = [w.copy() for w in W]; best_b = [x.copy() for x in b]
                else:
                    since_best += 1; since_lr += 1
                    if since_lr >= self.lr_reduce_patience:
                        lr = max(lr * self.lr_reduce_factor, self.min_lr)
                        since_lr = 0
                    if since_best >= self.patience:
                        break
            else:
                # No held-out slice is used inside this estimator. Epoch count is
                # selected by the development-set spatial tuning procedure.
                train_logits, _ = self._forward(Xtr, W, b, rng=None)
                history.append(self._bce_from_logits(ytr, train_logits))

        if self.early_stopping and best_W is not None:
            self.W_, self.b_ = best_W, best_b
        else:
            self.W_, self.b_ = W, b
        self.n_epochs_ = len(history)
        self.best_val_loss_ = best_loss if self.early_stopping else np.nan
        self.val_loss_history_ = history
        return self

    def predict_proba(self, X):
        check_is_fitted(self, ["W_", "b_"])
        X = check_array(X, dtype=np.float64)
        logits, _ = self._forward(X, self.W_, self.b_, rng=None)
        p1 = _mlp_sigmoid(logits)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X):
        return self.classes_[(self.predict_proba(X)[:, 1] >= 0.5).astype(int)]


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Compatibility across scikit-learn releases.
try:
    OHE = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    OHE = OneHotEncoder(handle_unknown="ignore", sparse=False)

if not NUMERIC_FEATURES and not CATEGORICAL_FEATURES:
    raise RuntimeError("No predictors available.")

PREPROCESSOR = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OHE, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

MODELS = {
    "Logistic Regression": Pipeline([
        ("prep", PREPROCESSOR),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight=None, random_state=42))
    ]),
    "SVM-RBF": Pipeline([
        ("prep", PREPROCESSOR),
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=False, class_weight=None, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("prep", PREPROCESSOR),
        ("model", RandomForestClassifier(
            n_estimators=500, max_features="sqrt", min_samples_leaf=2,
            class_weight=None, n_jobs=-1, random_state=42
        ))
    ])
}

try:
    from xgboost import XGBClassifier
    MODELS["XGBoost"] = Pipeline([
        ("prep", PREPROCESSOR),
        ("model", XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=.05,
            subsample=.8, colsample_bytree=.8, eval_metric="logloss",
            tree_method="hist", n_jobs=-1, random_state=42
        ))
    ])
except Exception as e:
    print("XGBoost unavailable:", e)

try:
    from lightgbm import LGBMClassifier
    MODELS["LightGBM"] = Pipeline([
        ("prep", PREPROCESSOR),
        ("model", LGBMClassifier(
            n_estimators=500, learning_rate=.05, num_leaves=31,
            subsample=.8, colsample_bytree=.8, class_weight=None,
            n_jobs=-1, verbosity=-1, random_state=42
        ))
    ])
except Exception as e:
    print("LightGBM unavailable:", e)

MODELS["MLP"] = Pipeline([
    ("prep", PREPROCESSOR),
    ("scaler", StandardScaler()),
    ("model", NumpyMLPClassifier(
        hidden_layer_sizes=(64, 32, 16),
        dropout=0.2,
        learning_rate=1e-3,
        batch_size=32,
        weight_decay=1e-4,
        max_epochs=150,
        patience=15,
        random_state=42
    ))
])

print("Validated model families:", list(MODELS))
print("Scientific predictors:", len(FEATURE_COLS))
print("Encoded feature count will be learned from the training folds only.")


# 7. Metrics and calibration

In [ ]:
from scipy.special import expit, logit
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression


def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.clip(np.asarray(prob, dtype=float), 0.0, 1.0)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = ((prob >= edges[i]) & (prob <= edges[i + 1])) if i == n_bins - 1 else (
            (prob >= edges[i]) & (prob < edges[i + 1]))
        if m.any():
            ece += m.mean() * abs(y_true[m].mean() - prob[m].mean())
    return float(ece)


def calibration_intercept_slope(y_true, prob):
    """Logistic calibration y ~ intercept + slope*logit(score)."""
    from scipy.optimize import minimize
    y_true = np.asarray(y_true, float)
    p = np.clip(np.asarray(prob, float), 1e-6, 1 - 1e-6)
    z = np.log(p / (1 - p))
    def objective(beta):
        eta = beta[0] + beta[1] * z
        return float(np.sum(np.logaddexp(0, eta) - y_true * eta))
    res = minimize(objective, x0=np.array([0.0, 1.0]), method="BFGS")
    if not res.success:
        return np.nan, np.nan
    return float(res.x[0]), float(res.x[1])


def _row_slice(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else np.asarray(X)[idx]


def _raw_model_score(estimator, X):
    """Uncalibrated 1-D score; SVM decision function is preferred over internal CV probabilities."""
    if hasattr(estimator, "decision_function"):
        score = np.asarray(estimator.decision_function(X), dtype=float).reshape(-1)
        return score
    if hasattr(estimator, "predict_proba"):
        p = np.asarray(estimator.predict_proba(X), dtype=float)[:, 1]
        return logit(np.clip(p, 1e-6, 1 - 1e-6))
    raise AttributeError("Estimator must expose decision_function or predict_proba.")


class CrossFittedCalibratedEstimator(ClassifierMixin, BaseEstimator):
    """Fit a base model and calibrate its raw score from out-of-fold predictions.

    `calibration_splits` must be built only from the current training partition.
    This legacy wrapper is retained only for the fixed-hyperparameter buffer-distance
    sensitivity experiment. The primary random and buffered-spatial experiments use
    `fit_nested_calibrated_model`, which retunes hyperparameters inside each
    calibration-training fold. The outer validation labels remain untouched.
    """
    def __init__(self, estimator, calibration_splits, calibration_C=1e6):
        self.estimator = estimator
        self.calibration_splits = calibration_splits
        self.calibration_C = calibration_C

    def fit(self, X, y):
        yv = np.asarray(y, dtype=int)
        if len(np.unique(yv)) != 2:
            raise ValueError("Cross-fitted calibration requires both classes in the training partition.")
        scores_sum = np.zeros(len(yv), dtype=float)
        score_count = np.zeros(len(yv), dtype=int)
        for fold, (tr, va) in enumerate(self.calibration_splits):
            if len(np.unique(yv[tr])) != 2 or len(np.unique(yv[va])) != 2:
                raise ValueError(f"Calibration fold {fold} does not contain both classes.")
            est = clone(self.estimator)
            est.fit(_row_slice(X, tr), yv[tr])
            raw = _raw_model_score(est, _row_slice(X, va))
            scores_sum[va] += raw
            score_count[va] += 1
        if np.any(score_count == 0):
            missing = np.where(score_count == 0)[0]
            raise ValueError(f"Calibration folds did not produce out-of-fold scores for {len(missing)} rows.")
        oof_score = scores_sum / score_count
        self.calibrator_ = LogisticRegression(C=float(self.calibration_C), solver="lbfgs", max_iter=2000)
        self.calibrator_.fit(oof_score.reshape(-1, 1), yv)
        self.estimator_ = clone(self.estimator).fit(X, yv)
        self.classes_ = np.asarray([0, 1], dtype=int)
        self.n_features_in_ = getattr(self.estimator_, "n_features_in_", None)
        self.calibration_oof_score_ = oof_score
        self.calibration_fold_count_ = score_count
        return self

    def predict_proba(self, X):
        raw = _raw_model_score(self.estimator_, X)
        return self.calibrator_.predict_proba(raw.reshape(-1, 1))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def calculate_metrics(y_true, prob, threshold=0.5):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    pred = (prob >= threshold).astype(int)
    y_true = np.asarray(y_true, dtype=int)
    prevalence = float(np.mean(y_true))
    brier = brier_score_loss(y_true, prob)
    brier_ref = prevalence * (1.0 - prevalence)
    cal_intercept, cal_slope = calibration_intercept_slope(y_true, prob)
    return {
        "ROC_AUC": roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan,
        "PR_AUC": average_precision_score(y_true, prob),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, pred),
        "Kappa": cohen_kappa_score(y_true, pred),
        "Brier": brier,
        "Brier_skill": 1.0 - brier / brier_ref if brier_ref > 0 else np.nan,
        "ECE": expected_calibration_error(y_true, prob),
        "Calibration_intercept": cal_intercept,
        "Calibration_slope": cal_slope,
        "Prevalence": prevalence,
    }


def get_probability(model, Xt):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(Xt))[:, 1]
    if hasattr(model, "decision_function"):
        return expit(np.asarray(model.decision_function(Xt)))
    raise AttributeError("Model needs predict_proba or decision_function.")


def fit_predict(model, Xtr, ytr, Xte):
    m = clone(model).fit(Xtr, ytr)
    return get_probability(m, Xte)


## 8. Repeated grouped-random validation benchmark

Fifteen nominal stratified 80/20 group-random train/test holdouts are generated using distinct random seeds, with each 1-km raster cell treated as an indivisible group. The number of distinct test partitions and the proportion of unique observations receiving held-out predictions must be read from the split and prediction audits. Repeated seeds do not guarantee distinct group partitions or complete prediction coverage. Hyperparameter optimization and probability calibration are performed within each outer training partition. These scores quantify random-split performance and are not geographic-transfer estimates.


## 9a. Buffered spatial split construction

The publication rerun uses the **buffered, nested spatial-validation implementation** below.
The earlier unbuffered/fixed-hyperparameter result is intentionally not executed or reported.

## 9b. Outer buffered spatial split diagnostics

The contiguous geographic-region assignment is followed by an explicit geodesic exclusion step. For each outer fold, every nominal training observation within `BUFFER_DISTANCE` of the nearest outer-test observation is removed. The diagnostics below record the retained training support and verify the realized minimum train–test separation. This is the primary validation regime reported in the manuscript.

### v3.2.2 partition-key overlap correction

Raster-cell group IDs are compact integers generated independently for each input array. Their numeric values are therefore not stable across subsets. Comparing `unique(group_ids[train])` with `unique(group_ids[test])` can report false overlaps even when the underlying raster cells are disjoint. v3.2.2 compares the stable raster `(row, column)` keys across train/test subsets, checks separation both before and after buffering, and retains whole-cell buffering. This is a diagnostic correction; it does not relax the geographic buffer or support thresholds.


In [ ]:
# ============================================================
# 9a. EMPIRICAL REFERENCE SCALE
# ============================================================
# The primary outer-fold design is selected in Cell 31 using geographically
# contiguous K-means Voronoi regions and geodesic exclusion buffers. This
# empirical range is retained as the reference scale for buffer candidates.

import numpy as np

if "SPATIAL_RANGE" not in globals() or SPATIAL_RANGE is None or not np.isfinite(SPATIAL_RANGE) or SPATIAL_RANGE <= 0:
    raise RuntimeError("A valid predictor-only SPATIAL_RANGE is required before spatial validation.")

PRIMARY_BLOCK_SIZE = float(SPATIAL_RANGE)  # legacy-compatible reference-scale variable
BUFFER_DISTANCE = np.nan  # assigned only after the support-feasibility search in Cell 31
print(f"Empirical predictor-dependence reference range: {SPATIAL_RANGE/1000:.1f} km")
print("Outer partition strategy will be selected in Cell 31; no buffer is selected here.")

def make_spatial_splits_buffered(xy, y, block_size, n_splits=5, seed=42, buffer_distance=None):
    """Build geographically contiguous outer regions and geodesically buffer them.

    `block_size` is retained in the function signature for compatibility with
    downstream calls and is recorded as the empirical reference scale; the
    outer folds themselves are K-means Voronoi regions, not randomly allocated
    square blocks. Every test fold is a single geographic region. The nearest
    retained-training distance is calculated on the WGS84 ellipsoid.

    Weak folds are returned with validity flags so the support-search cell can
    diagnose every candidate rather than stopping after the first bad fold.
    """
    if buffer_distance is None:
        buffer_distance = block_size
    xy = np.asarray(xy, dtype=float)
    yv = np.asarray(y, dtype=int)
    if len(xy) != len(yv):
        raise ValueError("Coordinate and label lengths do not match.")
    if n_splits < 3:
        raise ValueError("At least three geographic regions are required for the primary design.")

    groups, centers, region_counts = assign_contiguous_spatial_regions(
        xy, n_regions=n_splits, seed=seed
    )
    buffered_splits, diag_rows = [], []
    for fold in range(n_splits):
        te = np.where(groups == fold)[0]
        tr = np.where(groups != fold)[0]
        if len(te) == 0 or len(tr) == 0:
            raise RuntimeError(f"Geographic region fold {fold} is empty.")
        # Distance from each nominal training point to the nearest test point.
        # If a raster cell contains multiple inventory records, use the minimum
        # distance across that whole cell and retain/remove the complete cell.
        d_train_to_test = geodesic_nearest_distance_m(xy[te], xy[tr])
        train_pixel_groups = get_sampling_pixel_group_ids(xy[tr])
        _, train_group_inverse = np.unique(train_pixel_groups, return_inverse=True)
        n_train_pixel_groups = int(train_group_inverse.max()) + 1
        group_min_distance = np.full(n_train_pixel_groups, np.inf, dtype=float)
        np.minimum.at(group_min_distance, train_group_inverse, d_train_to_test)
        keep = group_min_distance[train_group_inverse] >= float(buffer_distance)
        tr_buffered = tr[keep]
        # Validate raster-cell separation using stable row/column keys.
        # Compact group IDs are re-numbered independently in each subset, so
        # comparing their integer values across subsets creates false overlaps.
        train_pixel_keys_before_buffer = get_sampling_pixel_keys(xy[tr])
        test_pixel_keys = get_sampling_pixel_keys(xy[te])
        prebuffer_pixel_overlap_n = count_sampling_pixel_key_overlap(
            train_pixel_keys_before_buffer, test_pixel_keys
        )
        if prebuffer_pixel_overlap_n:
            raise AssertionError(
                "A 1-km raster cell was assigned to more than one geographic region "
                "before buffering. Check assign_contiguous_spatial_regions()."
            )

        retained_pixel_keys = get_sampling_pixel_keys(xy[tr_buffered])
        n_pixel_group_overlap = count_sampling_pixel_key_overlap(
            retained_pixel_keys, test_pixel_keys
        )
        if n_pixel_group_overlap:
            raise AssertionError(
                "A 1-km raster cell crossed the buffered spatial train/test boundary."
            )

        n_train_presence = int(np.sum(yv[tr_buffered] == 1))
        n_train_background = int(np.sum(yv[tr_buffered] == 0))
        n_test_presence = int(np.sum(yv[te] == 1))
        n_test_background = int(np.sum(yv[te] == 0))
        train_n = int(len(tr_buffered))
        test_n = int(len(te))
        min_test_class = min(n_test_presence, n_test_background)
        min_train_class = min(n_train_presence, n_train_background)
        support_ok = (
            train_n >= MIN_RETAINED_TRAIN_N and
            min_train_class >= MIN_TRAIN_PER_CLASS_N
        )
        class_ok = min_test_class >= MIN_TEST_PER_CLASS_N
        train_has_both = (n_train_presence > 0 and n_train_background > 0)
        fold_valid = bool(support_ok and class_ok and train_has_both)

        diag_rows.append({
            "fold": int(fold),
            "partition_strategy": "kmeans_voronoi_geographic_regions",
            "n_train_original": int(len(tr)),
            "n_train_buffered": train_n,
            "n_train_pixel_groups": int(len(np.unique(retained_pixel_keys, axis=0))),
            "n_test_pixel_groups": int(len(np.unique(test_pixel_keys, axis=0))),
            "n_pixel_group_overlap_before_buffer": int(prebuffer_pixel_overlap_n),
            "n_pixel_group_overlap": int(n_pixel_group_overlap),
            "pixel_group_buffering": "whole_cell_minimum_distance",
            "n_train_presence": n_train_presence,
            "n_train_background": n_train_background,
            "min_train_class_n": min_train_class,
            "n_excluded": int(len(tr) - train_n),
            "pct_excluded": float((len(tr) - train_n) / max(len(tr), 1) * 100),
            "n_test": test_n,
            "n_test_presence": n_test_presence,
            "n_test_background": n_test_background,
            "min_test_class_n": min_test_class,
            "buffer_distance": float(buffer_distance),
            "reference_scale_m": float(block_size),
            "fold_support_adequate": bool(support_ok),
            "fold_test_classes_adequate": bool(class_ok),
            "fold_valid": fold_valid,
            "region_center_x": float(centers[fold, 0]),
            "region_center_y": float(centers[fold, 1]),
        })
        buffered_splits.append((tr_buffered, te))

    return buffered_splits, groups, pd.DataFrame(diag_rows)


def spatial_block_validation_buffered(X, y, coords, models, block_size, n_splits=5,
                                       seed=42, buffer_distance=None):
    """Fixed-default-hyperparameter buffered validation for buffer sensitivity only.

    This is not the primary performance estimate. Primary results use
    `spatial_block_validation_nested` in Cell 34, with nested tuning and calibration.
    """
    splits, groups, diagnostics = make_spatial_splits_buffered(
        coords, y, block_size, n_splits, seed, buffer_distance
    )
    rows, preds = [], []
    yv = np.asarray(y)
    coords_np = np.asarray(coords, dtype=float)
    if not diagnostics["fold_valid"].all():
        raise RuntimeError("Buffered regional validation contains a fold below the prespecified support/class requirements; inspect diagnostics before fitting.")

    for fold, (tr, te) in enumerate(splits):
        for name, model in models.items():
            cal_splits, cal_k, cal_type = make_inner_spatial_splits(
                coords_np[tr], yv[tr], block_size=block_size, n_splits=INNER_FOLDS,
                seed=seed + 50000 + fold
            )
            calibrated = CrossFittedCalibratedEstimator(
                _single_threaded_clone(model), cal_splits
            ).fit(X.iloc[tr], yv[tr])
            prob = get_probability(calibrated, X.iloc[te])
            rows.append({
                "validation": "spatial_kmeans_region_buffered_fixed_params_crossfit_calibrated",
                "reference_scale_m": block_size, "partition_strategy": "kmeans_voronoi_geographic_regions",
                "buffer_distance": buffer_distance if buffer_distance is not None else block_size,
                "repeat": seed, "fold": fold, "model": name,
                "calibration_folds": cal_k, "calibration_type": cal_type,
                **calculate_metrics(yv[te], prob)
            })
            preds.append(pd.DataFrame({
                "index": te, "y_true": yv[te], "probability": prob,
                "validation": "spatial_kmeans_region_buffered", "reference_scale_m": block_size, "partition_strategy": "kmeans_voronoi_geographic_regions",
                "repeat": seed, "fold": fold, "model": name
            }))
    return pd.DataFrame(rows), pd.concat(preds, ignore_index=True), diagnostics

In [ ]:
# ============================================================
# GEOGRAPHICALLY CONTIGUOUS, SUPPORT-AWARE DESIGN SELECTION
# ============================================================
# The previous square-block/StratifiedGroupKFold assignment scattered test
# blocks across the whole domain. Buffering around dispersed test blocks
# removed too much training data. Here, test folds are contiguous K-means
# Voronoi regions, followed by a WGS84 geodesic exclusion buffer.
#
# Search is based only on geometry, class-count feasibility and sample support;
# model performance is NOT used to choose the design. The explicit 500-row
# training floor and per-class floors are not reduced automatically.

BUFFER_CANDIDATE_MULTIPLIERS = [0.75, 0.50, 0.40, 0.30, 0.25, 0.20, 0.15, 0.10]
FOLD_COUNT_CANDIDATES = [5, 4, 3]  # prefer more geographic folds when feasible
if SPATIAL_RANGE is None or not np.isfinite(SPATIAL_RANGE) or SPATIAL_RANGE <= 0:
    raise RuntimeError("SPATIAL_RANGE is unavailable or invalid.")

buffer_detail_rows = []
partition_error_rows = []
for n_folds_candidate in FOLD_COUNT_CANDIDATES:
    for multiplier in BUFFER_CANDIDATE_MULTIPLIERS:
        candidate_buffer = float(SPATIAL_RANGE * multiplier)
        for seed in BUFFER_SELECTION_SEEDS:
            try:
                candidate_splits, candidate_groups, candidate_diag = make_spatial_splits_buffered(
                    coords, y, SPATIAL_RANGE, n_folds_candidate, seed, candidate_buffer
                )
                candidate_diag = candidate_diag.copy()
                candidate_diag["n_folds"] = int(n_folds_candidate)
                candidate_diag["buffer_multiplier"] = float(multiplier)
                candidate_diag["buffer_distance_m"] = candidate_buffer
                candidate_diag["seed"] = int(seed)
                candidate_diag["partition_constructed"] = True
                buffer_detail_rows.extend(candidate_diag.to_dict("records"))
            except Exception as exc:
                partition_error_rows.append({
                    "n_folds": int(n_folds_candidate),
                    "buffer_multiplier": float(multiplier),
                    "buffer_distance_m": candidate_buffer,
                    "seed": int(seed),
                    "error": f"{type(exc).__name__}: {exc}"
                })

buffer_candidate_detail = pd.DataFrame(buffer_detail_rows)
partition_errors = pd.DataFrame(partition_error_rows)
if buffer_candidate_detail.empty:
    raise RuntimeError(
        "No geographic partitions could be constructed. Inspect coordinate coverage, "
        "duplicate locations and the number of unique sample coordinates."
    )

summary_rows = []
expected_seeds = len(BUFFER_SELECTION_SEEDS)
for n_folds_candidate in FOLD_COUNT_CANDIDATES:
    for multiplier in BUFFER_CANDIDATE_MULTIPLIERS:
        g = buffer_candidate_detail[
            (buffer_candidate_detail["n_folds"] == n_folds_candidate) &
            (buffer_candidate_detail["buffer_multiplier"] == multiplier)
        ].copy()
        expected_rows = expected_seeds * int(n_folds_candidate)
        failed_seed_rows = partition_errors[
            (partition_errors["n_folds"] == n_folds_candidate) &
            (partition_errors["buffer_multiplier"] == multiplier)
        ] if not partition_errors.empty else pd.DataFrame()
        n_constructed_seeds = int(g["seed"].nunique()) if not g.empty else 0
        all_seeds_constructed = (n_constructed_seeds == expected_seeds and failed_seed_rows.empty)
        complete_fold_count = (len(g) == expected_rows)
        all_folds_valid = bool(complete_fold_count and g["fold_valid"].all())
        if g.empty:
            min_train = min_train_class = min_test_class = 0
            mean_train = 0.0
            max_train = 0
        else:
            min_train = int(g["n_train_buffered"].min())
            min_train_class = int(g["min_train_class_n"].min())
            min_test_class = int(g["min_test_class_n"].min())
            mean_train = float(g["n_train_buffered"].mean())
            max_train = int(g["n_train_buffered"].max())
        support_adequate = bool(
            all_seeds_constructed and all_folds_valid and
            min_train >= MIN_RETAINED_TRAIN_N and
            min_train_class >= MIN_TRAIN_PER_CLASS_N and
            min_test_class >= MIN_TEST_PER_CLASS_N
        )
        summary_rows.append({
            "n_folds": int(n_folds_candidate),
            "buffer_multiplier": float(multiplier),
            "buffer_distance_m": float(SPATIAL_RANGE * multiplier),
            "n_partition_seeds_expected": expected_seeds,
            "n_partition_seeds_constructed": n_constructed_seeds,
            "n_folds_evaluated": int(len(g)),
            "n_folds_expected": expected_rows,
            "n_partition_errors": int(len(failed_seed_rows)),
            "all_partitions_valid": bool(all_seeds_constructed and all_folds_valid),
            "min_train_n": min_train,
            "mean_train_n": mean_train,
            "max_train_n": max_train,
            "min_train_class_n": min_train_class,
            "min_test_class_n": min_test_class,
            "support_adequate": support_adequate,
        })

buffer_candidates = pd.DataFrame(summary_rows).sort_values(
    ["n_folds", "buffer_multiplier"], ascending=[False, False]
).reset_index(drop=True)
print("Prespecified support floors:", {
    "minimum_total_training_n": MIN_RETAINED_TRAIN_N,
    "minimum_training_n_per_class": MIN_TRAIN_PER_CLASS_N,
    "minimum_test_n_per_class": MIN_TEST_PER_CLASS_N,
})
display(buffer_candidates)
buffer_candidate_detail.to_csv(OUTPUT_DIR / "regional_buffer_support_by_fold_seed.csv", index=False)
buffer_candidates.to_csv(OUTPUT_DIR / "regional_buffer_support_summary.csv", index=False)
if not partition_errors.empty:
    partition_errors.to_csv(OUTPUT_DIR / "regional_partition_construction_errors.csv", index=False)

feasible = buffer_candidates.loc[buffer_candidates["support_adequate"]].copy()
if feasible.empty:
    best = buffer_candidates.sort_values(
        ["min_train_n", "min_train_class_n", "n_folds"], ascending=[False, False, False]
    ).iloc[0]
    raise RuntimeError(
        "No contiguous regional design met the prespecified support requirements. "
        f"Best diagnostic candidate: {int(best['n_folds'])} folds, "
        f"buffer={best['buffer_distance_m']/1000:.1f} km, "
        f"minimum training n={int(best['min_train_n'])}, "
        f"minimum training class n={int(best['min_train_class_n'])}, "
        f"minimum test class n={int(best['min_test_class_n'])}. "
        "Do not lower support floors automatically. Inspect the regional support CSVs; "
        "if no feasible geometry exists, revise the inventory or report a less ambitious "
        "regional validation design explicitly."
    )

# Preference rule declared before model fitting: most folds first, then the
# largest feasible geographic buffer. No model metric is involved.
selected_row = feasible.sort_values(
    ["n_folds", "buffer_multiplier"], ascending=[False, False]
).iloc[0]
N_SPATIAL_FOLDS = int(selected_row["n_folds"])
SELECTED_BUFFER_MULTIPLIER = float(selected_row["buffer_multiplier"])
BUFFER_DISTANCE = float(selected_row["buffer_distance_m"])
PRIMARY_BLOCK_SIZE = float(SPATIAL_RANGE)  # reference range; outer folds are regional K-means partitions

print("=" * 68)
print("SELECTED GEOGRAPHICALLY CONTIGUOUS VALIDATION DESIGN")
print("=" * 68)
print(f"Spatial dependence range     : {SPATIAL_RANGE/1000:.1f} km")
print(f"Outer partition strategy     : K-means Voronoi geographic regions")
print(f"Selected geographic folds    : {N_SPATIAL_FOLDS}")
print(f"Selected buffer multiplier   : {SELECTED_BUFFER_MULTIPLIER:.2f} × range")
print(f"Selected geodesic buffer     : {BUFFER_DISTANCE/1000:.1f} km")
print(f"Minimum training n           : {int(selected_row['min_train_n'])}")
print(f"Minimum training class n     : {int(selected_row['min_train_class_n'])}")
print(f"Minimum test class n         : {int(selected_row['min_test_class_n'])}")
print(f"Repeated partition seeds     : {BUFFER_SELECTION_SEEDS}")
if SELECTED_BUFFER_MULTIPLIER < 0.25:
    print("WARNING: selected buffer is <0.25× the empirical range; interpret transfer as geographically separated but not fully independent at the estimated dependence scale.")


## 9c. Outer buffered regional-split diagnostics

The selected contiguous geographic regions and geodesic buffer are constructed here. Model fitting is performed only by the nested spatial-validation cell that follows.

In [ ]:
# Diagnostics for every prespecified publication outer buffered partition.
# The selected geometry/buffer must satisfy support rules for all seeds, not just
# the single seed used for a preview in the previous notebook version.
buffered_split_previews = {}
buffer_preview_parts = []
for preview_seed in SPATIAL_CV_SEEDS:
    preview_splits, _, preview_diag = make_spatial_splits_buffered(
        coords, y, PRIMARY_BLOCK_SIZE, N_SPATIAL_FOLDS,
        int(preview_seed), BUFFER_DISTANCE
    )
    preview_diag = preview_diag.copy()
    preview_diag["repeat"] = int(preview_seed)
    buffered_split_previews[int(preview_seed)] = preview_splits
    buffer_preview_parts.append(preview_diag)

buffer_preview_diagnostics = pd.concat(buffer_preview_parts, ignore_index=True)
if (not buffer_preview_diagnostics["fold_valid"].all() or
        buffer_preview_diagnostics["n_train_buffered"].min() < MIN_RETAINED_TRAIN_N or
        buffer_preview_diagnostics["min_train_class_n"].min() < MIN_TRAIN_PER_CLASS_N or
        buffer_preview_diagnostics["min_test_class_n"].min() < MIN_TEST_PER_CLASS_N):
    raise RuntimeError(
        "The selected primary buffered design is not support-feasible across every partition seed: "
        f"minimum training n={buffer_preview_diagnostics['n_train_buffered'].min()}, "
        f"minimum training class n={buffer_preview_diagnostics['min_train_class_n'].min()}, "
        f"minimum test class n={buffer_preview_diagnostics['min_test_class_n'].min()}."
    )

buffer_preview_diagnostics.to_csv(
    OUTPUT_DIR / "buffer_exclusion_diagnostics_preview.csv", index=False
)
display(buffer_preview_diagnostics)

# Partition stability audit: compare geometry-only K-means assignments across
# the prespecified seeds. ARI=1 means identical assignments up to label
# permutation. These seeds are repeated design realizations, not independent
# geographic samples; report their similarity when interpreting fold SDs.
from sklearn.metrics import adjusted_rand_score
from itertools import combinations as _combinations

partition_groupings = {}
partition_rows = []
for part_seed in SPATIAL_CV_SEEDS:
    part_groups, part_centers, part_counts = assign_contiguous_spatial_regions(
        coords, n_regions=N_SPATIAL_FOLDS, seed=int(part_seed), n_init=30
    )
    partition_groupings[int(part_seed)] = part_groups
    for fold_id in range(N_SPATIAL_FOLDS):
        fold_mask = part_groups == fold_id
        partition_rows.append({
            "partition_seed": int(part_seed),
            "fold": int(fold_id),
            "n_observations": int(fold_mask.sum()),
            "n_presence": int(np.sum(np.asarray(y)[fold_mask] == 1)),
            "n_background": int(np.sum(np.asarray(y)[fold_mask] == 0)),
            "center_x_m": float(part_centers[fold_id, 0]),
            "center_y_m": float(part_centers[fold_id, 1]),
        })

ari_rows = []
for seed_a, seed_b in _combinations(SPATIAL_CV_SEEDS, 2):
    ari_rows.append({
        "seed_a": int(seed_a), "seed_b": int(seed_b),
        "adjusted_rand_index": float(adjusted_rand_score(
            partition_groupings[int(seed_a)], partition_groupings[int(seed_b)]
        )),
    })
partition_stability = pd.DataFrame(partition_rows)
partition_similarity = pd.DataFrame(ari_rows)
partition_stability.to_csv(OUTPUT_DIR / "regional_partition_fold_composition.csv", index=False)
partition_similarity.to_csv(OUTPUT_DIR / "regional_partition_stability.csv", index=False)
print("Partition stability (ARI; 1 means identical partitions):")
display(partition_similarity)
display(partition_stability)


## 9. Repeated nested buffered spatial validation

Outer test folds are contiguous K-means Voronoi regions. For each outer fold, training observations within the selected geodesic buffer are excluded. Hyperparameter tuning is spatially nested. Probability calibration is also cross-fitted: each calibration fold's base model is tuned using only that calibration fold's training subset before generating held-out calibration scores. A final base estimator is independently tuned on the complete outer-training subset and combined with the cross-fitted calibrator.

This nested calibration implementation is deliberately more computationally demanding than tuning once on the entire outer-training subset and then reusing those labels to generate calibration-fold predictions.

The calibration-fold tuning budget is `N_ITER_CALIBRATION_SEARCH = 3` randomized configurations per calibration-training fold; the final outer-training fit uses `N_ITER_SEARCH = 8`. Both values are written to the result manifest. Increase them only if the compute budget permits and record the change before interpreting the results.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

ENABLE_NESTED_TUNING = True
INNER_FOLDS = 2  # two geographically contiguous inner folds to reduce repeated fitting cost
N_ITER_SEARCH = 4
# Lower but still randomized tuning budget within each calibration-training
# fold; this is prespecified to keep nested calibration computationally viable.
N_ITER_CALIBRATION_SEARCH = 2

PARAM_DISTRIBUTIONS = {
    "Logistic Regression": {"model__C": np.logspace(-3, 2, 20)},
    "SVM-RBF": {
        "model__C": np.logspace(-2, 2, 16),
        "model__gamma": ["scale"] + list(np.logspace(-4, -1, 10)),
    },
    "Random Forest": {
        "model__n_estimators": [150, 250, 400],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": ["sqrt", "log2", 0.5],
    },
    "XGBoost": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [3, 5, 7, 9],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 1.0],
    },
    "LightGBM": {
        "model__n_estimators": [100, 200, 300],
        "model__num_leaves": [15, 31, 63, 127],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 1.0],
        "model__min_child_samples": [10, 20, 40],
    },
    "MLP": {
        "model__hidden_layer_sizes": [(64, 32), (64, 32, 16), (128, 64, 32)],
        "model__dropout": [0.1, 0.2, 0.3],
        "model__learning_rate": [1e-3, 3e-4],
        "model__batch_size": [32, 64],
        "model__weight_decay": [0.0, 1e-4, 1e-3],
        "model__max_epochs": [60, 100, 140],
    },
}
PARAM_DISTRIBUTIONS = {k: v for k, v in PARAM_DISTRIBUTIONS.items() if k in MODELS}


def _single_threaded_clone(model):
    """Clone an estimator and limit estimator-level threading when supported."""
    m = clone(model)
    params = m.get_params(deep=True) if hasattr(m, "get_params") else {}
    nested_model = getattr(m, "named_steps", {}).get("model") if hasattr(m, "named_steps") else None
    is_logistic = isinstance(nested_model if nested_model is not None else m, LogisticRegression)
    for param_name in ("model__n_jobs", "model__nthread", "model__num_threads", "n_jobs", "nthread"):
        if param_name not in params:
            continue
        if is_logistic and param_name.endswith("n_jobs"):
            continue  # LogisticRegression does not need estimator-level parallelism.
        try:
            m.set_params(**{param_name: 1})
        except (ValueError, TypeError):
            pass
    return m

def make_inner_spatial_splits(xy, y, block_size, n_splits=3, seed=42):
    """Construct contiguous spatial inner folds without a random fallback."""
    xy = np.asarray(xy, dtype=float)
    yv = np.asarray(y, dtype=int)
    n_unique = len(np.unique(xy, axis=0))
    k = min(int(n_splits), n_unique)
    if k < 2:
        raise ValueError("Fewer than two unique locations for inner spatial CV.")
    last_error = None
    for k_try in range(k, 1, -1):
        for offset in range(30):
            try:
                groups, _, _ = assign_contiguous_spatial_regions(
                    xy, n_regions=k_try, seed=int(seed) + offset, n_init=20
                )
                splits, valid = [], True
                pixel_groups = get_sampling_pixel_group_ids(xy)
                for fold in range(k_try):
                    va = np.where(groups == fold)[0]
                    tr = np.where(groups != fold)[0]
                    if np.intersect1d(np.unique(pixel_groups[tr]), np.unique(pixel_groups[va])).size:
                        raise AssertionError("Raster-cell group crossed an inner spatial fold.")
                    if len(tr) == 0 or len(va) == 0:
                        valid = False; break
                    if len(np.unique(yv[tr])) < 2 or len(np.unique(yv[va])) < 2:
                        valid = False; break
                    if min(int((yv[va] == 1).sum()), int((yv[va] == 0).sum())) < 10:
                        valid = False; break
                    splits.append((tr, va))
                if valid and len(splits) == k_try:
                    return splits, k_try, "spatial_kmeans_voronoi_regions"
            except Exception as exc:
                last_error = exc
    raise ValueError(
        "Could not form valid inner spatial folds with both classes and at least "
        f"10 validation observations per class. Last error: {last_error!r}"
    )


def make_random_cv_splits(y, coords, n_splits=3, seed=42, max_attempts=30):
    """Stratified random inner folds with indivisible 1-km raster-cell groups."""
    yv = np.asarray(y, dtype=int)
    xy = np.asarray(coords, dtype=float)
    if len(yv) != len(xy):
        raise ValueError("y and coords lengths differ in grouped random CV.")
    groups = get_sampling_pixel_group_ids(xy)
    group_counts_by_class = [len(np.unique(groups[yv == cls])) for cls in [0, 1]]
    k = min(int(n_splits), len(np.unique(groups)), min(group_counts_by_class))
    if k < 2:
        raise ValueError("Too few independent raster-cell groups for stratified inner CV.")

    for attempt in range(int(max_attempts)):
        cv = StratifiedGroupKFold(
            n_splits=k, shuffle=True, random_state=int(seed) + attempt
        )
        try:
            splits = list(cv.split(np.zeros(len(yv)), yv, groups=groups))
        except ValueError:
            continue
        valid = True
        for tr, va in splits:
            if len(np.unique(yv[tr])) != 2 or len(np.unique(yv[va])) != 2:
                valid = False
                break
            if np.intersect1d(np.unique(groups[tr]), np.unique(groups[va])).size:
                raise AssertionError("Raster-cell group crossed an inner random fold.")
        if valid:
            return splits, k, "stratified_group_random_raster_pixel"
    raise ValueError(
        "Could not construct valid stratified group-random inner folds with both classes."
    )

def _get_inner_splits(X_part, y_part, coords_part, scheme, seed, n_splits=3):
    if scheme == "spatial":
        return make_inner_spatial_splits(
            np.asarray(coords_part), np.asarray(y_part), PRIMARY_BLOCK_SIZE,
            n_splits=n_splits, seed=seed
        )
    if scheme == "random":
        return make_random_cv_splits(y_part, coords_part, n_splits=n_splits, seed=seed)
    raise ValueError("scheme must be 'spatial' or 'random'.")


def _tune_base_estimator(model_name, base_model, X_fit, y_fit, coords_fit,
                         scheme, seed, n_iter):
    """Tune using folds made only from the supplied fitting subset."""
    yv = np.asarray(y_fit, dtype=int)
    if not ENABLE_NESTED_TUNING or model_name not in PARAM_DISTRIBUTIONS:
        fitted = _single_threaded_clone(base_model).fit(X_fit, yv)
        return fitted, {}, 0, "not_tuned"
    inner_splits, inner_k, inner_type = _get_inner_splits(
        X_fit, yv, coords_fit, scheme, seed, INNER_FOLDS
    )
    search = RandomizedSearchCV(
        estimator=_single_threaded_clone(base_model),
        param_distributions=PARAM_DISTRIBUTIONS[model_name],
        n_iter=int(n_iter),
        cv=inner_splits, scoring="roc_auc", random_state=int(seed),
        n_jobs=1, refit=True, error_score="raise"  # avoid Kaggle RAM/CPU oversubscription
    )
    search.fit(X_fit, yv)
    return search.best_estimator_, search.best_params_, inner_k, inner_type


class ScoreCalibratedModel:
    """Fitted estimator plus logistic calibrator trained on nested OOF raw scores."""
    def __init__(self, estimator, calibrator):
        self.estimator_ = estimator
        self.calibrator_ = calibrator
        self.classes_ = np.asarray([0, 1], dtype=int)

    def predict_proba(self, X):
        raw = _raw_model_score(self.estimator_, X)
        return self.calibrator_.predict_proba(raw.reshape(-1, 1))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def fit_nested_calibrated_model(model_name, base_model, X_train, y_train, coords_train,
                                cv_scheme="spatial", seed=42,
                                n_iter_final=None, n_iter_calibration=None):
    """Nested hyperparameter tuning + cross-fitted calibration. Random inner folds are grouped by 1-km raster cell.

    For each calibration fold, hyperparameters are selected using only that
    fold's calibration-training observations. The resulting estimator predicts
    raw scores for the held-out calibration fold. A logistic calibrator is fit
    to the assembled OOF scores. Finally, a separate estimator is tuned and
    fitted on the complete current training partition. The outer test is never
    used by any of these steps.
    """
    yv = np.asarray(y_train, dtype=int)
    xy = np.asarray(coords_train, dtype=float)
    if len(X_train) != len(yv) or len(xy) != len(yv):
        raise ValueError("X, y and coordinate lengths differ in nested calibration.")
    if len(np.unique(yv)) != 2:
        raise ValueError("Nested calibration requires both classes.")

    if cv_scheme == "spatial":
        cal_splits, cal_k, cal_type = make_inner_spatial_splits(
            xy, yv, PRIMARY_BLOCK_SIZE, INNER_FOLDS, seed + 10007
        )
    else:
        cal_splits, cal_k, cal_type = make_random_cv_splits(
            yv, xy, INNER_FOLDS, seed + 10007
        )

    oof_raw = np.full(len(yv), np.nan, dtype=float)
    fold_params = []
    calibration_search_n = N_ITER_CALIBRATION_SEARCH if n_iter_calibration is None else int(n_iter_calibration)
    final_search_n = N_ITER_SEARCH if n_iter_final is None else int(n_iter_final)

    for cal_fold, (cal_tr, cal_va) in enumerate(cal_splits):
        if len(np.unique(yv[cal_tr])) != 2 or len(np.unique(yv[cal_va])) != 2:
            raise ValueError(f"Calibration fold {cal_fold} lacks one of the classes.")
        estimator_fold, params_fold, inner_k, inner_type = _tune_base_estimator(
            model_name, base_model,
            _row_slice(X_train, cal_tr), yv[cal_tr], xy[cal_tr],
            cv_scheme, seed + 20000 + cal_fold, calibration_search_n
        )
        oof_raw[cal_va] = _raw_model_score(estimator_fold, _row_slice(X_train, cal_va))
        fold_params.append({
            "calibration_fold": int(cal_fold), "best_params": params_fold,
            "inner_folds": int(inner_k), "inner_type": inner_type
        })

    if not np.isfinite(oof_raw).all():
        raise RuntimeError("Nested calibration failed to generate exactly one OOF raw score per training observation.")

    calibrator = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000)
    calibrator.fit(oof_raw.reshape(-1, 1), yv)

    final_estimator, final_params, final_inner_k, final_inner_type = _tune_base_estimator(
        model_name, base_model, X_train, yv, xy,
        cv_scheme, seed + 40009, final_search_n
    )
    fitted = ScoreCalibratedModel(final_estimator, calibrator)
    info = {
        "calibration_folds": int(cal_k),
        "calibration_type": cal_type,
        "final_inner_folds": int(final_inner_k),
        "final_inner_type": final_inner_type,
        "best_params": final_params,
        "calibration_fold_searches": fold_params,
        "calibration_search_iterations": calibration_search_n,
        "final_search_iterations": final_search_n,
        "nested_calibration_tuning": True,
    }
    return fitted, info


def fit_predict_nested(model_name, base_model, X_tr, y_tr, X_te, coords_tr,
                       block_size, buffer_distance, seed=42, cv_scheme="spatial"):
    calibrated, tune_info = fit_nested_calibrated_model(
        model_name, base_model, X_tr, y_tr, coords_tr,
        cv_scheme=cv_scheme, seed=seed
    )
    prob = get_probability(calibrated, X_te)
    return prob, calibrated, tune_info


def spatial_block_validation_nested(X, y, coords, models, block_size, n_splits=5,
                                    seed=42, buffer_distance=None):
    splits, groups, diagnostics = make_spatial_splits_buffered(
        coords, y, block_size, n_splits, seed, buffer_distance
    )
    rows, preds, fitted_models = [], [], {}
    yv = np.asarray(y, dtype=int)
    xy = np.asarray(coords, dtype=float)
    if not diagnostics["fold_valid"].all():
        raise RuntimeError("Outer regional folds fail prespecified train/test support requirements.")

    for fold, (tr, te) in enumerate(splits):
        if (len(tr) < MIN_RETAINED_TRAIN_N or
            min(int((yv[tr] == 1).sum()), int((yv[tr] == 0).sum())) < MIN_TRAIN_PER_CLASS_N or
            min(int((yv[te] == 1).sum()), int((yv[te] == 0).sum())) < MIN_TEST_PER_CLASS_N):
            raise RuntimeError(f"Outer fold {fold} fails the prespecified support rules.")
        for name, model in models.items():
            prob, fitted, tune_info = fit_predict_nested(
                name, model, X.iloc[tr], yv[tr], X.iloc[te], xy[tr],
                block_size=block_size,
                buffer_distance=buffer_distance if buffer_distance is not None else block_size,
                seed=seed + fold * 101, cv_scheme="spatial"
            )
            fitted_models[(seed, fold, name)] = fitted
            print(f"  Spatial seed {seed} | fold {fold+1}/{len(splits)} | model {name} completed", flush=True)
            rows.append({
                "validation": "buffered_spatial_kmeans_nested_calibrated",
                "reference_scale_m": block_size,
                "partition_strategy": "kmeans_voronoi_geographic_regions",
                "buffer_distance": buffer_distance if buffer_distance is not None else block_size,
                "repeat": int(seed), "fold": int(fold), "model": name,
                "nested_tuning_applied": True,
                "nested_calibration_tuning": bool(tune_info["nested_calibration_tuning"]),
                "inner_folds_used": tune_info["final_inner_folds"],
                "inner_cv_type": tune_info["final_inner_type"],
                "calibration_folds": tune_info["calibration_folds"],
                "calibration_type": tune_info["calibration_type"],
                "calibration_search_iterations": tune_info["calibration_search_iterations"],
                "best_params": repr(tune_info["best_params"]),
                **calculate_metrics(yv[te], prob),
            })
            preds.append(pd.DataFrame({
                "index": te, "y_true": yv[te], "probability": prob,
                "validation": "buffered_spatial_kmeans_nested_calibrated",
                "reference_scale_m": block_size,
                "partition_strategy": "kmeans_voronoi_geographic_regions",
                "buffer_distance": buffer_distance if buffer_distance is not None else block_size,
                "repeat": int(seed), "fold": int(fold), "model": name,
            }))
            # Persist completed model/fold results immediately. These are audit checkpoints;
            # the fitted estimator remains in memory for map generation in the same run.
            pd.DataFrame(rows).to_csv(OUTPUT_DIR / "spatial_validation_checkpoint_current_seed.csv", index=False)
            pd.concat(preds, ignore_index=True).to_csv(
                OUTPUT_DIR / "spatial_predictions_checkpoint_current_seed.csv", index=False
            )
        diagnostics["repeat"] = int(seed)
    return pd.DataFrame(rows), pd.concat(preds, ignore_index=True), diagnostics, fitted_models


# Run repeated nested buffered spatial CV on the COMPLETE corrected inventory.
spatial_result_parts, spatial_prediction_parts, buffer_diag_parts = [], [], []
spatial_outer_models_by_repeat = {}
for spatial_seed in SPATIAL_CV_SEEDS:
    print(f"Running nested buffered spatial CV, seed={spatial_seed} ...")
    r, p, d, fitted = spatial_block_validation_nested(
        X, y, coords, MODELS, PRIMARY_BLOCK_SIZE,
        N_SPATIAL_FOLDS, spatial_seed, BUFFER_DISTANCE
    )
    spatial_result_parts.append(r)
    spatial_prediction_parts.append(p)
    buffer_diag_parts.append(d)
    spatial_outer_models_by_repeat.update(fitted)

spatial_results = pd.concat(spatial_result_parts, ignore_index=True)
spatial_predictions = pd.concat(spatial_prediction_parts, ignore_index=True)
buffer_diagnostics = pd.concat(buffer_diag_parts, ignore_index=True)
spatial_outer_models = {
    (fold, name): model
    for (seed, fold, name), model in spatial_outer_models_by_repeat.items()
    if seed == RANDOM_STATE_FINAL
}

spatial_results.to_csv(OUTPUT_DIR / "spatial_validation_results_primary_buffered_nested.csv", index=False)
spatial_predictions.to_csv(OUTPUT_DIR / "spatial_validation_predictions_primary_buffered_nested.csv", index=False)
buffer_diagnostics.to_csv(OUTPUT_DIR / "buffer_exclusion_diagnostics.csv", index=False)

print("Repeated buffered nested spatial results (full inventory):")
display(spatial_results.groupby("model")[
    ["ROC_AUC", "PR_AUC", "Accuracy", "F1", "MCC", "Brier", "Brier_skill", "ECE"]
].agg(["mean", "std"]).round(4))
print("Spatial partition seeds:", SPATIAL_CV_SEEDS)
print("Buffer diagnostics by partition seed:")
display(buffer_diagnostics.groupby("repeat")["n_train_buffered"].agg(["min", "mean", "max"]))

## 10. Spatial model comparison and full-inventory mapping model

Model rankings are summarized from repeated buffered spatial CV. The leading model is selected for constructing the full-inventory susceptibility surface. Because it is selected using these cross-validation results, its winning mean AUC is descriptive and selection-affected; it is not an independent test estimate. The final mapping model is fitted using the complete modelling inventory with spatially nested tuning and nested cross-fitted calibration.

In [ ]:
# ============================================================
# MODEL RANKING + FINAL FULL-INVENTORY MAPPING MODEL
# ============================================================
model_selection_summary = (
    spatial_results.groupby("model")["ROC_AUC"]
    .agg(["mean", "std", "count"])
    .sort_values("mean", ascending=False)
)
BEST_MODEL_NAME = model_selection_summary.index[0]
print("Selected model for the full-inventory susceptibility surface:", BEST_MODEL_NAME)
display(model_selection_summary.round(4))

# Rank-stability diagnostic across the three independent partition seeds.
seed_ranks = spatial_results.groupby(["repeat", "model"])["ROC_AUC"].mean().reset_index()
seed_ranks["rank_within_seed"] = seed_ranks.groupby("repeat")["ROC_AUC"].rank(ascending=False, method="min")
rank_stability = seed_ranks.pivot(index="model", columns="repeat", values="rank_within_seed")
rank_stability["mean_rank"] = rank_stability.mean(axis=1)
rank_stability.to_csv(OUTPUT_DIR / "model_rank_stability_by_partition_seed.csv")
display(rank_stability.sort_values("mean_rank").round(3))

# Full-data model is for mapping only; cross-validation scores remain the
# performance evidence. No separate locked/external test is performed.
final_model, final_fit_info = fit_nested_calibrated_model(
    BEST_MODEL_NAME, MODELS[BEST_MODEL_NAME], X, y, coords,
    cv_scheme="spatial", seed=RANDOM_STATE_FINAL + 90001,
    n_iter_final=N_ITER_SEARCH,
    n_iter_calibration=N_ITER_CALIBRATION_SEARCH
)
FINAL_BEST_PARAMS = final_fit_info["best_params"]

final_manifest = {
    "selected_model_for_mapping": BEST_MODEL_NAME,
    "n_full_inventory_observations": int(len(y)),
    "n_presence": int(np.sum(np.asarray(y) == 1)),
    "n_background": int(np.sum(np.asarray(y) == 0)),
    "n_predictors": int(len(FEATURE_COLS)),
    "model_selection_metric": "mean repeated nested buffered spatial ROC-AUC",
    "model_selection_is_independent_test": False,
    "independent_external_geographic_test_performed": False,
    "final_mapping_fit_scope": "complete corrected modelling inventory",
    "final_tuning_cv": final_fit_info["final_inner_type"],
    "final_tuning_folds": int(final_fit_info["final_inner_folds"]),
    "final_calibration_cv": final_fit_info["calibration_type"],
    "final_calibration_folds": int(final_fit_info["calibration_folds"]),
    "nested_calibration_tuning": True,
    "final_best_params": FINAL_BEST_PARAMS,
    "calibration_search_iterations": int(final_fit_info["calibration_search_iterations"]),
    "final_search_iterations": int(final_fit_info["final_search_iterations"]),
    "outer_partition_strategy": "kmeans_voronoi_geographic_regions",
    "n_spatial_folds": int(N_SPATIAL_FOLDS),
    "spatial_cv_seeds": list(SPATIAL_CV_SEEDS),
    "primary_buffer_m": float(BUFFER_DISTANCE),
    "reference_spatial_range_m": float(SPATIAL_RANGE),
    "target_crs": TARGET_CRS.to_string(),
    "target_resolution_m": TARGET_RESOLUTION_M,
    "event_id_col": EVENT_ID_COL,
}
(OUTPUT_DIR / "final_model_manifest.json").write_text(
    json.dumps(final_manifest, indent=2, default=str), encoding="utf-8"
)
print("Final mapping-model tuning:", FINAL_BEST_PARAMS)
print("The final mapping model uses all observations; no independent test score is claimed.")

## 9e. Nested tuning for the repeated random-validation reference

To isolate validation strategy rather than tuning strategy, the random-validation reference is also nested. Hyperparameters are selected using inner **random stratified CV inside each outer random split**; the outer test set is never used during tuning.

This makes the final random-versus-buffered-spatial comparison fair: both schemes use nested tuning, while only the spatial scheme enforces geographic separation.


In [ ]:
import time
import hashlib


def repeated_random_validation_nested(X, y, coords, models, seeds,
                                      test_size=0.20, verbose=True):
    """Repeated stratified group-random validation by 1-km raster-cell group."""
    rows, preds, split_audits = [], [], []
    test_indices_by_repeat = []
    yv = np.asarray(y, dtype=int)
    xy = np.asarray(coords, dtype=float)

    for si, seed in enumerate(seeds):
        t0 = time.time()
        tr, te, split_info = make_random_group_holdout_split(
            yv, xy, test_size=test_size, seed=int(seed)
        )
        pixel_groups = get_sampling_pixel_group_ids(xy)
        group_overlap = np.intersect1d(
            np.unique(pixel_groups[tr]), np.unique(pixel_groups[te])
        )
        if len(group_overlap):
            raise AssertionError("A 1-km raster cell crossed the random outer train/test split.")
        test_indices_by_repeat.append(np.asarray(te, dtype=np.int64).copy())
        test_signature = hashlib.sha256(
            np.sort(np.asarray(te, dtype=np.int64)).tobytes()
        ).hexdigest()
        split_audits.append({
            "repeat": int(seed),
            "test_partition_signature": test_signature,
            "n_train": int(len(tr)), "n_test": int(len(te)),
            "n_train_presence": int(np.sum(yv[tr] == 1)),
            "n_train_background": int(np.sum(yv[tr] == 0)),
            "n_test_presence": int(np.sum(yv[te] == 1)),
            "n_test_background": int(np.sum(yv[te] == 0)),
            "n_train_pixel_groups": int(len(np.unique(pixel_groups[tr]))),
            "n_test_pixel_groups": int(len(np.unique(pixel_groups[te]))),
            "n_pixel_group_overlap": int(len(group_overlap)),
            **split_info,
        })
        for name, model in models.items():
            prob, calibrated, info = fit_predict_nested(
                name, model, X.iloc[tr], yv[tr], X.iloc[te], xy[tr],
                block_size=PRIMARY_BLOCK_SIZE,
                buffer_distance=0.0,
                seed=int(seed) + 70000,
                cv_scheme="random"
            )
            print(f"  Random repeat {si+1}/{len(seeds)} | model {name} completed", flush=True)
            rows.append({
                "validation": "stratified_group_random_pixel_nested_tuned_crossfit_calibrated",
                "repeat": int(seed), "fold": 0, "model": name,
                "nested_tuning_applied": True,
                "nested_calibration_tuning": bool(info["nested_calibration_tuning"]),
                "random_outer_sampling_unit": "1-km raster cell",
                "random_test_fraction": float(len(te) / len(yv)),
                "calibration_type": info["calibration_type"],
                "calibration_folds": info["calibration_folds"],
                "calibration_search_iterations": info["calibration_search_iterations"],
                "best_params": repr(info["best_params"]),
                **calculate_metrics(yv[te], prob),
            })
            preds.append(pd.DataFrame({
                "index": te, "y_true": yv[te], "probability": prob,
                "pixel_group_id": pixel_groups[te],
                "validation": "stratified_group_random_pixel_nested_tuned_crossfit_calibrated",
                "repeat": int(seed), "fold": 0, "model": name,
            }))
            pd.DataFrame(rows).to_csv(OUTPUT_DIR / "random_validation_checkpoint.csv", index=False)
            pd.concat(preds, ignore_index=True).to_csv(
                OUTPUT_DIR / "random_predictions_checkpoint.csv", index=False
            )
        if verbose:
            print(f"Random seed {si+1}/{len(seeds)} completed in {time.time()-t0:.1f} s", flush=True)
    audit_df = pd.DataFrame(split_audits)
    unique_test_indices = (
        np.unique(np.concatenate(test_indices_by_repeat))
        if test_indices_by_repeat else np.asarray([], dtype=np.int64)
    )
    n_distinct_partitions = int(audit_df["test_partition_signature"].nunique())
    unique_coverage = float(len(unique_test_indices) / max(len(yv), 1))
    audit_df["n_distinct_test_partitions_in_run"] = n_distinct_partitions
    audit_df["n_unique_observations_tested_in_run"] = int(len(unique_test_indices))
    audit_df["unique_observation_coverage_in_run"] = unique_coverage
    audit_df.to_csv(OUTPUT_DIR / "random_group_split_audit.csv", index=False)
    if not audit_df["n_pixel_group_overlap"].eq(0).all():
        raise AssertionError("Random split group audit detected pixel-level leakage.")
    print(
        f"Random split audit: {n_distinct_partitions}/{len(seeds)} distinct test partitions; "
        f"{len(unique_test_indices)}/{len(yv)} unique observations tested "
        f"({unique_coverage:.1%} coverage)."
    )
    return pd.DataFrame(rows), pd.concat(preds, ignore_index=True), audit_df


random_results, random_predictions, random_group_split_audit = repeated_random_validation_nested(
    X, y, coords, MODELS, SEEDS, TEST_SIZE
)
random_results.to_csv(OUTPUT_DIR / "random_validation_results_nested.csv", index=False)
random_predictions.to_csv(OUTPUT_DIR / "random_validation_predictions_nested.csv", index=False)
print("Grouped random-validation results (1-km raster cells are kept intact):")
display(random_results.groupby("model")[
    ["ROC_AUC", "PR_AUC", "Accuracy", "F1", "MCC", "Brier", "ECE"]
].agg(["mean", "std"]).round(4))

## 9f. Event-level dependence audit (supplementary; runs only when an event ID exists)

A random split can be optimistic not only because of spatial proximity but also because multiple observations
may originate from the same mapped flood event. The notebook therefore performs an event-ID audit when a stable
event identifier is available in the source inventory. If no event ID is available, the final package records that
event-level independence could not be tested; no event-grouped result is invented.

In [ ]:
# ============================================================
# EVENT-LEVEL DEPENDENCE AUDIT (supplementary)
# ============================================================
# Uses the complete modelling inventory. Flood presences are grouped by flood-event ID;
# background observations are grouped by spatial block so they are not all
# collapsed into one artificial "missing event" group.
EVENT_GROUPED_AUDIT_RERUN = True

if EVENT_GROUPED_AUDIT_RERUN and EVENT_ID_COL and EVENT_ID_COL in df.columns:
    event_values = df[EVENT_ID_COL]
    presence_mask = np.asarray(y == 1)
    if event_values.loc[presence_mask].isna().any():
        raise ValueError("Some flood-presence rows lack event IDs; event-grouped audit is not defensible until resolved.")
    block_ids = assign_square_blocks(coords, PRIMARY_BLOCK_SIZE)
    event_groups = np.empty(len(df), dtype=object)
    for i in range(len(df)):
        if presence_mask[i]:
            event_groups[i] = "EVENT::" + str(event_values.iloc[i])
        else:
            event_groups[i] = "BACKGROUND_BLOCK::" + str(block_ids[i])
    if len(np.unique(event_groups)) < 5:
        raise RuntimeError("Fewer than five independent event/background groups are available.")

    cv_event = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE_FINAL)
    event_rows, event_preds = [], []
    y_dev = np.asarray(y)
    for fold, (tr, te) in enumerate(cv_event.split(X, y_dev, groups=event_groups)):
        if len(np.unique(y_dev[tr])) < 2 or len(np.unique(y_dev[te])) < 2:
            raise RuntimeError(f"Event-grouped fold {fold} lacks both classes.")
        for name, model in MODELS.items():
            fitted = _single_threaded_clone(model).fit(X.iloc[tr], y_dev[tr])
            # These event-audit outputs are discrimination-only; no calibration
            # metric is reported from this uncalibrated fast sensitivity run.
            score = (_raw_model_score(fitted, X.iloc[te]))
            auc = roc_auc_score(y_dev[te], score)
            pr = average_precision_score(y_dev[te], score)
            event_rows.append({"validation": "event_and_pixel_grouped_development_only", "fold": fold,
                               "model": name, "ROC_AUC": auc, "PR_AUC": pr,
                               "n_test": len(te), "n_event_groups_test": len(np.unique(event_groups[te]))})
            event_preds.append(pd.DataFrame({"index": te, "y_true": y_dev[te], "raw_score": score,
                                             "validation": "event_and_pixel_grouped_development_only",
                                             "fold": fold, "model": name}))
    event_grouped_results = pd.DataFrame(event_rows)
    event_grouped_predictions = pd.concat(event_preds, ignore_index=True)
    event_grouped_results.to_csv(OUTPUT_DIR / "event_grouped_validation_results.csv", index=False)
    event_grouped_predictions.to_csv(OUTPUT_DIR / "event_grouped_validation_predictions.csv", index=False)
    display(event_grouped_results.groupby("model")["ROC_AUC"].agg(["mean", "std"]).round(4))
else:
    print("Event-level grouped validation unavailable: no stable event identifier is configured. "
          "Do not infer event independence from point-level spatial separation alone.")

## 12a. Spatial-block bootstrap for the random-vs-spatial gap

The earlier bootstrap significance test resampled individual *predictions* as if they
were i.i.d., which is inconsistent with the paper's own premise that nearby predictions
are spatially dependent -- it understates the true sampling uncertainty. This cell
instead resamples whole **folds/blocks** with replacement (a block bootstrap), which
respects the within-block dependence structure and gives an honestly wider, more
conservative confidence interval on the random-vs-spatial gap.

In [ ]:

# ============================================================
# SPATIAL-BLOCK BOOTSTRAP FOR RANDOM-VS-SPATIAL AUC GAP
# Corrected for incomplete paired OOF prediction coverage
# ============================================================
#
# Predictions are paired by model and observation index.
# Only observations with valid predictions under BOTH designs
# are included in the paired AUC comparison.
#
# Coverage is reported against:
#   1. Full modelling inventory
#   2. Random-validation prediction coverage
#   3. Spatial-validation prediction coverage
#
# Bootstrap confidence intervals represent spatial-block
# resampling uncertainty conditional on the fitted procedures
# and the available paired OOF observations.
# ============================================================

import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

N_BLOCK_BOOT = 300  # Increase to 1,000 for final sensitivity if feasible


def spatial_block_bootstrap_paired_auc(
    random_predictions,
    spatial_predictions,
    y_local,
    coords_local,
    block_size,
    n_boot=300,
    seed=42
):

    rng = np.random.default_rng(seed)

    y_local = np.asarray(y_local, dtype=int)
    coords_local = np.asarray(coords_local, dtype=float)

    if len(y_local) != len(coords_local):
        raise ValueError(
            "y_local and coords_local have different lengths."
        )

    if not np.isfinite(coords_local).all():
        raise ValueError(
            "Non-finite coordinates found in coords_local."
        )

    if n_boot < 1:
        raise ValueError("n_boot must be at least 1.")

    # --------------------------------------------------------
    # 1. Validate and aggregate prediction records
    # --------------------------------------------------------

    def prepare_predictions(frame, design_name):

        required = {"model", "index", "probability"}

        missing = required - set(frame.columns)

        if missing:
            raise ValueError(
                f"{design_name} predictions are missing columns: "
                f"{sorted(missing)}"
            )

        tmp = frame[
            ["model", "index", "probability"]
        ].copy()

        tmp["index"] = pd.to_numeric(
            tmp["index"], errors="coerce"
        )

        tmp["probability"] = pd.to_numeric(
            tmp["probability"], errors="coerce"
        )

        # Remove invalid rows.
        tmp = tmp.dropna(
            subset=["model", "index", "probability"]
        )

        # Observation indices must be integers.
        if not np.all(
            np.isclose(
                tmp["index"].to_numpy(dtype=float),
                np.round(tmp["index"].to_numpy(dtype=float))
            )
        ):
            raise ValueError(
                f"{design_name} contains non-integer observation indices."
            )

        tmp["index"] = tmp["index"].astype(np.int64)

        # Remove non-finite probabilities.
        tmp = tmp[
            np.isfinite(tmp["probability"].to_numpy(dtype=float))
        ].copy()

        # Predicted probabilities must lie between 0 and 1.
        if (
            (tmp["probability"] < 0).any()
            or (tmp["probability"] > 1).any()
        ):
            raise ValueError(
                f"{design_name} contains probabilities outside [0, 1]."
            )

        if (
            (tmp["index"] < 0).any()
            or (tmp["index"] >= len(y_local)).any()
        ):
            raise IndexError(
                f"{design_name} contains indices outside the modelling inventory."
            )

        # Average repeated OOF predictions for each observation.
        tmp = (
            tmp.groupby(["model", "index"], as_index=False)
            ["probability"]
            .mean()
        )

        return tmp

    rand = prepare_predictions(
        random_predictions, "Random validation"
    )

    spat = prepare_predictions(
        spatial_predictions, "Spatial validation"
    )

    # --------------------------------------------------------
    # 2. Check model availability
    # --------------------------------------------------------

    random_models = set(rand["model"].unique())
    spatial_models = set(spat["model"].unique())

    common_models = sorted(
        random_models.intersection(spatial_models)
    )

    if not common_models:
        raise RuntimeError(
            "No models have predictions under both validation designs."
        )

    missing_random = sorted(
        spatial_models - random_models
    )

    missing_spatial = sorted(
        random_models - spatial_models
    )

    if missing_random:
        warnings.warn(
            "Models missing from random predictions: "
            + ", ".join(map(str, missing_random))
        )

    if missing_spatial:
        warnings.warn(
            "Models missing from spatial predictions: "
            + ", ".join(map(str, missing_spatial))
        )

    # --------------------------------------------------------
    # 3. Assign spatial blocks to the complete inventory
    # --------------------------------------------------------

    all_groups = np.asarray(
        assign_square_blocks(
            coords_local,
            block_size
        )
    )

    if len(all_groups) != len(y_local):
        raise ValueError(
            "Spatial block assignment length does not match "
            "the modelling inventory."
        )

    out = []

    # --------------------------------------------------------
    # 4. Calculate paired AUC and spatial-block bootstrap
    # --------------------------------------------------------

    for model_name in common_models:

        r = rand[
            rand["model"] == model_name
        ].copy()

        s = spat[
            spat["model"] == model_name
        ].copy()

        # Inner join ensures identical observations are compared.
        d = r.merge(
            s,
            on=["model", "index"],
            how="inner",
            validate="one_to_one",
            suffixes=("_random", "_spatial")
        )

        # Additional finite-value check.
        d = d.replace(
            [np.inf, -np.inf],
            np.nan
        ).dropna(
            subset=["probability_random", "probability_spatial"]
        )

        idx = d["index"].to_numpy(dtype=int)

        if len(idx) == 0:
            warnings.warn(
                f"No paired predictions available for {model_name}. "
                "Skipping this model."
            )
            continue

        # ----------------------------------------------------
        # Coverage diagnostics
        # ----------------------------------------------------

        n_inventory = len(y_local)
        n_random = int(len(r))
        n_spatial = int(len(s))
        n_paired = int(len(d))

        coverage_random = n_random / max(n_inventory, 1)
        coverage_spatial = n_spatial / max(n_inventory, 1)
        paired_coverage = n_paired / max(n_inventory, 1)

        coverage_of_random = (
            n_paired / max(n_random, 1)
        )

        coverage_of_spatial = (
            n_paired / max(n_spatial, 1)
        )

        if paired_coverage < 0.95:

            warnings.warn(
                f"{model_name}: paired prediction coverage is "
                f"{paired_coverage:.1%} of the full inventory. "
                "The paired AUC and bootstrap interval will be "
                "conditional on this observed subset."
            )

        # ----------------------------------------------------
        # Attach true labels and spatial block IDs
        # ----------------------------------------------------

        d["y_true"] = y_local[idx]

        d["block"] = all_groups[idx]

        # Check that both classes are represented.
        if d["y_true"].nunique() < 2:
            warnings.warn(
                f"{model_name}: paired observations contain only "
                "one class. AUC cannot be calculated. Skipping."
            )
            continue

        n_blocks = int(d["block"].nunique())

        if n_blocks < 3:
            warnings.warn(
                f"{model_name}: only {n_blocks} spatial blocks "
                "are available. At least three are required. "
                "Skipping."
            )
            continue

        # ----------------------------------------------------
        # Paired observed AUC values
        # ----------------------------------------------------

        auc_r = roc_auc_score(
            d["y_true"],
            d["probability_random"]
        )

        auc_s = roc_auc_score(
            d["y_true"],
            d["probability_spatial"]
        )

        gap = auc_r - auc_s

        # ----------------------------------------------------
        # Spatial-block bootstrap
        # ----------------------------------------------------

        blocks = d["block"].drop_duplicates().to_numpy()

        by_block = {
            block: d.loc[d["block"] == block]
            for block in blocks
        }

        boot_gap = []
        attempts = 0

        max_attempts = max(
            10 * n_boot,
            n_boot + 100
        )

        while (
            len(boot_gap) < n_boot
            and attempts < max_attempts
        ):

            attempts += 1

            # Resample spatial blocks with replacement.
            sampled_blocks = rng.choice(
                blocks,
                size=len(blocks),
                replace=True
            )

            dd = pd.concat(
                [by_block[b] for b in sampled_blocks],
                ignore_index=True
            )

            # Both classes must occur in a bootstrap sample.
            if dd["y_true"].nunique() < 2:
                continue

            auc_random_boot = roc_auc_score(
                dd["y_true"],
                dd["probability_random"]
            )

            auc_spatial_boot = roc_auc_score(
                dd["y_true"],
                dd["probability_spatial"]
            )

            boot_gap.append(
                auc_random_boot - auc_spatial_boot
            )

        # Require a sufficient number of valid replicates.
        minimum_valid = max(
            100,
            int(0.80 * n_boot)
        )

        if len(boot_gap) < minimum_valid:
            raise RuntimeError(
                f"{model_name}: only {len(boot_gap)} valid "
                f"bootstrap samples after {attempts} attempts. "
                f"Required: {minimum_valid}. Consider revisiting "
                "the spatial block size or bootstrap design."
            )

        boot_gap = np.asarray(
            boot_gap,
            dtype=float
        )

        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        out.append({

            "model": model_name,

            "paired_random_auc": float(auc_r),

            "paired_spatial_auc": float(auc_s),

            "paired_auc_gap": float(gap),

            "bootstrap_ci_low": float(
                np.quantile(boot_gap, 0.025)
            ),

            "bootstrap_ci_high": float(
                np.quantile(boot_gap, 0.975)
            ),

            "n_bootstrap_valid": int(len(boot_gap)),

            "n_bootstrap_attempts": int(attempts),

            "n_inventory_observations": int(n_inventory),

            "n_random_observations": int(n_random),

            "n_spatial_observations": int(n_spatial),

            "n_matched_observations": int(n_paired),

            "random_prediction_coverage": float(
                coverage_random
            ),

            "spatial_prediction_coverage": float(
                coverage_spatial
            ),

            "paired_prediction_coverage": float(
                paired_coverage
            ),

            "paired_coverage_of_random": float(
                coverage_of_random
            ),

            "paired_coverage_of_spatial": float(
                coverage_of_spatial
            ),

            "n_spatial_blocks": int(n_blocks),

            "interval_scope": (
                "Spatial-block resampling of paired mean OOF "
                "predictions; conditional on fitted resampling "
                "procedures and observations with paired predictions"
            ),

        })

        print(
            f"{model_name}: "
            f"paired n={n_paired:,}/{n_inventory:,} "
            f"({paired_coverage:.1%}), "
            f"random AUC={auc_r:.4f}, "
            f"spatial AUC={auc_s:.4f}, "
            f"gap={gap:.4f}, "
            f"valid bootstrap={len(boot_gap):,}"
        )

    # --------------------------------------------------------
    # 5. Final result table
    # --------------------------------------------------------

    result = pd.DataFrame(out)

    if result.empty:
        raise RuntimeError(
            "No models produced valid paired bootstrap results."
        )

    return result


# ============================================================
# RUN PAIRED SPATIAL-BLOCK BOOTSTRAP
# ============================================================

paired_bootstrap = spatial_block_bootstrap_paired_auc(

    random_predictions=random_predictions,

    spatial_predictions=spatial_predictions,

    y_local=np.asarray(y),

    coords_local=coords,

    block_size=PRIMARY_BLOCK_SIZE,

    n_boot=N_BLOCK_BOOT,

    seed=RANDOM_STATE_FINAL,

)

# Save results.
bootstrap_path = (
    OUTPUT_DIR /
    "spatial_block_bootstrap_random_vs_spatial.csv"
)

paired_bootstrap.to_csv(
    bootstrap_path,
    index=False
)

print("\n" + "=" * 65)
print("PAIRED SPATIAL-BLOCK BOOTSTRAP COMPLETED")
print("=" * 65)

print("Saved:", bootstrap_path)

display(
    paired_bootstrap.round(4)
)


## 9g. Size-matched random benchmark — isolate spatial separation from training-support loss

For each buffered outer spatial test fold, random training samples of exactly the same size as the retained buffered training set are drawn from all observations outside the current spatial test fold. The same geographic test fold is then evaluated. This paired comparison is run for all six candidate models and is an approximate diagnostic of the residual difference associated with geographic proximity at matched sample size. Because random training samples may still include locations close to the test fold, it is not a pure causal estimate of spatial leakage.

In [ ]:
# ============================================================
# SIZE-MATCHED RANDOM REFERENCE — TRAINING SUPPORT VS PROXIMITY
# ============================================================
# For each spatial test fold, a random training set is drawn from all other
# observations with exactly the same sample size AND class composition as the
# buffered training set. It may include nearby points, intentionally reflecting
# the conventional random-training design. All six candidate models are included
# so the support-matched comparison covers the complete model benchmark.
MATCHED_MODELS = list(MODELS.keys())
matched_rows = []
matched_group_audits = []
yv = np.asarray(y, dtype=int)
rng_match = np.random.default_rng(RANDOM_STATE_FINAL)

for val_seed in SPATIAL_CV_SEEDS:
    buffered_splits_for_match, _, _ = make_spatial_splits_buffered(
        coords, y, PRIMARY_BLOCK_SIZE, N_SPATIAL_FOLDS, val_seed, BUFFER_DISTANCE
    )
    for fold, (tr_buf, te) in enumerate(buffered_splits_for_match):
        n_train = len(tr_buf)
        eligible = np.setdiff1d(np.arange(len(X)), te, assume_unique=False)
        y_test = yv[te]
        for model_name in MATCHED_MODELS:
            spatial_rows = spatial_predictions[
                (spatial_predictions.model == model_name) &
                (spatial_predictions.fold == fold) &
                (spatial_predictions.repeat == val_seed)
            ].sort_values("index")
            p_sp = spatial_rows["probability"].to_numpy()
            i_sp = spatial_rows["index"].to_numpy(dtype=int)
            if not np.array_equal(i_sp, te):
                raise ValueError(f"Spatial predictions do not align for seed={val_seed}, fold={fold}, model={model_name}.")
            auc_sp = roc_auc_score(y_test, p_sp)
            mcc_sp = matthews_corrcoef(y_test, (p_sp >= 0.5).astype(int))
            n_pos = int(np.sum(yv[tr_buf] == 1))
            n_neg = int(np.sum(yv[tr_buf] == 0))
            eligible_pos = eligible[yv[eligible] == 1]
            eligible_neg = eligible[yv[eligible] == 0]
            if len(eligible_pos) < n_pos or len(eligible_neg) < n_neg:
                raise RuntimeError("Insufficient class-specific observations for the matched random sample.")

            for rep in range(N_MATCHED_REPEATS):
                rand_seed = RANDOM_STATE_FINAL + val_seed * 1000 + fold * 100 + rep
                rng_fold = np.random.default_rng(rand_seed)
                rand_idx = sample_group_disjoint_class_matched_indices(
                    eligible, yv, coords, n_pos, n_neg, rng_fold
                )
                y_rand = yv[rand_idx]
                test_pixel_groups = SAMPLE_PIXEL_GROUP_IDS[te]
                train_pixel_groups = SAMPLE_PIXEL_GROUP_IDS[rand_idx]
                overlap_groups = np.intersect1d(
                    np.unique(test_pixel_groups), np.unique(train_pixel_groups)
                )
                if len(rand_idx) != n_train or int(y_rand.sum()) != n_pos:
                    raise AssertionError("Matched random sample did not preserve size and class counts.")
                if len(overlap_groups):
                    raise AssertionError("Matched random training set shares a raster cell with its test fold.")
                matched_group_audits.append({
                    "partition_seed": int(val_seed), "fold": int(fold),
                    "repeat": int(rep), "model": model_name,
                    "n_train_matched": int(len(rand_idx)),
                    "n_train_pixel_groups": int(len(np.unique(train_pixel_groups))),
                    "n_test_pixel_groups": int(len(np.unique(test_pixel_groups))),
                    "n_pixel_group_overlap": int(len(overlap_groups)),
                    "n_train_presence": int(np.sum(y_rand == 1)),
                    "n_train_background": int(np.sum(y_rand == 0)),
                })

                calibrated_random, info = fit_nested_calibrated_model(
                    model_name, MODELS[model_name], X.iloc[rand_idx], y_rand,
                    coords[rand_idx], cv_scheme="spatial", seed=rand_seed,
                    n_iter_final=N_ITER_SEARCH,
                    n_iter_calibration=N_ITER_CALIBRATION_SEARCH
                )
                p_rand = get_probability(calibrated_random, X.iloc[te])
                auc_rand = roc_auc_score(y_test, p_rand)
                mcc_rand = matthews_corrcoef(y_test, (p_rand >= 0.5).astype(int))
                matched_rows.append({
                    "partition_seed": int(val_seed), "fold": int(fold), "repeat": int(rep),
                    "model": model_name, "n_train_matched": int(n_train),
                    "n_train_presence_matched": n_pos, "n_train_background_matched": n_neg,
                    "random_matched_AUC": auc_rand, "spatial_AUC_same_test": auc_sp,
                    "paired_AUC_gap": auc_rand - auc_sp,
                    "random_matched_MCC": mcc_rand, "spatial_MCC_same_test": mcc_sp,
                    "paired_MCC_gap": mcc_rand - mcc_sp,
                    "calibration_type": info["calibration_type"],
                    "calibration_folds": info["calibration_folds"],
                    "nested_calibration_tuning": True,
                    "random_matched_sampling_unit": "1-km raster cell",
                    "n_train_pixel_groups": int(len(np.unique(train_pixel_groups))),
                })
                pd.DataFrame(matched_rows).to_csv(
                    OUTPUT_DIR / "size_matched_checkpoint.csv", index=False
                )
                print(f"  Matched benchmark: seed {val_seed}, fold {fold+1}/{N_SPATIAL_FOLDS}, {model_name} completed", flush=True)

matched_results = pd.DataFrame(matched_rows)
matched_group_split_audit = pd.DataFrame(matched_group_audits)
if not matched_group_split_audit.empty:
    matched_group_split_audit.to_csv(
        OUTPUT_DIR / "size_matched_group_split_audit.csv", index=False
    )
    if not matched_group_split_audit["n_pixel_group_overlap"].eq(0).all():
        raise AssertionError("Size-matched group audit detected pixel-level leakage.")
if matched_results.empty:
    raise RuntimeError("Size-matched benchmark produced no valid comparisons.")
matched_results.to_csv(OUTPUT_DIR / "size_matched_random_vs_spatial.csv", index=False)
matched_summary = matched_results.groupby("model")[
    ["random_matched_AUC", "spatial_AUC_same_test", "paired_AUC_gap",
     "random_matched_MCC", "spatial_MCC_same_test", "paired_MCC_gap"]
].agg(["mean", "std"])
display(matched_summary.round(4))

# 10. Buffer-distance sensitivity on fixed regional partitions

This secondary fixed-hyperparameter experiment tests sensitivity to the geodesic exclusion distance while holding each seed’s regional partition fixed.

In [ ]:
# ============================================================
# BUFFER SENSITIVITY — FIXED REGIONAL PARTITIONS, VARYING EXCLUSION DISTANCE
# ============================================================
# The prespecified primary K-means geographic partition is reused across
# three buffer distances. Only the geodesic exclusion buffer changes. Results use fixed model
# hyperparameters and cross-fitted spatial calibration, so this is a sensitivity
# analysis, not a replacement for the repeated nested primary validation.
BUFFER_SENSITIVITY_MULTIPLIERS = [0.0, 0.50, 1.00]
spatial_sensitivity_parts = []
spatial_sensitivity_diag = []

for mult in BUFFER_SENSITIVITY_MULTIPLIERS:
    test_buffer = float(SPATIAL_RANGE * mult)
    for seed in [RANDOM_STATE_FINAL]:  # fixed primary partition seed; sensitivity is descriptive
        print(f"K-means regional folds={N_SPATIAL_FOLDS}; buffer={test_buffer/1000:.1f} km; seed={seed}", flush=True)
        try:
            _, _, diag = make_spatial_splits_buffered(
                coords, y, PRIMARY_BLOCK_SIZE, N_SPATIAL_FOLDS, seed, test_buffer
            )
            diag["buffer_multiplier"] = mult
            diag["repeat"] = seed
            spatial_sensitivity_diag.append(diag)
            min_train = int(diag["n_train_buffered"].min())
            if (not diag["fold_valid"].all() or
                    diag["min_train_class_n"].min() < MIN_TRAIN_PER_CLASS_N or
                    diag["min_test_class_n"].min() < MIN_TEST_PER_CLASS_N):
                print(
                    "  SKIPPED model fitting: at least one region fails the "
                    f"prespecified support/class checks (min train n={min_train}, "
                    f"min train class n={diag['min_train_class_n'].min()}, "
                    f"min test class n={diag['min_test_class_n'].min()})."
                )
                continue
            r, p, diag_fit = spatial_block_validation_buffered(
                X, y, coords, MODELS, PRIMARY_BLOCK_SIZE,
                N_SPATIAL_FOLDS, seed, buffer_distance=test_buffer
            )
            r["buffer_multiplier"] = mult
            r["support_adequate"] = min_train >= MIN_RETAINED_TRAIN_N
            r["min_retained_train_n"] = min_train
            spatial_sensitivity_parts.append(r)
        except Exception as exc:
            print("  Sensitivity configuration failed:", repr(exc))

if not spatial_sensitivity_parts:
    raise RuntimeError("No buffer sensitivity configuration met the training-support floor.")
spatial_sensitivity_results = pd.concat(spatial_sensitivity_parts, ignore_index=True)
spatial_sensitivity_diagnostics = pd.concat(spatial_sensitivity_diag, ignore_index=True)
spatial_sensitivity_results.to_csv(OUTPUT_DIR / "spatial_buffer_sensitivity_fixed_regions.csv", index=False)
spatial_sensitivity_diagnostics.to_csv(OUTPUT_DIR / "spatial_buffer_sensitivity_diagnostics.csv", index=False)

region_summary = (spatial_sensitivity_results.groupby(["buffer_distance", "buffer_multiplier", "model"])
    .agg(AUC_mean=("ROC_AUC", "mean"), AUC_sd=("ROC_AUC", "std"),
         min_retained_train_n=("min_retained_train_n", "min"))
    .reset_index())
fig, ax = plt.subplots(figsize=(9, 6))
for model_name, g in region_summary.groupby("model"):
    g = g.sort_values("buffer_distance")
    ax.errorbar(g.buffer_distance / 1000, g.AUC_mean, yerr=g.AUC_sd,
                marker="o", capsize=3, label=model_name)
ax.set_xlabel("Geodesic exclusion buffer (km; regional partitions fixed within seed)")
ax.set_ylabel("ROC-AUC")
ax.set_title("Sensitivity to geographic separation under fixed regional partitions")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_buffer_sensitivity_fixed_regions.png", dpi=300, bbox_inches="tight")
plt.show()


# 12. Validation Optimism Gap (VOG)

For higher-is-better metrics:

\[
VOG = M_{random} - M_{spatial}
\]

For Brier/ECE, higher error is worse, so the deterioration is:

\[
VOG_{error}=M_{spatial}-M_{random}
\]

In [ ]:
metric_cols = ["ROC_AUC","PR_AUC","Accuracy","Precision","Recall","F1","MCC","Kappa","Brier","ECE"]

rmean = random_results.groupby("model")[metric_cols].mean()
smean = spatial_results.groupby("model")[metric_cols].mean()

comparison = rmean.add_suffix("_random").join(smean.add_suffix("_spatial")).reset_index()

higher_better = ["ROC_AUC","PR_AUC","Accuracy","Precision","Recall","F1","MCC","Kappa"]
for m in higher_better:
    comparison[f"VOG_{m}"] = comparison[f"{m}_random"] - comparison[f"{m}_spatial"]
    comparison[f"RVOG_{m}_pct"] = comparison[f"VOG_{m}"] / comparison[f"{m}_spatial"].replace(0,np.nan) * 100

for m in ["Brier","ECE"]:
    comparison[f"VOG_{m}_error"] = comparison[f"{m}_spatial"] - comparison[f"{m}_random"]

comparison.to_csv(OUTPUT_DIR/"validation_optimism_gap.csv", index=False)
display(comparison.round(4))

# 13. Main figure — random versus spatial ROC-AUC

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

for i, row in comparison.iterrows():
    ax.plot(
        [i,i],
        [row.ROC_AUC_spatial, row.ROC_AUC_random],
        marker="o"
    )

ax.set_xticks(range(len(comparison)))
ax.set_xticklabels(comparison.model, rotation=35, ha="right")
ax.set_ylabel("ROC-AUC")
ax.set_title("Conventional versus spatially independent validation")
plt.tight_layout()
plt.savefig(OUTPUT_DIR/"fig_random_vs_spatial_auc.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. Calibration under random and buffered validation

Reliability curves use per-observation mean held-out predictions within each validation regime. They are descriptive summaries of repeated internal validation and remain conditional on the case/background sampling design. The cross-fitted probability calibration procedure is nested with hyperparameter selection as described above.

In [ ]:
def reliability_curve(y_true, prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    mean_pred, observed = [], []
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    for i in range(n_bins):
        mask = ((prob >= edges[i]) & (prob <= edges[i+1])) if i == n_bins-1 else (
            (prob >= edges[i]) & (prob < edges[i+1]))
        if mask.sum():
            mean_pred.append(prob[mask].mean())
            observed.append(y_true[mask].mean())
    return np.asarray(mean_pred), np.asarray(observed)

best_model = BEST_MODEL_NAME
rp = random_predictions[random_predictions.model == best_model]
sp = spatial_predictions[spatial_predictions.model == best_model]

# Compare only observations with held-out predictions under both schemes.
rmean_pred = rp.groupby("index").agg(y_true=("y_true", "first"), probability=("probability", "mean"))
smean_pred = sp.groupby("index").agg(y_true=("y_true", "first"), probability=("probability", "mean"))
common = rmean_pred.index.intersection(smean_pred.index)
if len(common) < 100:
    raise RuntimeError("Too few common held-out observations for the calibration comparison.")
if not np.array_equal(rmean_pred.loc[common, "y_true"].to_numpy(),
                      smean_pred.loc[common, "y_true"].to_numpy()):
    raise AssertionError("Random and spatial calibration subsets have misaligned labels.")

fig, ax = plt.subplots(figsize=(7, 7))
for table, label in [
    (rmean_pred.loc[common], "Repeated random: per-observation mean"),
    (smean_pred.loc[common], "Buffered spatial: per-observation mean"),
]:
    mp, fp = reliability_curve(table.y_true, table.probability, n_bins=10)
    ax.plot(mp, fp, marker="o", label=label)
ax.plot([0, 1], [0, 1], "--", color="black", linewidth=1, label="Ideal calibration")
ax.set_xlabel("Mean predicted susceptibility score")
ax.set_ylabel("Observed background/presence frequency")
ax.set_title(f"Calibration comparison: {best_model}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_calibration_random_vs_spatial.png", dpi=600, bbox_inches="tight")
plt.show()

calibration_summary = pd.DataFrame([
    {"validation": "random", "model": best_model,
     **calculate_metrics(rmean_pred.loc[common, "y_true"], rmean_pred.loc[common, "probability"])},
    {"validation": "buffered_spatial", "model": best_model,
     **calculate_metrics(smean_pred.loc[common, "y_true"], smean_pred.loc[common, "probability"])},
])
calibration_summary.to_csv(OUTPUT_DIR / "selected_model_calibration_comparison.csv", index=False)
display(calibration_summary.round(4))

## 15. Prediction variability across spatial resamples

For each observation, variability is calculated across the repeated buffered spatial-CV predictions from the selected model (one held-out prediction per partition seed). This measures sensitivity to the specific geographic partition/training subset; it is not a posterior uncertainty estimate and has not been calibrated against error.

In [ ]:
spatial_uncertainty = (
    spatial_predictions[spatial_predictions.model == BEST_MODEL_NAME]
    .groupby("index")["probability"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
    .rename(columns={"std": "prediction_sd"})
)
spatial_uncertainty["prediction_sd"] = spatial_uncertainty["prediction_sd"].fillna(0)
spatial_uncertainty.to_csv(OUTPUT_DIR / "spatial_cv_observation_prediction_variability.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(spatial_uncertainty.prediction_sd, bins=40)
ax.set_xlabel("Prediction SD across buffered spatial partitions")
ax.set_ylabel("Number of observations")
ax.set_title(f"Spatial-resampling prediction variability: {BEST_MODEL_NAME}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_spatial_prediction_variability.png", dpi=300, bbox_inches="tight")
plt.show()
display(spatial_uncertainty["prediction_sd"].describe(percentiles=[.5, .9, .95, .99]))

## 16. Verified training–testing geographic separation

This diagnostic MUST use the post-buffer training indices actually used in the final spatial validation. The previous version accidentally measured distance to the original unbuffered training points, which understated the realized separation and contradicted the enforced buffer.

The code below therefore uses the exact buffered training set for each outer fold and asserts that every test point is at least `BUFFER_DISTANCE` from the nearest retained training point.


In [ ]:


def nearest_train_distance(train_xy, test_xy):
    train_xy = np.asarray(train_xy, dtype=float)
    test_xy = np.asarray(test_xy, dtype=float)
    if len(train_xy) == 0:
        raise ValueError("No retained training observations remain after buffering.")
    return geodesic_nearest_distance_m(train_xy, test_xy)

separation_parts = []
for seed in SPATIAL_CV_SEEDS:
    buffered_splits, _, diagnostics = make_spatial_splits_buffered(
        coords, y, PRIMARY_BLOCK_SIZE, N_SPATIAL_FOLDS, seed, BUFFER_DISTANCE
    )
    for fold, (tr_buffered, te) in enumerate(buffered_splits):
        dmin = nearest_train_distance(coords[tr_buffered], coords[te])
        fold_df = pd.DataFrame({
            "partition_seed": seed, "fold": fold, "index": te,
            "nearest_training_distance_m": dmin,
            "nearest_training_distance_km": dmin / 1000.0,
            "buffer_distance_m": BUFFER_DISTANCE,
            "buffer_constraint_satisfied": dmin >= (BUFFER_DISTANCE - 1e-6),
        })
        if not fold_df["buffer_constraint_satisfied"].all():
            raise AssertionError(f"Buffer constraint violated in seed={seed}, fold={fold}.")
        separation_parts.append(fold_df)
separation_df = pd.concat(separation_parts, ignore_index=True)
print(f"Required exclusion distance: {BUFFER_DISTANCE/1000:.2f} km")
display(separation_df["nearest_training_distance_km"].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]))
display(separation_df.groupby(["partition_seed", "fold"])["nearest_training_distance_km"].min().rename("min_distance_km"))
separation_df.to_csv(OUTPUT_DIR / "training_testing_separation_buffered_verified.csv", index=False)


# 17. Susceptibility-map comparison — raster prediction template

Provide one raster per predictor in `RASTER_STACK`. All rasters must have identical CRS, grid, extent, transform, and alignment.

For the publication comparison, generate:
1. Random-validation-trained map
2. Spatially robust map
3. Difference map
4. Uncertainty map

Do not describe either map as "ground truth".

In [ ]:
RASTER_STACK = None

try:
    import rasterio
    RASTERIO_AVAILABLE = True
except Exception:
    RASTERIO_AVAILABLE = False

def _open_and_check_raster_stack(raster_stack, feature_cols):
    """Open all predictor rasters once and verify exact grid alignment."""
    if not RASTERIO_AVAILABLE:
        raise ImportError("rasterio is required.")
    if not feature_cols:
        raise ValueError("feature_cols is empty.")

    ref_path = raster_stack[feature_cols[0]]
    ref = rasterio.open(ref_path)

    sig_ref = (
        ref.crs, ref.width, ref.height, tuple(ref.transform),
        tuple(ref.bounds), tuple(ref.res)
    )
    sources = [ref]

    for col in feature_cols[1:]:
        src = rasterio.open(raster_stack[col])
        sig = (
            src.crs, src.width, src.height, tuple(src.transform),
            tuple(src.bounds), tuple(src.res)
        )
        if sig != sig_ref:
            src.close()
            for opened in sources:
                opened.close()
            raise ValueError(
                f"Raster '{col}' is not aligned to reference '{feature_cols[0]}'."
            )
        sources.append(src)

    return ref, sources

def predict_raster_windowed(model, raster_stack, feature_cols, output_path, nodata=-9999):
    ref, sources = _open_and_check_raster_stack(raster_stack, feature_cols)
    try:
        profile = ref.profile.copy()
        profile.update(dtype="float32", count=1, nodata=nodata, compress="deflate")

        with rasterio.open(output_path, "w", **profile) as dst:
            for _, window in ref.block_windows(1):
                arrays = []
                valid = np.ones(window.height * window.width, dtype=bool)

                for col, src in zip(feature_cols, sources):
                    a = src.read(1, window=window).astype("float32").reshape(-1)
                    arrays.append(a)
                    ok = np.isfinite(a)
                    if src.nodata is not None:
                        ok &= a != src.nodata
                    valid &= ok

                stack = np.column_stack(arrays)
                valid &= np.all(np.isfinite(stack), axis=1)

                out = np.full(stack.shape[0], nodata, dtype="float32")
                if valid.any():
                    Xw = pd.DataFrame(stack[valid], columns=feature_cols)
                    for c in CATEGORICAL_FEATURES:
                        if c in Xw.columns:
                            Xw[c] = Xw[c].astype(str)
                    p = get_probability(model, Xw)
                    out[valid] = p.astype("float32")

                dst.write(
                    out.reshape(window.height, window.width), 1, window=window
                )
    finally:
        for src in sources:
            src.close()

    return output_path

print("Windowed raster prediction template ready.")

### 17a. Populate `RASTER_STACK` and verify alignment

Reuses your predictor-to-raster mapping and alignment check; the previous version's fix
was preserved (fixed here rather than in cell 41's template, since this cell is what's
actually run for the final flood prediction).

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio

# Use the same harmonized target rasters used to build the modelling table.
missing_mapping = [c for c in FEATURE_COLS if c not in raster_paths]
if missing_mapping:
    raise ValueError(f"No corrected harmonized raster registered for: {missing_mapping}")
RASTER_STACK = {c: str(raster_paths[c]) for c in FEATURE_COLS}
missing = [(f, p) for f, p in RASTER_STACK.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError("Corrected raster(s) missing: " + repr(missing))

reference_feature = FEATURE_COLS[0]
with rasterio.open(RASTER_STACK[reference_feature]) as ref:
    ref_crs, ref_transform, ref_width, ref_height = ref.crs, ref.transform, ref.width, ref.height

misaligned = []
for feature in FEATURE_COLS:
    with rasterio.open(RASTER_STACK[feature]) as src:
        same_shape = (src.width == ref_width and src.height == ref_height)
        same_crs = (src.crs == ref_crs)
        same_transform = src.transform.almost_equals(ref_transform)
        if not (same_shape and same_crs and same_transform):
            misaligned.append(feature)
        if src.crs != TARGET_CRS:
            raise ValueError(f"Map predictor {feature} is not in the corrected target CRS.")
if misaligned:
    raise ValueError(f"Corrected map stack is not aligned: {misaligned}")
if not np.allclose((ref_transform.a, abs(ref_transform.e)), (1000.0, 1000.0), atol=1e-6):
    raise ValueError("Map grid is not 1-km in the corrected equal-area projection.")
print(f"Verified {len(RASTER_STACK)} aligned 1-km predictors in {ref_crs}.")


### 17b. Build the comparison models

Two map products are retained for the methodological comparison: (A) a size-matched random-training reference and (B) the spatial-fold ensemble mean. A separate operational surface is produced from the selected model fitted and calibrated using the complete corrected inventory. No independent geographic test is performed.

### 17b. Build spatial-fold and size-matched random-reference models

The final operational surface is fitted on the complete inventory. For map comparison, each spatial-fold model from the primary partition seed is paired with a random-reference model trained on the same number of observations and the same class counts. Both are calibrated with nested cross-fitting.

In [ ]:
print("Selected model for full-inventory mapping:", BEST_MODEL_NAME)
spatial_fold_models = [
    model for (seed, fold, name), model in sorted(spatial_outer_models_by_repeat.items())
    if name == BEST_MODEL_NAME and seed == RANDOM_STATE_FINAL
]
expected_spatial_models = N_SPATIAL_FOLDS  # primary partition seed only
if len(spatial_fold_models) != expected_spatial_models:
    raise RuntimeError(f"Expected {expected_spatial_models} calibrated spatial fold models, found {len(spatial_fold_models)}.")

train_size_table = (buffer_diagnostics[buffer_diagnostics["repeat"] == RANDOM_STATE_FINAL].sort_values(["repeat", "fold"])[
    ["repeat", "fold", "n_train_buffered", "n_train_presence", "n_train_background"]
].to_dict("records"))
if len(train_size_table) != len(spatial_fold_models):
    raise RuntimeError("Spatial fold support records do not match the fitted spatial model ensemble.")

random_subsample_models = []
all_positive = np.where(np.asarray(y, dtype=int) == 1)[0]
all_background = np.where(np.asarray(y, dtype=int) == 0)[0]
for i, train_spec in enumerate(train_size_table):
    n_train = int(train_spec["n_train_buffered"])
    n_pos = int(train_spec["n_train_presence"])
    n_neg = int(train_spec["n_train_background"])
    if n_train >= len(X):
        raise ValueError("Matched random sample size must be smaller than the full inventory.")
    if n_pos + n_neg != n_train or n_pos < 1 or n_neg < 1:
        raise ValueError(f"Invalid matched class counts: {train_spec}.")
    rng_map = np.random.default_rng(RANDOM_STATE_FINAL + i * 101)
    idx = np.concatenate([
        rng_map.choice(all_positive, size=n_pos, replace=False),
        rng_map.choice(all_background, size=n_neg, replace=False),
    ])
    rng_map.shuffle(idx)
    y_i = np.asarray(y.iloc[idx], dtype=int)
    if len(idx) != n_train or int(y_i.sum()) != n_pos:
        raise AssertionError("Random map-reference model failed size/class matching.")
    calibrated_random, info = fit_nested_calibrated_model(
        BEST_MODEL_NAME, MODELS[BEST_MODEL_NAME], X.iloc[idx], y_i,
        coords[idx], cv_scheme="spatial",
        seed=RANDOM_STATE_FINAL + 60000 + i,
        n_iter_final=N_ITER_SEARCH,
        n_iter_calibration=N_ITER_CALIBRATION_SEARCH
    )
    random_subsample_models.append(calibrated_random)
    print(f"Random reference {i+1}/{len(train_size_table)}: n_train={n_train}")

print(f"Prepared {len(spatial_fold_models)} spatial-fold models and {len(random_subsample_models)} size-matched random reference models.")
print("Mapping ensemble uses the prespecified primary spatial partition seed only; it is not a three-seed ensemble.")

### 17c. Generate the susceptibility maps (random, spatially-robust mean, spatial-fold prediction SD, and difference)

Generates the full-inventory operational susceptibility surface, the size-matched random-reference ensemble, the primary-seed spatial-fold ensemble mean, the spatial-fold prediction SD and their difference.

In [ ]:
def predict_raster_windowed_ensemble(
    models, raster_stack, feature_cols, mean_output_path, std_output_path, nodata=-9999
):
    """Predict all models window-by-window while keeping raster handles open."""
    ref, sources = _open_and_check_raster_stack(raster_stack, feature_cols)
    try:
        profile = ref.profile.copy()
        profile.update(dtype="float32", count=1, nodata=nodata, compress="deflate")

        with rasterio.open(mean_output_path, "w", **profile) as dst_mean, \
             rasterio.open(std_output_path, "w", **profile) as dst_std:

            windows = list(ref.block_windows(1))
            total_windows = len(windows)
            for window_i, (_, window) in enumerate(windows, start=1):
                arrays = []
                valid = np.ones(window.height * window.width, dtype=bool)

                for col, src in zip(feature_cols, sources):
                    a = src.read(1, window=window).astype("float32").reshape(-1)
                    arrays.append(a)
                    ok = np.isfinite(a)
                    if src.nodata is not None:
                        ok &= a != src.nodata
                    valid &= ok

                stack = np.column_stack(arrays)
                valid &= np.all(np.isfinite(stack), axis=1)

                out_mean = np.full(stack.shape[0], nodata, dtype="float32")
                out_std = np.full(stack.shape[0], nodata, dtype="float32")

                if valid.any():
                    Xw = pd.DataFrame(stack[valid], columns=feature_cols)
                    for c in CATEGORICAL_FEATURES:
                        if c in Xw.columns:
                            Xw[c] = Xw[c].astype(str)

                    probs = np.column_stack([
                        get_probability(model, Xw) for model in models
                    ])
                    out_mean[valid] = probs.mean(axis=1).astype("float32")
                    out_std[valid] = probs.std(axis=1).astype("float32")

                dst_mean.write(
                    out_mean.reshape(window.height, window.width), 1, window=window
                )
                dst_std.write(
                    out_std.reshape(window.height, window.width), 1, window=window
                )
                if window_i == 1 or window_i % max(1, total_windows // 20) == 0 or window_i == total_windows:
                    print(f"Raster prediction windows: {window_i}/{total_windows} ({100*window_i/total_windows:.1f}%)", flush=True)
    finally:
        for src in sources:
            src.close()

    return mean_output_path, std_output_path


MAP_DIR = OUTPUT_DIR / "susceptibility_maps"
MAP_DIR.mkdir(parents=True, exist_ok=True)

path_final_map = MAP_DIR / f"susceptibility_FINAL_full_inventory_{BEST_MODEL_NAME.replace(' ','_')}.tif"
predict_raster_windowed(final_model, RASTER_STACK, FEATURE_COLS, path_final_map)

path_random_mean, path_random_std = predict_raster_windowed_ensemble(
    random_subsample_models, RASTER_STACK, FEATURE_COLS,
    MAP_DIR / f"susceptibility_random_matched_mean_{BEST_MODEL_NAME.replace(' ','_')}.tif",
    MAP_DIR / f"susceptibility_random_matched_sd_{BEST_MODEL_NAME.replace(' ','_')}.tif",
)

path_spatial_mean, path_spatial_std = predict_raster_windowed_ensemble(
    spatial_fold_models, RASTER_STACK, FEATURE_COLS,
    MAP_DIR / f"susceptibility_spatially_robust_mean_{BEST_MODEL_NAME.replace(' ','_')}.tif",
    MAP_DIR / f"susceptibility_spatially_robust_prediction_sd_{BEST_MODEL_NAME.replace(' ','_')}.tif",
)

nodata = -9999  # fix: this name is used below but was never defined at this scope
path_diff = MAP_DIR / f"susceptibility_difference_{BEST_MODEL_NAME.replace(' ','_')}.tif"
with rasterio.open(path_random_mean) as ra, rasterio.open(path_spatial_mean) as sp:
    if (
        ra.crs != sp.crs or ra.width != sp.width or ra.height != sp.height or
        tuple(ra.transform) != tuple(sp.transform)
    ):
        raise ValueError("Random and spatial output rasters are not grid-aligned.")

    profile = ra.profile.copy()
    with rasterio.open(path_diff, "w", **profile) as dst:
        for _, window in ra.block_windows(1):
            a = ra.read(1, window=window)
            s = sp.read(1, window=window)
            valid = (a != nodata) & (s != nodata) & np.isfinite(a) & np.isfinite(s)
            out = np.full(a.shape, nodata, dtype="float32")
            out[valid] = (a[valid] - s[valid]).astype("float32")
            dst.write(out, 1, window=window)

path_random = path_random_mean
print("Random matched reference map:", path_random_mean)
print("Random matched SD map:", path_random_std)
print("Spatial cross-fitted mean map:", path_spatial_mean)
print("Full-inventory operational score map:", path_final_map)
print("Spatial-fold prediction SD map:", path_spatial_std)
print("Difference map (random − spatial):", path_diff)

In [ ]:
# ---- Fix: write ONLY the difference raster (the prediction rasters already exist on disk) ----
nodata = -9999          # global default; later cells rely on it as well

MAP_DIR = OUTPUT_DIR / "susceptibility_maps"
_tag = BEST_MODEL_NAME.replace(" ", "_")
path_final_map    = MAP_DIR / f"susceptibility_FINAL_full_inventory_{_tag}.tif"
path_random_mean  = MAP_DIR / f"susceptibility_random_matched_mean_{_tag}.tif"
path_random_std   = MAP_DIR / f"susceptibility_random_matched_sd_{_tag}.tif"
path_spatial_mean = MAP_DIR / f"susceptibility_spatially_robust_mean_{_tag}.tif"
path_spatial_std  = MAP_DIR / f"susceptibility_spatially_robust_prediction_sd_{_tag}.tif"
path_diff         = MAP_DIR / f"susceptibility_difference_{_tag}.tif"

_missing = [p.name for p in (path_final_map, path_random_mean, path_random_std,
                             path_spatial_mean, path_spatial_std) if not p.exists()]
if _missing:
    raise FileNotFoundError(f"These rasters are missing, so the prediction step has to be rerun: {_missing}")

with rasterio.open(path_random_mean) as ra, rasterio.open(path_spatial_mean) as sp:
    if (ra.crs != sp.crs or ra.width != sp.width or ra.height != sp.height
            or tuple(ra.transform) != tuple(sp.transform)):
        raise ValueError("Random and spatial output rasters are not grid-aligned.")
    profile = ra.profile.copy()
    with rasterio.open(path_diff, "w", **profile) as dst:
        for _, window in ra.block_windows(1):
            a = ra.read(1, window=window)
            s = sp.read(1, window=window)
            valid = (a != nodata) & (s != nodata) & np.isfinite(a) & np.isfinite(s)
            out = np.full(a.shape, nodata, dtype="float32")
            out[valid] = (a[valid] - s[valid]).astype("float32")
            dst.write(out, 1, window=window)

path_random = path_random_mean
print("Difference map (random − spatial):", path_diff)

### 17d. Quantify map differences (Table S9 / manuscript §3.9)

In [ ]:
from scipy.stats import pearsonr, spearmanr

def raster_profile_signature(path):
    with rasterio.open(path) as src:
        return (
            str(src.crs), src.width, src.height,
            tuple(src.transform), tuple(src.bounds), tuple(src.res)
        )

sig_random = raster_profile_signature(path_random)
sig_spatial = raster_profile_signature(path_spatial_mean)
if sig_random != sig_spatial:
    raise ValueError("Random and spatial susceptibility rasters are not grid-aligned.")

def read_valid(path, nodata=-9999):
    with rasterio.open(path) as src:
        arr = src.read(1)
        nd = src.nodata if src.nodata is not None else nodata
        transform = src.transform
        pixel_area_km2 = abs(transform.a * transform.e) / 1e6
        return arr, np.isfinite(arr) & (arr != nd), pixel_area_km2

arr_r, valid_r, pixel_area_km2 = read_valid(path_random)
arr_s, valid_s, _ = read_valid(path_spatial_mean)
valid = valid_r & valid_s

r_flat, s_flat = arr_r[valid], arr_s[valid]
if len(r_flat) < 100:
    raise RuntimeError("Too few valid cells for map comparison.")

rmse = float(np.sqrt(np.mean((r_flat - s_flat) ** 2)))
mae = float(np.mean(np.abs(r_flat - s_flat)))
pearson_r, _ = pearsonr(r_flat, s_flat)
sample_n = min(250000, len(r_flat))
rng_map = np.random.default_rng(RANDOM_STATE_FINAL)
sample_idx = rng_map.choice(len(r_flat), size=sample_n, replace=False)
spearman_r, _ = spearmanr(r_flat[sample_idx], s_flat[sample_idx])

summary_rows = []

# Threshold comparison: absolute score threshold 0.5 and independent top-20% classes.
high_r_05 = r_flat >= 0.5
high_s_05 = s_flat >= 0.5
intersection = int((high_r_05 & high_s_05).sum())
union = int((high_r_05 | high_s_05).sum())
summary_rows.append({
    "threshold_definition": "score >= 0.5",
    "random_high_area_km2": float(high_r_05.sum() * pixel_area_km2),
    "spatial_high_area_km2": float(high_s_05.sum() * pixel_area_km2),
    "area_difference_km2": float((high_s_05.sum() - high_r_05.sum()) * pixel_area_km2),
    "jaccard_similarity": intersection / union if union else np.nan,
})

q_r = np.quantile(r_flat, 0.80)
q_s = np.quantile(s_flat, 0.80)
high_r_20 = r_flat >= q_r
high_s_20 = s_flat >= q_s
intersection = int((high_r_20 & high_s_20).sum())
union = int((high_r_20 | high_s_20).sum())
summary_rows.append({
    "threshold_definition": "top 20% within each surface",
    "random_high_area_km2": float(high_r_20.sum() * pixel_area_km2),
    "spatial_high_area_km2": float(high_s_20.sum() * pixel_area_km2),
    "area_difference_km2": float((high_s_20.sum() - high_r_20.sum()) * pixel_area_km2),
    "jaccard_similarity": intersection / union if union else np.nan,
})

map_comparison = pd.DataFrame(summary_rows)
map_comparison["RMSE"] = rmse
map_comparison["MAE"] = mae
map_comparison["pearson_r"] = pearson_r
map_comparison["spearman_r"] = spearman_r
map_comparison["model"] = BEST_MODEL_NAME
map_comparison["spearman_sample_n"] = sample_n
map_comparison["map_reference"] = "size-matched random reference vs spatial cross-fitted mean"
map_comparison["hazard"] = "flood"
map_comparison["score_interpretation"] = "relative susceptibility score; not absolute flood probability"

map_comparison.to_csv(OUTPUT_DIR / "map_comparison_summary.csv", index=False)
display(map_comparison.round(4))

print(f"RMSE: {rmse:.4f}")
print(f"Pearson r: {pearson_r:.4f} | Spearman r: {spearman_r:.4f}")

### 17e. Map figure (random vs. spatially-robust vs. difference vs. spatial-fold prediction SD)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, path, title, cmap in [
    (axes[0,0], path_final_map,  f"Final operational map ({BEST_MODEL_NAME})", "YlOrRd"),
    (axes[0,1], path_spatial_mean,"Spatial cross-fitted mean",   "YlOrRd"),
    (axes[1,0], path_diff,        "Matched random − spatial cross-fitted",       "RdBu_r"),
    (axes[1,1], path_spatial_std, "Spatial-fold prediction variability (SD)",             "viridis"),
]:
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        arr[arr == -9999] = np.nan
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(title, fontsize=11)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.04)

plt.suptitle(f"Flood susceptibility: validation-scheme comparison ({BEST_MODEL_NAME})", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_susceptibility_map_comparison.png", dpi=600, bbox_inches="tight")
plt.show()


## 18. Supplementary descriptive spatial clustering and country summaries

Gi* and country-level summaries are computed from the full-inventory fitted susceptibility surface. They describe spatial concentration in model output; they do not provide independent validation of flood occurrences. Because the modelled surface is spatially autocorrelated, inferential interpretations of analytic Gi* z-scores are limited. Treat these outputs as descriptive supplementary material.

In [ ]:
# ============================================================
# FINAL NATIVE-GRID Gi* SUMMARY AND COUNTRY STATISTICS — CORRECTED
# ============================================================
# All regional and country summaries use the same unique country-union mask.
# NoData cells are never counted as non-significant; country overlap is audited;
# pixel area is calculated from the affine determinant.

import json
from scipy.ndimage import convolve
from scipy.stats import norm
from rasterio.features import rasterize
from rasterio.enums import MergeAlg
from pyproj import CRS as PyCRS

FINAL_RASTER = Path(path_final_map)
GI_DIR = OUTPUT_DIR / "final_gistar"
GI_DIR.mkdir(parents=True, exist_ok=True)

# Use a locally archived country boundary for the final submission whenever possible.
COUNTRY_VECTOR_ENV = os.environ.get("FLOOD_GEOAI_COUNTRY_VECTOR_PATH", "").strip()
COUNTRY_VECTOR_PATH = Path(COUNTRY_VECTOR_ENV).expanduser() if COUNTRY_VECTOR_ENV else None
# Prefer a locally archived, version-pinned country boundary for strict reproduction.
# If no path is configured, the existing remote Natural Earth fallback is used and
# the output manifest records that the remote source was not version-pinned.
COUNTRY_NAME_FIELD = "ADMIN"
TARGET_COUNTRIES = ["Afghanistan", "Bangladesh", "Bhutan", "India", "Nepal", "Pakistan", "Sri Lanka"]

if COUNTRY_VECTOR_PATH:
    countries = gpd.read_file(COUNTRY_VECTOR_PATH)
    boundary_source = str(COUNTRY_VECTOR_PATH)
else:
    import warnings
    warnings.warn(
        "Using a remote Natural Earth fallback. For a fully reproducible submission, "
        "download and pin the boundary version locally, then set COUNTRY_VECTOR_PATH."
    )
    countries = gpd.read_file(
        "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
    )
    boundary_source = "Natural Earth 110m remote fallback; version not pinned in this run"

if COUNTRY_NAME_FIELD not in countries.columns:
    raise KeyError(f"Country name field '{COUNTRY_NAME_FIELD}' not found. Fields: {countries.columns.tolist()}")
countries = countries.to_crs(TARGET_CRS)
countries = countries[countries[COUNTRY_NAME_FIELD].isin(TARGET_COUNTRIES)].copy()
found = set(countries[COUNTRY_NAME_FIELD].unique())
missing_countries = sorted(set(TARGET_COUNTRIES) - found)
if missing_countries:
    raise ValueError(f"Missing target country polygons: {missing_countries}")

with rasterio.open(FINAL_RASTER) as src:
    z0 = src.read(1).astype("float64")
    transform = src.transform
    crs = src.crs
    profile = src.profile.copy()
    raster_nodata = src.nodata if src.nodata is not None else -9999
    if crs is None or not crs.is_projected:
        raise ValueError("Final susceptibility raster must use a projected CRS.")
    if PyCRS.from_user_input(crs) != TARGET_CRS:
        raise ValueError(
            "Final susceptibility map is not in the corrected target CRS. "
            "Rerun map prediction using Cell 63's harmonized target rasters."
        )
    determinant = abs(transform.a * transform.e - transform.b * transform.d)
    pixel_area_km2 = determinant / 1e6
    if not np.isclose(pixel_area_km2, 1.0, atol=1e-5):
        raise ValueError(f"Expected 1-km equal-area cells; affine cell area is {pixel_area_km2:.6f} km².")
    raster_valid = np.isfinite(z0) & (z0 != raster_nodata)
    raster_shape = z0.shape

# Rasterize the seven countries into one unique integer-ID grid.
country_order = list(TARGET_COUNTRIES)
country_id_lookup = {name: i + 1 for i, name in enumerate(country_order)}
shapes = [(row.geometry, country_id_lookup[row[COUNTRY_NAME_FIELD]])
          for _, row in countries.iterrows()]
country_id = rasterize(
    shapes, out_shape=raster_shape, transform=transform,
    fill=0, dtype="int16", all_touched=False, merge_alg=MergeAlg.replace
)
study_mask = country_id > 0
valid = raster_valid & study_mask
N = int(valid.sum())
if N < 100:
    raise RuntimeError("Too few valid cells within the seven-country study mask.")

# Detect whether independent country rasterizations overlap at any cell centre.
individual_masks = []
for _, row in countries.iterrows():
    cm = rasterize([(row.geometry, 1)], out_shape=raster_shape, transform=transform,
                   fill=0, dtype="uint8", all_touched=False).astype(bool)
    individual_masks.append(cm)
individual_count = np.sum(np.stack(individual_masks, axis=0), axis=0)
overlap_cells = int(np.sum(individual_count > 1))
if overlap_cells > 0:
    raise ValueError(
        f"Country polygons overlap on {overlap_cells} raster cell centres. "
        "Resolve the boundary topology before reporting country totals; do not double-count cells."
    )

# Local Gi* using a 3x3 queen neighbourhood. Invalid cells outside countries or
# with missing predictors are masked, not filled as zeros in the denominator.
x = np.where(valid, z0, 0.0)
mask = valid.astype("float64")
kernel = np.ones((3, 3), dtype="float64")
local_n = convolve(mask, kernel, mode="constant", cval=0.0)
local_sum = convolve(x, kernel, mode="constant", cval=0.0)
xbar = float(z0[valid].mean())
S = float(np.sqrt(np.sum((z0[valid] - xbar) ** 2) / N))
denominator = S * np.sqrt(np.maximum((N * local_n - local_n ** 2) / max(N - 1, 1), 0.0))
zgi = np.full_like(z0, np.nan, dtype="float64")
ok = valid & (local_n > 1) & (denominator > 0)
zgi[ok] = (local_sum[ok] - xbar * local_n[ok]) / denominator[ok]

# Nominal p-values with BH-FDR; because the modelled surface is spatially
# autocorrelated, these are treated as descriptive concentration bands only.
p_nom = np.full_like(zgi, np.nan)
p_nom[ok] = 2.0 * norm.sf(np.abs(zgi[ok]))
pvals = p_nom[ok]
m = len(pvals)
order = np.argsort(pvals)
ranked = pvals[order]
q_ranked = ranked * m / np.arange(1, m + 1)
q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
q_ranked = np.clip(q_ranked, 0, 1)
qvals = np.full(m, np.nan)
qvals[order] = q_ranked
q = np.full_like(zgi, np.nan)
q[ok] = qvals

# -99 is explicit NoData; class 0 means valid but not in an FDR-retained band.
cls = np.full(z0.shape, -99, dtype=np.int16)
cls[valid] = 0
sig = valid & np.isfinite(zgi) & (q <= FDR_Q)
cls[sig & (zgi >= 2.576)] = 3
cls[sig & (zgi >= 1.960) & (zgi < 2.576)] = 2
cls[sig & (zgi >= 1.645) & (zgi < 1.960)] = 1
cls[sig & (zgi <= -1.645) & (zgi > -1.960)] = -1
cls[sig & (zgi <= -1.960) & (zgi > -2.576)] = -2
cls[sig & (zgi <= -2.576)] = -3

class_defs = {
    3: "99% high-score cluster band", 2: "95% high-score cluster band",
    1: "90% high-score cluster band", 0: "not in an FDR-retained cluster",
    -1: "90% low-score cluster band", -2: "95% low-score cluster band",
    -3: "99% low-score cluster band"
}
summary_rows = []
for code, label in class_defs.items():
    n_cells = int(np.sum(cls[valid] == code))
    summary_rows.append({
        "cluster_band": label, "class_code": code, "n_cells": n_cells,
        "area_km2": n_cells * pixel_area_km2,
        "share_valid_study_area_pct": 100.0 * n_cells / N
    })
gi_summary = pd.DataFrame(summary_rows)
if int(gi_summary.n_cells.sum()) != N:
    raise AssertionError("Gi* class counts do not sum to the valid study-area cell count.")
gi_summary.to_csv(GI_DIR / "gi_star_domain_summary.csv", index=False)

# Raster outputs retain exactly the country-union validity mask.
def write_raster(path, arr, dtype="float32", nodata_value=-9999):
    prof = profile.copy()
    prof.update(dtype=dtype, count=1, nodata=nodata_value, compress="deflate")
    arr = np.asarray(arr)
    out = np.full(arr.shape, nodata_value, dtype=dtype)
    finite = np.isfinite(arr)
    out[finite] = arr[finite].astype(dtype)
    with rasterio.open(path, "w", **prof) as dst:
        dst.write(out, 1)

write_raster(GI_DIR / "gi_star_z_country_masked_1km.tif", zgi, "float32", -9999)
write_raster(GI_DIR / "gi_star_p_nominal_country_masked_1km.tif", p_nom, "float32", -9999)
write_raster(GI_DIR / "gi_star_q_fdr_country_masked_1km.tif", q, "float32", -9999)
write_raster(GI_DIR / "gi_star_class_fdr_country_masked_1km.tif",
             np.where(valid, cls, np.nan), "int16", -99)

# Quintile classification on the same unique valid-cell mask.
vals = z0[valid]
quintile_breaks = np.quantile(vals, [0, .2, .4, .6, .8, 1.0])
if len(np.unique(quintile_breaks)) != 6:
    raise ValueError("Susceptibility score quantiles contain ties; equal-area five-class breaks are not unique.")
susc_class = np.full(z0.shape, -99, dtype=np.int16)
susc_class[valid] = np.digitize(vals, quintile_breaks[1:-1], right=True) + 1
if not np.isin(susc_class[valid], [1, 2, 3, 4, 5]).all():
    raise AssertionError("Unexpected susceptibility class code.")
write_raster(GI_DIR / "susceptibility_5class_final.tif",
             np.where(valid, susc_class, np.nan), "int16", -99)

country_gi_rows, country_susc_rows = [], []
for country_name in country_order:
    cid = country_id_lookup[country_name]
    cmask = (country_id == cid)
    mapped = valid & cmask
    n_country = int(mapped.sum())
    if n_country == 0:
        raise RuntimeError(f"No valid mapped cells for {country_name}.")
    ccls = cls[mapped]
    csusc = susc_class[mapped]
    geom = countries.loc[countries[COUNTRY_NAME_FIELD] == country_name, "geometry"].unary_union
    area_km2 = gpd.GeoSeries([geom], crs=crs).to_crs("EPSG:6933").iloc[0].area / 1e6
    country_gi_rows.append({
        "Country": country_name,
        "national_polygon_area_km2": float(area_km2),
        "mapped_cells": n_country,
        "mapped_area_km2": n_country * pixel_area_km2,
        "mapped_coverage_pct": 100.0 * n_country * pixel_area_km2 / area_km2,
        "high99_cluster_pct_of_mapped": 100.0 * np.sum(ccls == 3) / n_country,
        "high95_cluster_pct_of_mapped": 100.0 * np.sum(ccls == 2) / n_country,
        "high90_cluster_pct_of_mapped": 100.0 * np.sum(ccls == 1) / n_country,
        "high_total_cluster_pct_of_mapped": 100.0 * np.isin(ccls, [1,2,3]).sum() / n_country,
        "low90_cluster_pct_of_mapped": 100.0 * np.sum(ccls == -1) / n_country,
        "low95_cluster_pct_of_mapped": 100.0 * np.sum(ccls == -2) / n_country,
        "low99_cluster_pct_of_mapped": 100.0 * np.sum(ccls == -3) / n_country,
        "low_total_cluster_pct_of_mapped": 100.0 * np.isin(ccls, [-1,-2,-3]).sum() / n_country,
    })
    high_vh = np.isin(csusc, [4, 5])
    country_susc_rows.append({
        "Country": country_name,
        "mapped_cells": n_country,
        "mapped_area_km2": n_country * pixel_area_km2,
        "VeryLow_pct": 100.0 * np.sum(csusc == 1) / n_country,
        "Low_pct": 100.0 * np.sum(csusc == 2) / n_country,
        "Moderate_pct": 100.0 * np.sum(csusc == 3) / n_country,
        "High_pct": 100.0 * np.sum(csusc == 4) / n_country,
        "VeryHigh_pct": 100.0 * np.sum(csusc == 5) / n_country,
        "HighPlusVeryHigh_pct": 100.0 * high_vh.sum() / n_country,
        "HighPlusVeryHigh_km2": high_vh.sum() * pixel_area_km2,
    })

country_gi = pd.DataFrame(country_gi_rows).sort_values("Country")
country_susc = pd.DataFrame(country_susc_rows).sort_values("Country")
country_gi.to_csv(GI_DIR / "country_gistar_summary.csv", index=False)
country_susc.to_csv(GI_DIR / "country_susceptibility_summary.csv", index=False)

# Area reconciliation: unique country-assigned cells must sum exactly to the
# country-union valid cells. This is the primary check against previous totals
# that exceeded the valid raster area.
country_cell_total = int(country_gi["mapped_cells"].sum())
if country_cell_total != N:
    raise AssertionError(
        f"Country mapped cells ({country_cell_total}) do not equal valid country-union cells ({N})."
    )

class_counts = {int(code): int(np.sum(cls[valid] == code)) for code in class_defs}
regional_gi = {
    "mapped_cells": N,
    "mapped_area_km2": N * pixel_area_km2,
    "high99_area_km2": class_counts[3] * pixel_area_km2,
    "high95_area_km2": class_counts[2] * pixel_area_km2,
    "high90_area_km2": class_counts[1] * pixel_area_km2,
    "low90_area_km2": class_counts[-1] * pixel_area_km2,
    "low95_area_km2": class_counts[-2] * pixel_area_km2,
    "low99_area_km2": class_counts[-3] * pixel_area_km2,
    "not_significant_area_km2": class_counts[0] * pixel_area_km2,
}
regional_gi["high_cluster_area_km2"] = sum(regional_gi[k] for k in
    ["high99_area_km2", "high95_area_km2", "high90_area_km2"])
regional_gi["low_cluster_area_km2"] = sum(regional_gi[k] for k in
    ["low99_area_km2", "low95_area_km2", "low90_area_km2"])
regional_gi["high_cluster_pct_of_mapped"] = 100.0 * regional_gi["high_cluster_area_km2"] / regional_gi["mapped_area_km2"]
regional_gi["low_cluster_pct_of_mapped"] = 100.0 * regional_gi["low_cluster_area_km2"] / regional_gi["mapped_area_km2"]
regional_gi["not_significant_pct_of_mapped"] = 100.0 * regional_gi["not_significant_area_km2"] / regional_gi["mapped_area_km2"]
pd.DataFrame([regional_gi]).to_csv(GI_DIR / "regional_gistar_summary.csv", index=False)

pd.DataFrame({
    "class": ["VeryLow", "Low", "Moderate", "High", "VeryHigh"],
    "lower_bound": quintile_breaks[:-1], "upper_bound": quintile_breaks[1:]
}).to_csv(GI_DIR / "susceptibility_quintile_breaks.csv", index=False)

mask_qc = {
    "raster_valid_cells_before_country_mask": int(raster_valid.sum()),
    "country_union_cells": int(study_mask.sum()),
    "valid_cells_inside_country_union": int(N),
    "country_overlap_cell_centres": overlap_cells,
    "country_mapped_cell_sum": country_cell_total,
    "pixel_area_km2": float(pixel_area_km2),
    "country_boundary_source": boundary_source,
    "country_name_field": COUNTRY_NAME_FIELD,
    "target_countries": TARGET_COUNTRIES,
    "class_counts_sum": int(sum(class_counts.values())),
    "valid_cells_classified": int(N),
}
if mask_qc["class_counts_sum"] != N:
    raise AssertionError("Gi* summary classes do not cover all valid study cells exactly once.")
(GI_DIR / "country_mask_and_area_qc.json").write_text(
    json.dumps(mask_qc, indent=2, default=str), encoding="utf-8"
)

print("Valid unique country-union cells:", N)
print("Valid mapped area (km²):", N * pixel_area_km2)
print("Country mapped-cell sum:", country_cell_total)
print("Gi* class counts (sum must equal valid cells):", class_counts)
display(gi_summary)
display(country_gi.round(3))
display(country_susc.round(3))
display(pd.DataFrame([regional_gi]).round(3))


## 17g. Final publication figure — susceptibility and descriptive Gi* clustering

The final map figure is generated directly from the operational susceptibility raster and the native-resolution FDR-aware Gi* outputs.

In [ ]:
# Publication-ready final spatial figure.
from matplotlib.colors import TwoSlopeNorm

with rasterio.open(FINAL_RASTER) as src:
    final_arr = src.read(1).astype(float)
    final_arr[final_arr == src.nodata] = np.nan
with rasterio.open(GI_DIR / "gi_star_z_country_masked_1km.tif") as src:
    z_arr = src.read(1).astype(float)
    z_arr[z_arr == src.nodata] = np.nan
with rasterio.open(GI_DIR / "gi_star_q_fdr_country_masked_1km.tif") as src:
    q_arr = src.read(1).astype(float)
    q_arr[q_arr == src.nodata] = np.nan

fdr_z = z_arr.copy()
fdr_z[q_arr > FDR_Q] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), constrained_layout=True)

im0 = axes[0].imshow(fdr_z, cmap="RdBu_r", vmin=-5, vmax=5)
axes[0].set_title("(a) FDR-adjusted Gi* score clusters", fontsize=14, fontweight="bold", loc="left")
axes[0].axis("off")
cb0 = fig.colorbar(im0, ax=axes[0], fraction=0.035, pad=0.02)
cb0.set_label("Gi* z-score", fontsize=11)

im1 = axes[1].imshow(final_arr, cmap="YlOrRd", vmin=0, vmax=1)
axes[1].set_title(f"(b) Final Flood Susceptibility — {BEST_MODEL_NAME}", fontsize=14, fontweight="bold", loc="left")
axes[1].axis("off")
cb1 = fig.colorbar(im1, ax=axes[1], fraction=0.035, pad=0.02)
cb1.set_label("Susceptibility score [0–1]", fontsize=11)

fig.suptitle("Flood susceptibility score and local spatial concentration across South Asia",
             fontsize=15, fontweight="bold")
fig.savefig(OUTPUT_DIR / "fig_final_susceptibility_gistar_600dpi.png", dpi=600, bbox_inches="tight", facecolor="white")
plt.show()


## 19. Publication master validation table

The principal comparison is between repeated random and repeated buffered spatial validation. No geographic-holdout or locked-test columns are included.

In [ ]:
metric_cols = ["ROC_AUC", "PR_AUC", "Accuracy", "Precision", "Recall", "F1", "MCC", "Kappa", "Brier", "ECE"]
rmean = random_results.groupby("model")[metric_cols].mean().add_suffix("_random")
smean = spatial_results.groupby("model")[metric_cols].mean().add_suffix("_spatial")
master = rmean.join(smean).reset_index()

higher_better = ["ROC_AUC", "PR_AUC", "Accuracy", "Precision", "Recall", "F1", "MCC", "Kappa"]
for metric in higher_better:
    master[f"VOG_{metric}"] = master[f"{metric}_random"] - master[f"{metric}_spatial"]
    master[f"RVOG_{metric}_pct"] = (
        master[f"VOG_{metric}"] / master[f"{metric}_spatial"].replace(0, np.nan) * 100
    )
for metric in ["Brier", "ECE"]:
    master[f"VOG_{metric}_error"] = master[f"{metric}_spatial"] - master[f"{metric}_random"]

if "Brier_skill" in random_results.columns:
    bsr = random_results.groupby("model")["Brier_skill"].mean().rename("Brier_skill_random")
    bss = spatial_results.groupby("model")["Brier_skill"].mean().rename("Brier_skill_spatial")
    master = master.merge(pd.concat([bsr, bss], axis=1).reset_index(), on="model", how="left")

master.to_csv(OUTPUT_DIR / "MASTER_VALIDATION_RESULTS.csv", index=False)
display(master.round(4))

# 19. Model ranking under spatial validation

Do not select the final model using random-split AUC alone. The methodological paper should emphasize spatial generalization.

In [ ]:
spatial_ranking = (
    spatial_results.groupby("model")[["ROC_AUC","PR_AUC","MCC","Brier","ECE"]]
    .mean()
    .sort_values(["ROC_AUC","PR_AUC","MCC"], ascending=[False, False, False])
)

display(spatial_ranking.round(4))
spatial_ranking.to_csv(OUTPUT_DIR / "spatial_model_ranking.csv")
print(
    "This ordering is descriptive for the specified buffered spatial-CV regime; "
    "it is not a claim of universally superior geographic transfer."
)

## 20. Final manuscript result package

This cell compiles only outputs from the lock-free full-inventory methodology. It records explicitly that independent external geographic testing was not performed, so manuscript claims can remain aligned with the evidence.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# ============================================================
# FINAL MANUSCRIPT RESULTS PACKAGE — v3.2.2 GROUPED VALIDATION
# ============================================================

required_outputs = [
    "projection_and_grid_manifest.json",
    "projection_scale_diagnostics.csv",
    "raster_reprojection_manifest.csv",
    "dataset_qc_summary.csv",
    "predictor_manifest.csv",
    "regional_buffer_support_summary.csv",
    "regional_partition_fold_composition.csv",
    "regional_partition_stability.csv",
    "regional_buffer_support_by_fold_seed.csv",
    "spatial_validation_results_primary_buffered_nested.csv",
    "random_validation_results_nested.csv",
    "random_group_split_audit.csv",
    "spatial_block_bootstrap_random_vs_spatial.csv",
    "size_matched_random_vs_spatial.csv",
    "size_matched_group_split_audit.csv",
    "spatial_buffer_sensitivity_fixed_regions.csv",
    "MASTER_VALIDATION_RESULTS.csv",
    "map_comparison_summary.csv",
    "final_model_manifest.json",
    "raster_pixel_group_audit.csv",
    "mixed_label_raster_pixel_groups.csv",
    "mixed_label_raster_pixel_records.csv",
    "duplicate_coordinate_group_audit.csv",
    "SR_GEOAI_Flood_Master_Dataset_Grouped.csv",
    "final_gistar/country_mask_and_area_qc.json",
    "final_gistar/country_susceptibility_summary.csv",
    "final_gistar/country_gistar_summary.csv",
    "final_gistar/regional_gistar_summary.csv",
]
missing = [name for name in required_outputs if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Final result package incomplete; missing:\n" + "\n".join(missing)
    )

master = pd.read_csv(OUTPUT_DIR / "MASTER_VALIDATION_RESULTS.csv")
spatial = pd.read_csv(OUTPUT_DIR / "spatial_validation_results_primary_buffered_nested.csv")
random = pd.read_csv(OUTPUT_DIR / "random_validation_results_nested.csv")
matched = pd.read_csv(OUTPUT_DIR / "size_matched_random_vs_spatial.csv")
mapc = pd.read_csv(OUTPUT_DIR / "map_comparison_summary.csv")
boot = pd.read_csv(OUTPUT_DIR / "spatial_block_bootstrap_random_vs_spatial.csv")
buffer_sensitivity = pd.read_csv(OUTPUT_DIR / "spatial_buffer_sensitivity_fixed_regions.csv")
reg = pd.read_csv(OUTPUT_DIR / "final_gistar" / "regional_gistar_summary.csv")
cs = pd.read_csv(OUTPUT_DIR / "final_gistar" / "country_susceptibility_summary.csv")
cg = pd.read_csv(OUTPUT_DIR / "final_gistar" / "country_gistar_summary.csv")
mask_qc = json.loads((OUTPUT_DIR / "final_gistar" / "country_mask_and_area_qc.json").read_text())
model_manifest = json.loads((OUTPUT_DIR / "final_model_manifest.json").read_text())
random_group_audit = pd.read_csv(OUTPUT_DIR / "random_group_split_audit.csv")
matched_group_audit = pd.read_csv(OUTPUT_DIR / "size_matched_group_split_audit.csv")
pixel_group_audit = pd.read_csv(OUTPUT_DIR / "raster_pixel_group_audit.csv")
mixed_pixel_audit = pd.read_csv(OUTPUT_DIR / "mixed_label_raster_pixel_groups.csv")

if int(mask_qc["valid_cells_inside_country_union"]) != int(mask_qc["country_mapped_cell_sum"]):
    raise AssertionError("Country-area reconciliation failed.")
if int(mask_qc["class_counts_sum"]) != int(mask_qc["valid_cells_inside_country_union"]):
    raise AssertionError("Gi* classes do not reconcile with valid mapped cells.")
if len(spatial["repeat"].unique()) != len(SPATIAL_CV_SEEDS):
    raise AssertionError("Repeated spatial-validation outputs are incomplete.")
if set(random["model"].unique()) != set(spatial["model"].unique()):
    raise AssertionError("Random and spatial validation do not contain the same model set.")
if not random_group_audit["n_pixel_group_overlap"].eq(0).all():
    raise AssertionError("Random-validation pixel-group overlap detected.")
if not matched_group_audit["n_pixel_group_overlap"].eq(0).all():
    raise AssertionError("Size-matched pixel-group overlap detected.")
if len(mixed_pixel_audit) and not ALLOW_MIXED_LABEL_PIXEL_GROUPS:
    raise AssertionError("Mixed-label raster-cell groups remain unresolved.")


def clean_json_value(value):
    """Convert Pandas/NumPy values to strict JSON-safe values."""
    if value is None:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return [clean_json_value(v) for v in value.tolist()]
    if isinstance(value, dict):
        return {str(k): clean_json_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json_value(v) for v in value]
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def dataframe_to_json_records(df):
    """Flatten MultiIndex columns and return JSON-safe records."""
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            "_".join(str(part) for part in col if str(part) != "")
            for col in out.columns
        ]
    else:
        out.columns = [str(col) for col in out.columns]
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.astype(object).where(pd.notna(out), None)
    return clean_json_value(out.to_dict(orient="records"))


def dataframe_group_summary(df, group_col, value_cols):
    summary = (
        df.groupby(group_col)[value_cols]
        .agg(["mean", "std"])
        .round(6)
        .reset_index()
    )
    return dataframe_to_json_records(summary)


spatial_leaders = (
    spatial.groupby("model")["ROC_AUC"]
    .mean().sort_values(ascending=False).to_dict()
)
matched_summary = dataframe_to_json_records(
    matched.groupby("model")["paired_AUC_gap"].agg(["mean", "std"]).reset_index()
)
random_summary = dataframe_group_summary(random, "model", metric_cols)
spatial_summary = dataframe_group_summary(spatial, "model", metric_cols)
buffer_grouped = (
    buffer_sensitivity.groupby(["buffer_multiplier", "model"])["ROC_AUC"]
    .agg(["mean", "std"]).reset_index()
)
buffer_summary = dataframe_to_json_records(buffer_grouped)

result_package = {
    "study": "South Asia flood susceptibility — repeated spatial validation",
    "analysis_n": int(len(y)),
    "presence_n": int(np.sum(np.asarray(y) == 1)),
    "background_n": int(np.sum(np.asarray(y) == 0)),
    "predictor_count": int(len(FEATURE_COLS)),
    "selected_model_for_mapping": BEST_MODEL_NAME,
    "model_selection_is_independent_test": False,
    "independent_external_geographic_test_performed": False,
    "performance_evidence": "repeated nested grouped-random and buffered spatial cross-validation",
    "sampling_unit": "1-km harmonized raster cell; all records from a cell are assigned to one validation group",
    "n_sample_pixel_groups": int(N_SAMPLE_PIXEL_GROUPS),
    "n_multi_observation_pixel_groups": int(N_MULTI_OBSERVATION_PIXELS),
    "n_duplicate_coordinate_groups": int(N_DUPLICATE_COORDINATE_GROUPS),
    "n_mixed_label_pixel_groups": int(N_MIXED_LABEL_PIXEL_GROUPS),
    "mixed_label_pixel_groups_explicitly_allowed": bool(ALLOW_MIXED_LABEL_PIXEL_GROUPS),
    "runtime_profile": "v3.2.2 pixel-grouped; 15 random repeats; 2 inner folds; 4 final and 2 calibration tuning draws",
    "mapping_ensemble_scope": "five spatial folds from the primary partition seed only",
    "random_validation_summary": random_summary,
    "random_group_split_audit": dataframe_to_json_records(random_group_audit),
    "size_matched_group_split_audit": dataframe_to_json_records(matched_group_audit),
    "spatial_validation_summary": spatial_summary,
    "validation_optimism_gap": dataframe_to_json_records(master),
    "spatial_model_ranking": spatial_leaders,
    "size_matched_random_spatial_gap": matched_summary,
    "spatial_block_bootstrap_gap": dataframe_to_json_records(boot),
    "spatial_block_bootstrap_scope": "spatial-block resampling of paired mean out-of-fold predictions; conditional on fitted resampling procedures",
    "buffer_sensitivity": buffer_summary,
    "map_comparison": dataframe_to_json_records(mapc),
    "regional_descriptive_gistar": dataframe_to_json_records(reg),
    "country_susceptibility_descriptive": dataframe_to_json_records(cs),
    "country_gistar_descriptive": dataframe_to_json_records(cg),
    "spatial_cv_seeds": [int(seed) for seed in SPATIAL_CV_SEEDS],
    "mapping_ensemble_seed": int(RANDOM_STATE_FINAL),
    "mapping_ensemble_scope": "five spatial fold models from the prespecified primary partition seed only",
    "partition_similarity_ari": dataframe_to_json_records(partition_similarity),
    "nested_tuning_final_iterations": int(N_ITER_SEARCH),
    "nested_tuning_calibration_fold_iterations": int(N_ITER_CALIBRATION_SEARCH),
    "n_spatial_folds": int(N_SPATIAL_FOLDS),
    "reference_spatial_range_m": float(SPATIAL_RANGE),
    "primary_buffer_distance_m": float(BUFFER_DISTANCE),
    "target_crs": TARGET_CRS.to_string(),
    "target_resolution_m": float(TARGET_RESOLUTION_M),
    "final_model_manifest": model_manifest,
    "country_mask_area_qc": mask_qc,
    "event_id_col": EVENT_ID_COL,
    "event_grouped_audit_available": bool((OUTPUT_DIR / "event_grouped_validation_results.csv").exists()),
}

json_path = OUTPUT_DIR / "FINAL_MANUSCRIPT_RESULTS.json"
json_path.write_text(
    json.dumps(clean_json_value(result_package), indent=2, ensure_ascii=False, allow_nan=False),
    encoding="utf-8"
)

md_lines = [
    "# Final manuscript quantitative results — 1-km pixel-grouped repeated validation (v3.2.2)\n",
    f"Full modelling inventory: **n = {len(y):,}** ({int(np.sum(np.asarray(y) == 1)):,} flood-presence; {int(np.sum(np.asarray(y) == 0)):,} background).\n",
    f"Sampling support: **{N_SAMPLE_PIXEL_GROUPS:,} distinct 1-km raster cells**, including {N_MULTI_OBSERVATION_PIXELS:,} multi-observation cells.\n",
    f"Mixed-label raster-cell groups retained: **{N_MIXED_LABEL_PIXEL_GROUPS}** (explicitly allowed = {ALLOW_MIXED_LABEL_PIXEL_GROUPS}).\n",
    f"Selected mapping model: **{BEST_MODEL_NAME}** (selected using mean repeated buffered spatial-CV ROC-AUC; selection-affected, not an independent test result).\n",
    f"Spatial design: **{N_SPATIAL_FOLDS} contiguous K-means geographic folds**, {len(SPATIAL_CV_SEEDS)} partition seeds, geodesic buffer **{BUFFER_DISTANCE / 1000:.1f} km**.\n",
    "Random outer, inner tuning, calibration, and size-matched splits keep every 1-km raster cell intact.\n",
    (f"Grouped random holdout test fraction: mean {random_group_audit['realized_test_fraction'].mean():.3f} "
     f"(range {random_group_audit['realized_test_fraction'].min():.3f}–"
     f"{random_group_audit['realized_test_fraction'].max():.3f}).\n"),
    "**No locked geographic test or independent external geographic test was performed.**\n",
    "## Repeated spatial model ranking\n",
]
for name, value in spatial_leaders.items():
    md_lines.append(f"- {name}: mean buffered spatial ROC-AUC = {value:.3f}.\n")
md_lines.append("\n## Size-matched benchmark\n")
for row in matched_summary:
    mean_value, std_value = row.get("mean"), row.get("std")
    if mean_value is not None:
        md_lines.append(
            f"- {row['model']}: mean paired random-minus-spatial AUC gap = "
            f"{mean_value:.3f} ± {std_value:.3f}.\n"
        )
md_lines.append("\n## Map comparison\n")
for _, row in mapc.iterrows():
    md_lines.append(
        f"- {row['threshold_definition']}: Jaccard = {row['jaccard_similarity']:.3f}; "
        f"area difference = {row['area_difference_km2']:.0f} km²; "
        f"RMSE = {row['RMSE']:.3f}; MAE = {row['MAE']:.3f}.\n"
    )
md_lines.append("\n## Descriptive Gi* and country summaries\n")
if len(reg):
    rr = reg.iloc[0]
    md_lines.append(f"- High-score Gi* cluster bands: {rr['high_cluster_pct_of_mapped']:.2f}% of mapped area.\n")
    md_lines.append(f"- Low-score Gi* cluster bands: {rr['low_cluster_pct_of_mapped']:.2f}% of mapped area.\n")
md_lines.append("\n## Country susceptibility summaries (descriptive)\n\n")
md_lines.append(cs.to_markdown(index=False, floatfmt=".3f"))
md_lines.append("\n\n## Country Gi* summaries (descriptive)\n\n")
md_lines.append(cg.to_markdown(index=False, floatfmt=".3f"))
md_lines.append(
    "\n\n## Interpretation boundary\n\n"
    "These results estimate performance under the specified resampling designs. They do not constitute an independent "
    "external test or direct map-accuracy assessment. Scores are relative susceptibility scores conditional on the "
    "case/background sampling design. Grouping by 1-km raster cell prevents records sharing the same raster-derived "
    "predictor support from crossing validation boundaries.\n"
)
md_path = OUTPUT_DIR / "FINAL_MANUSCRIPT_RESULTS.md"
md_path.write_text("\n".join(md_lines), encoding="utf-8")
print("Final v3.2.2 grouped-validation manuscript package written:")
print(json_path)
print(md_path)

## 21. Final methodological checklist

- Full corrected inventory retained; no locked region or locked-region buffer.
- Complete-case selection uses only the 21 final model predictors.
- CRS and 1-km raster alignment verified for all predictors.
- Spatial design selected from geometry and support diagnostics, not model performance.
- All spatial folds satisfy the declared total and class-specific support rules.
- Random and buffered spatial validation use nested tuning and nested cross-fitted calibration.
- Size-matched benchmark preserves both sample size and class counts.
- Paired bootstrap uses full-inventory local indices and projected spatial blocks.
- Map comparison surfaces share an identical grid and valid-cell mask.
- Gi* / country statistics reconcile with the valid country-union mask and are described as secondary model-output summaries.
- No independent external geographic test is claimed.
- Flood event identifiers and background sampling provenance remain explicitly documented limitations until recovered.

## Final submission principle

Do not copy numerical results from v11 or the earlier locked-test notebook into the revised manuscript. Use only the fresh outputs produced by a clean v3 execution. The final paper must describe repeated nested random and buffered spatial cross-validation, the support-matched comparison, buffer sensitivity, calibration and map comparison. It must not claim independent external geographic validation or unbiased map accuracy.

## 22. Selective output packaging

The following cell packages the analysis outputs while excluding intermediate aligned rasters and scratch files. Retain the notebook, source metadata, and final result tables/figures with the archive.

### Required execution order

Run from Cell 2 in a clean runtime through the full-inventory setup, spatial dependence, support-aware buffer selection, nested buffered spatial CV, nested random validation, size-matched benchmark, buffer sensitivity, calibration and map generation, descriptive Gi* summaries, final result package and preflight. No locked-test or soil-ablation stage is present in v3.

In [ ]:
import os
import zipfile
from pathlib import Path

# --- 1. Resolve the output directory -----------------------------------------------
try:
    SRC_DIR = Path(OUTPUT_DIR)                       # use the notebook's own variable if defined
except NameError:
    SRC_DIR = Path("sr_geoai_outputs_v3_2_2_pixel_grouped")  # standalone fallback

# --- 2. What actually belongs in the deliverable -------------------------------------
INCLUDE_EXTENSIONS = {".csv", ".png", ".tif", ".tiff", ".json", ".md"}

# Any directory whose name matches one of these (anywhere in the path) is skipped
# entirely, regardless of what's inside it.
EXCLUDE_DIR_NAMES = {
    "aligned_rasters",     # intermediate rasters
    "harmonized_aea_1km",  # corrected predictor inputs; large, separately archived
    "__pycache__",
    ".ipynb_checkpoints",
    "tmp", "temp", "scratch",
}

# Specific filename patterns to skip even if the extension matches (e.g. accidental
# full-resolution copies of raw predictor rasters that aren't your actual outputs).
EXCLUDE_FILENAME_SUBSTRINGS = {
    "_aligned",            # e.g. "Slope_aligned.tif"
}

def should_include(path: Path) -> bool:
    if path.suffix.lower() not in INCLUDE_EXTENSIONS:
        return False
    if any(part in EXCLUDE_DIR_NAMES for part in path.parts):
        return False
    if any(bad in path.name for bad in EXCLUDE_FILENAME_SUBSTRINGS):
        return False
    return True

# --- 3. Walk and collect -------------------------------------------------------------
files_to_zip = [p for p in SRC_DIR.rglob("*") if p.is_file() and should_include(p)]

if not files_to_zip:
    raise FileNotFoundError(f"No matching files found under {SRC_DIR}. "
                             f"Check OUTPUT_DIR / SRC_DIR is correct.")

# --- 4. Build the zip, report size before/after --------------------------------------
zip_path = SRC_DIR.parent / f"{SRC_DIR.name}_clean.zip"

total_included_bytes = sum(p.stat().st_size for p in files_to_zip)
total_dir_bytes = sum(p.stat().st_size for p in SRC_DIR.rglob("*") if p.is_file())

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_zip:
        zf.write(p, arcname=p.relative_to(SRC_DIR))

def human(n):
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"

print(f"Everything under {SRC_DIR}:  {human(total_dir_bytes)}  ({sum(1 for _ in SRC_DIR.rglob('*') if _.is_file())} files)")
print(f"Included in clean zip:       {human(total_included_bytes)}  ({len(files_to_zip)} files)")
print(f"Excluded:                    {human(total_dir_bytes - total_included_bytes)}")
print(f"\nWrote: {zip_path}  ({human(zip_path.stat().st_size)})")
print("\nFiles included:")
for p in sorted(files_to_zip):
    print(f"  {p.relative_to(SRC_DIR)}  ({human(p.stat().st_size)})")

## 23. Final rerun preflight and provenance lock

This cell verifies that the entire corrected inventory was used, repeated buffered spatial CV completed across all seeds, the selected buffer was respected, and map/country summaries reconcile. It deliberately contains no locked-test checks.

In [ ]:
# ============================================================
# FINAL PREFLIGHT — FULL INVENTORY, NO LOCKED TEST
# ============================================================
assert set(np.unique(np.asarray(y, dtype=int))).issubset({0, 1})
assert len(df) == len(X) == len(y) == len(coords)
assert len(df) == len(FULL_INVENTORY_IDX)
assert BEST_MODEL_NAME in MODELS
assert len(spatial_results["repeat"].unique()) == len(SPATIAL_CV_SEEDS)
assert len(spatial_results) == len(SPATIAL_CV_SEEDS) * N_SPATIAL_FOLDS * len(MODELS)
assert np.all(buffer_diagnostics["n_train_buffered"] >= MIN_RETAINED_TRAIN_N)
assert np.all(buffer_diagnostics["min_train_class_n"] >= MIN_TRAIN_PER_CLASS_N)
assert np.all(buffer_diagnostics["min_test_class_n"] >= MIN_TEST_PER_CLASS_N)
assert not separation_df.empty and separation_df["buffer_constraint_satisfied"].all()
assert "nested_calibration_tuning" in spatial_results.columns
assert spatial_results["nested_calibration_tuning"].astype(bool).all()
assert "nested_calibration_tuning" in random_results.columns
assert random_results["nested_calibration_tuning"].astype(bool).all()
assert len(random_results) == len(SEEDS) * len(MODELS)
assert len(random_group_split_audit) == len(SEEDS)
assert random_group_split_audit["n_pixel_group_overlap"].eq(0).all()
assert not matched_group_split_audit.empty
assert matched_group_split_audit["n_pixel_group_overlap"].eq(0).all()
assert int(N_SAMPLE_PIXEL_GROUPS) == len(np.unique(SAMPLE_PIXEL_GROUP_IDS))
assert len(buffer_diagnostics) == len(SPATIAL_CV_SEEDS) * N_SPATIAL_FOLDS
assert len(partition_similarity) == len(SPATIAL_CV_SEEDS) * (len(SPATIAL_CV_SEEDS) - 1) // 2
assert partition_similarity["adjusted_rand_index"].between(-1, 1).all()

with rasterio.open(path_final_map) as final_src:
    assert final_src.crs == TARGET_CRS
    assert np.allclose((final_src.transform.a, abs(final_src.transform.e)), (1000, 1000), atol=1e-6)

mask_qc = json.loads((OUTPUT_DIR / "final_gistar" / "country_mask_and_area_qc.json").read_text())
assert mask_qc["valid_cells_inside_country_union"] == mask_qc["country_mapped_cell_sum"]
assert mask_qc["class_counts_sum"] == mask_qc["valid_cells_inside_country_union"]

required_final = [
    "FINAL_MANUSCRIPT_RESULTS.json", "FINAL_MANUSCRIPT_RESULTS.md",
    "final_model_manifest.json", "projection_and_grid_manifest.json",
    "dataset_qc_summary.csv", "predictor_manifest.csv",
    "regional_buffer_support_summary.csv", "regional_buffer_support_by_fold_seed.csv",
    "regional_partition_fold_composition.csv", "regional_partition_stability.csv",
    "model_rank_stability_by_partition_seed.csv",
    "random_validation_results_nested.csv",
    "random_group_split_audit.csv",
    "raster_pixel_group_audit.csv",
    "mixed_label_raster_pixel_groups.csv",
    "mixed_label_raster_pixel_records.csv",
    "SR_GEOAI_Flood_Master_Dataset_Grouped.csv",
    "size_matched_group_split_audit.csv",
    "spatial_validation_results_primary_buffered_nested.csv",
    "spatial_block_bootstrap_random_vs_spatial.csv",
    "size_matched_random_vs_spatial.csv",
    "spatial_buffer_sensitivity_fixed_regions.csv",
    "MASTER_VALIDATION_RESULTS.csv", "map_comparison_summary.csv",
]
missing = [f for f in required_final if not (OUTPUT_DIR / f).exists()]
if missing:
    raise FileNotFoundError("Preflight failed; required outputs are missing:\n" + "\n".join(missing))

final_preflight = {
    "n_full_corrected_inventory": int(len(y)),
    "n_presence": int(np.sum(np.asarray(y) == 1)),
    "n_background": int(np.sum(np.asarray(y) == 0)),
    "n_scientific_predictors": int(len(FEATURE_COLS)),
    "independent_external_geographic_test_performed": False,
    "spatial_partition_strategy": "weighted_kmeans_voronoi_regions_over_1km_raster_cell_groups",
    "n_spatial_folds": int(N_SPATIAL_FOLDS),
    "spatial_cv_seeds": list(SPATIAL_CV_SEEDS),
    "mapping_ensemble_seed": int(RANDOM_STATE_FINAL),
    "mapping_ensemble_scope": "primary partition seed only",
    "reference_spatial_range_km": float(SPATIAL_RANGE / 1000),
    "primary_buffer_km": float(BUFFER_DISTANCE / 1000),
    "selected_model_for_mapping": BEST_MODEL_NAME,
    "final_mapping_fit_scope": "full corrected modelling inventory",
    "target_crs": TARGET_CRS.to_string(),
    "target_resolution_m": TARGET_RESOLUTION_M,
    "mapped_country_union_cells": int(mask_qc["valid_cells_inside_country_union"]),
    "country_mapped_cell_sum": int(mask_qc["country_mapped_cell_sum"]),
    "event_id_col": EVENT_ID_COL,
    "event_grouped_audit_available": bool((OUTPUT_DIR / "event_grouped_validation_results.csv").exists()),
    "sampling_unit": "1-km harmonized raster cell",
    "n_sample_pixel_groups": int(N_SAMPLE_PIXEL_GROUPS),
    "n_multi_observation_pixel_groups": int(N_MULTI_OBSERVATION_PIXELS),
    "n_duplicate_coordinate_groups": int(N_DUPLICATE_COORDINATE_GROUPS),
    "n_mixed_label_pixel_groups": int(N_MIXED_LABEL_PIXEL_GROUPS),
    "mixed_label_pixel_groups_explicitly_allowed": bool(ALLOW_MIXED_LABEL_PIXEL_GROUPS),
    "random_outer_pixel_group_overlap_events": int(random_group_split_audit["n_pixel_group_overlap"].sum()),
    "size_matched_pixel_group_overlap_events": int(matched_group_split_audit["n_pixel_group_overlap"].sum()),
    "timestamp_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
(OUTPUT_DIR / "RUN_PREFLIGHT_MANIFEST.json").write_text(
    json.dumps(final_preflight, indent=2, default=str), encoding="utf-8"
)
print("FULL-INVENTORY LOCK-FREE PREFLIGHT PASSED")
display(pd.DataFrame([final_preflight]))

## Reproduction and unresolved source metadata

Run this notebook from Cell 2 with the original point inventory and predictor rasters. The notebook will fail early if source files, CRS, raster alignment, class support, spatial partitions, buffers, or area accounting are invalid. Do not combine outputs from this lock-free v3 analysis with the earlier v11 or locked-test outputs.

The original flood-event linkage and exact background sampling rule still need to be recovered from the inventory-generation workflow. If these cannot be reconstructed, state explicitly that event independence cannot be verified and that non-flood samples are background/pseudo-absence observations rather than confirmed absences.